In [ ]:
# Phase 3 gold: star schema in wh_gold via T-SQL, executed from this
# Spark notebook via pyodbc + an AAD access token (confirmed available in
# Fabric's default runtime -- ODBC Driver 18 for SQL Server,
# notebookutils.credentials.getToken). Cross-database T-SQL queries let
# the Warehouse read Lakehouse Delta tables directly
# ([lh_silver].[dbo].[table]) -- confirmed working (COUNT(*) across
# lh_silver.fact_flight_leg returned 27,787,225, matching the full silver
# run), so the big loads need no Spark data movement at all.
#
# Every cell writes its result (or, on failure, the full traceback) to
# Files/diagnostics/<name>.json instead of relying on printed cell
# output -- the Fabric REST job-status API never surfaces cell-level
# errors (confirmed repeatedly this project), and this lets failures be
# self-diagnosed via the OneLake DFS API with no portal access needed.
import pyodbc, struct, json as _json, traceback as _tb

WORKSPACE_ID = "16b2a9c6-6418-485f-865f-c47442b8b6c6"
LH_BRONZE_ID = "09a8b1f5-f33c-4eed-912d-40b9cbdc9b06"

def write_diag(name, obj):
    path = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{LH_BRONZE_ID}/Files/diagnostics/{name}.json"
    notebookutils.fs.put(path, _json.dumps(obj, indent=2, default=str), overwrite=True)
    print(f"wrote {path}")

try:
    server = "4oa7g2g2i24uposxn4zcxdynue-y2u3efqymrpurbs7yr2efofwyy.datawarehouse.fabric.microsoft.com"
    database = "wh_gold"
    token = notebookutils.credentials.getToken("https://database.windows.net/")
    token_bytes = token.encode("utf-16-le")
    token_struct = struct.pack(f"<I{len(token_bytes)}s", len(token_bytes), token_bytes)
    SQL_COPT_SS_ACCESS_TOKEN = 1256
    conn_str = f"DRIVER={{ODBC Driver 18 for SQL Server}};SERVER={server};DATABASE={database};Encrypt=yes;"
    conn = pyodbc.connect(conn_str, attrs_before={SQL_COPT_SS_ACCESS_TOKEN: token_struct}, autocommit=True)
    cursor = conn.cursor()
    print("connected")
    write_diag("gold_connect", {"status": "ok"})
except Exception as e:
    write_diag("gold_connect", {"status": "failed", "error": repr(e), "traceback": _tb.format_exc()})
    raise


In [ ]:
ddl_statements = ['DROP TABLE IF EXISTS dbo.dim_date', 'CREATE TABLE dbo.dim_date (\n        date_key INT NOT NULL,\n        full_date DATE NOT NULL,\n        [year] INT NOT NULL,\n        [month] INT NOT NULL,\n        [day] INT NOT NULL,\n        day_of_week INT NOT NULL,\n        day_name VARCHAR(10) NOT NULL,\n        month_name VARCHAR(10) NOT NULL,\n        quarter INT NOT NULL,\n        is_weekend BIT NOT NULL,\n        is_holiday BIT NOT NULL,\n        schedule_season VARCHAR(10) NOT NULL,\n        is_post_dfw_rebank BIT NOT NULL\n    )', 'DROP TABLE IF EXISTS dbo.dim_time_15', 'CREATE TABLE dbo.dim_time_15 (\n        time_key INT NOT NULL,\n        hour24 INT NOT NULL,\n        minute15 INT NOT NULL,\n        hhmm VARCHAR(5) NOT NULL,\n        daypart VARCHAR(20) NOT NULL\n    )', 'DROP TABLE IF EXISTS dbo.dim_airport', 'CREATE TABLE dbo.dim_airport (\n        airport_key VARCHAR(3) NOT NULL,\n        airport_name VARCHAR(100) NULL,\n        municipality VARCHAR(100) NULL,\n        iso_region VARCHAR(10) NULL,\n        latitude FLOAT NULL,\n        longitude FLOAT NULL,\n        tz VARCHAR(50) NULL,\n        is_aa_hub BIT NOT NULL\n    )', 'DROP TABLE IF EXISTS dbo.dim_carrier', 'CREATE TABLE dbo.dim_carrier (\n        carrier_key VARCHAR(2) NOT NULL,\n        carrier_name VARCHAR(100) NOT NULL,\n        is_mainline BIT NOT NULL,\n        is_aa_family BIT NOT NULL\n    )', 'DROP TABLE IF EXISTS dbo.dim_aircraft', 'CREATE TABLE dbo.dim_aircraft (\n        tail_number VARCHAR(10) NOT NULL,\n        manufacturer VARCHAR(50) NULL,\n        model VARCHAR(50) NULL,\n        fleet_family VARCHAR(50) NULL,\n        seats INT NULL,\n        num_engines INT NULL,\n        year_mfr INT NULL\n    )', 'DROP TABLE IF EXISTS dbo.dim_delay_cause', 'CREATE TABLE dbo.dim_delay_cause (\n        cause_code VARCHAR(20) NOT NULL,\n        label VARCHAR(50) NOT NULL,\n        is_controllable BIT NOT NULL\n    )', 'DROP TABLE IF EXISTS dbo.dim_cancel_reason', 'CREATE TABLE dbo.dim_cancel_reason (\n        reason_code VARCHAR(2) NOT NULL,\n        label VARCHAR(50) NOT NULL,\n        is_controllable BIT NOT NULL\n    )', 'DROP TABLE IF EXISTS dbo.dim_bank', 'CREATE TABLE dbo.dim_bank (\n        bank_key INT NOT NULL,\n        station VARCHAR(3) NOT NULL,\n        period VARCHAR(30) NOT NULL,\n        direction VARCHAR(15) NOT NULL,\n        bank_number INT NOT NULL,\n        peak_time VARCHAR(5) NULL,\n        peak_avg_flights FLOAT NULL,\n        window_start VARCHAR(5) NOT NULL,\n        window_end VARCHAR(5) NOT NULL\n    )', 'DROP TABLE IF EXISTS dbo.fact_flight_leg', 'CREATE TABLE dbo.fact_flight_leg (\n        date_key INT NOT NULL,\n        origin_airport_key VARCHAR(3) NOT NULL,\n        dest_airport_key VARCHAR(3) NOT NULL,\n        marketing_carrier_key VARCHAR(2) NOT NULL,\n        operating_carrier_key VARCHAR(2) NOT NULL,\n        flight_number VARCHAR(10) NULL,\n        tail_number VARCHAR(10) NULL,\n        sched_dep_utc DATETIME2(6) NULL,\n        sched_arr_utc DATETIME2(6) NULL,\n        actual_dep_utc DATETIME2(6) NULL,\n        actual_arr_utc DATETIME2(6) NULL,\n        dep_delay_min FLOAT NULL,\n        arr_delay_min FLOAT NULL,\n        taxi_out_min FLOAT NULL,\n        taxi_in_min FLOAT NULL,\n        crs_elapsed_min FLOAT NULL,\n        actual_elapsed_min FLOAT NULL,\n        carrier_delay_min FLOAT NULL,\n        weather_delay_min FLOAT NULL,\n        nas_delay_min FLOAT NULL,\n        security_delay_min FLOAT NULL,\n        late_aircraft_delay_min FLOAT NULL,\n        is_cancelled BIT NOT NULL,\n        is_diverted BIT NOT NULL,\n        is_d0 BIT NULL,\n        is_a14 BIT NULL,\n        is_b0 BIT NULL,\n        is_controllable_cancel BIT NOT NULL,\n        cancellation_code VARCHAR(2) NULL,\n        distance_mi FLOAT NULL\n    )', 'DROP TABLE IF EXISTS dbo.fact_checkpoint_hour', 'CREATE TABLE dbo.fact_checkpoint_hour (\n        station VARCHAR(3) NOT NULL,\n        [checkpoint] VARCHAR(50) NULL,\n        date_key INT NOT NULL,\n        hour24 INT NOT NULL,\n        week_start DATE NULL,\n        passengers_screened INT NULL\n    )', 'DROP TABLE IF EXISTS dbo.fact_cbp_wait_hour', 'CREATE TABLE dbo.fact_cbp_wait_hour (\n        station VARCHAR(10) NOT NULL,\n        terminal VARCHAR(50) NULL,\n        date_key INT NOT NULL,\n        hour_range VARCHAR(20) NULL,\n        avg_wait_us FLOAT NULL,\n        max_wait_us FLOAT NULL,\n        avg_wait_non_us FLOAT NULL,\n        max_wait_non_us FLOAT NULL,\n        booths_open INT NULL,\n        flight_count INT NULL,\n        total_passengers INT NULL\n    )', 'DROP TABLE IF EXISTS dbo.fact_rebank_effect', 'CREATE TABLE dbo.fact_rebank_effect (\n        hub VARCHAR(3) NOT NULL,\n        metric VARCHAR(20) NOT NULL,\n        model VARCHAR(200) NOT NULL,\n        estimate FLOAT NOT NULL,\n        se FLOAT NULL,\n        ci_low FLOAT NULL,\n        ci_high FLOAT NULL,\n        p_value FLOAT NULL,\n        n BIGINT NULL,\n        controls VARCHAR(500) NULL\n    )', 'DROP TABLE IF EXISTS dbo.fact_rebank_event_study', 'CREATE TABLE dbo.fact_rebank_event_study (\n        week_relative INT NOT NULL,\n        coef FLOAT NOT NULL,\n        ci_low FLOAT NULL,\n        ci_high FLOAT NULL\n    )']
ddl_errors = []
ran = 0
for i, stmt in enumerate(ddl_statements):
    try:
        cursor.execute(stmt)
        ran += 1
    except Exception as e:
        ddl_errors.append({"index": i, "statement": stmt, "error": repr(e), "traceback": _tb.format_exc()})
print(f"ran {ran}/{len(ddl_statements)} DDL statements, {len(ddl_errors)} errors")
write_diag("gold_ddl", {"statements_run": ran, "total": len(ddl_statements), "errors": ddl_errors})
if ddl_errors:
    raise RuntimeError(f"{len(ddl_errors)} DDL statement(s) failed -- see Files/diagnostics/gold_ddl.json")


In [ ]:
try:
    # dim_date: generated in Python (not a T-SQL recursive CTE -- simpler and
    # avoids any uncertainty about Fabric Warehouse's recursive-CTE support)
    # for 2023-01-01..2026-12-31, covering lh_silver.fact_flight_leg's
    # real FlightDate range (2023-01-01 to 2026-08-31) with headroom. Holidays
    # are the real US federal holiday observed dates (scripts/fabric/
    # compute_us_holidays.py's logic, inlined) -- Nth-weekday-of-month or
    # fixed-date-with-nearest-weekday-observance rules, not a hand-typed list.
    # is_post_dfw_rebank uses week 15 (April 6-12, 2026), the sustained-jump
    # week detected from the data per ADR 0003 -- not the April 2026 the
    # Cranky Flier article names loosely.
    dim_date_rows = [(20230101, '2023-01-01', 2023, 1, 1, 7, 'Sunday', 'January', 1, True, False, 'winter', False), (20230102, '2023-01-02', 2023, 1, 2, 1, 'Monday', 'January', 1, False, True, 'winter', False), (20230103, '2023-01-03', 2023, 1, 3, 2, 'Tuesday', 'January', 1, False, False, 'winter', False), (20230104, '2023-01-04', 2023, 1, 4, 3, 'Wednesday', 'January', 1, False, False, 'winter', False), (20230105, '2023-01-05', 2023, 1, 5, 4, 'Thursday', 'January', 1, False, False, 'winter', False), (20230106, '2023-01-06', 2023, 1, 6, 5, 'Friday', 'January', 1, False, False, 'winter', False), (20230107, '2023-01-07', 2023, 1, 7, 6, 'Saturday', 'January', 1, True, False, 'winter', False), (20230108, '2023-01-08', 2023, 1, 8, 7, 'Sunday', 'January', 1, True, False, 'winter', False), (20230109, '2023-01-09', 2023, 1, 9, 1, 'Monday', 'January', 1, False, False, 'winter', False), (20230110, '2023-01-10', 2023, 1, 10, 2, 'Tuesday', 'January', 1, False, False, 'winter', False), (20230111, '2023-01-11', 2023, 1, 11, 3, 'Wednesday', 'January', 1, False, False, 'winter', False), (20230112, '2023-01-12', 2023, 1, 12, 4, 'Thursday', 'January', 1, False, False, 'winter', False), (20230113, '2023-01-13', 2023, 1, 13, 5, 'Friday', 'January', 1, False, False, 'winter', False), (20230114, '2023-01-14', 2023, 1, 14, 6, 'Saturday', 'January', 1, True, False, 'winter', False), (20230115, '2023-01-15', 2023, 1, 15, 7, 'Sunday', 'January', 1, True, False, 'winter', False), (20230116, '2023-01-16', 2023, 1, 16, 1, 'Monday', 'January', 1, False, True, 'winter', False), (20230117, '2023-01-17', 2023, 1, 17, 2, 'Tuesday', 'January', 1, False, False, 'winter', False), (20230118, '2023-01-18', 2023, 1, 18, 3, 'Wednesday', 'January', 1, False, False, 'winter', False), (20230119, '2023-01-19', 2023, 1, 19, 4, 'Thursday', 'January', 1, False, False, 'winter', False), (20230120, '2023-01-20', 2023, 1, 20, 5, 'Friday', 'January', 1, False, False, 'winter', False), (20230121, '2023-01-21', 2023, 1, 21, 6, 'Saturday', 'January', 1, True, False, 'winter', False), (20230122, '2023-01-22', 2023, 1, 22, 7, 'Sunday', 'January', 1, True, False, 'winter', False), (20230123, '2023-01-23', 2023, 1, 23, 1, 'Monday', 'January', 1, False, False, 'winter', False), (20230124, '2023-01-24', 2023, 1, 24, 2, 'Tuesday', 'January', 1, False, False, 'winter', False), (20230125, '2023-01-25', 2023, 1, 25, 3, 'Wednesday', 'January', 1, False, False, 'winter', False), (20230126, '2023-01-26', 2023, 1, 26, 4, 'Thursday', 'January', 1, False, False, 'winter', False), (20230127, '2023-01-27', 2023, 1, 27, 5, 'Friday', 'January', 1, False, False, 'winter', False), (20230128, '2023-01-28', 2023, 1, 28, 6, 'Saturday', 'January', 1, True, False, 'winter', False), (20230129, '2023-01-29', 2023, 1, 29, 7, 'Sunday', 'January', 1, True, False, 'winter', False), (20230130, '2023-01-30', 2023, 1, 30, 1, 'Monday', 'January', 1, False, False, 'winter', False), (20230131, '2023-01-31', 2023, 1, 31, 2, 'Tuesday', 'January', 1, False, False, 'winter', False), (20230201, '2023-02-01', 2023, 2, 1, 3, 'Wednesday', 'February', 1, False, False, 'winter', False), (20230202, '2023-02-02', 2023, 2, 2, 4, 'Thursday', 'February', 1, False, False, 'winter', False), (20230203, '2023-02-03', 2023, 2, 3, 5, 'Friday', 'February', 1, False, False, 'winter', False), (20230204, '2023-02-04', 2023, 2, 4, 6, 'Saturday', 'February', 1, True, False, 'winter', False), (20230205, '2023-02-05', 2023, 2, 5, 7, 'Sunday', 'February', 1, True, False, 'winter', False), (20230206, '2023-02-06', 2023, 2, 6, 1, 'Monday', 'February', 1, False, False, 'winter', False), (20230207, '2023-02-07', 2023, 2, 7, 2, 'Tuesday', 'February', 1, False, False, 'winter', False), (20230208, '2023-02-08', 2023, 2, 8, 3, 'Wednesday', 'February', 1, False, False, 'winter', False), (20230209, '2023-02-09', 2023, 2, 9, 4, 'Thursday', 'February', 1, False, False, 'winter', False), (20230210, '2023-02-10', 2023, 2, 10, 5, 'Friday', 'February', 1, False, False, 'winter', False), (20230211, '2023-02-11', 2023, 2, 11, 6, 'Saturday', 'February', 1, True, False, 'winter', False), (20230212, '2023-02-12', 2023, 2, 12, 7, 'Sunday', 'February', 1, True, False, 'winter', False), (20230213, '2023-02-13', 2023, 2, 13, 1, 'Monday', 'February', 1, False, False, 'winter', False), (20230214, '2023-02-14', 2023, 2, 14, 2, 'Tuesday', 'February', 1, False, False, 'winter', False), (20230215, '2023-02-15', 2023, 2, 15, 3, 'Wednesday', 'February', 1, False, False, 'winter', False), (20230216, '2023-02-16', 2023, 2, 16, 4, 'Thursday', 'February', 1, False, False, 'winter', False), (20230217, '2023-02-17', 2023, 2, 17, 5, 'Friday', 'February', 1, False, False, 'winter', False), (20230218, '2023-02-18', 2023, 2, 18, 6, 'Saturday', 'February', 1, True, False, 'winter', False), (20230219, '2023-02-19', 2023, 2, 19, 7, 'Sunday', 'February', 1, True, False, 'winter', False), (20230220, '2023-02-20', 2023, 2, 20, 1, 'Monday', 'February', 1, False, True, 'winter', False), (20230221, '2023-02-21', 2023, 2, 21, 2, 'Tuesday', 'February', 1, False, False, 'winter', False), (20230222, '2023-02-22', 2023, 2, 22, 3, 'Wednesday', 'February', 1, False, False, 'winter', False), (20230223, '2023-02-23', 2023, 2, 23, 4, 'Thursday', 'February', 1, False, False, 'winter', False), (20230224, '2023-02-24', 2023, 2, 24, 5, 'Friday', 'February', 1, False, False, 'winter', False), (20230225, '2023-02-25', 2023, 2, 25, 6, 'Saturday', 'February', 1, True, False, 'winter', False), (20230226, '2023-02-26', 2023, 2, 26, 7, 'Sunday', 'February', 1, True, False, 'winter', False), (20230227, '2023-02-27', 2023, 2, 27, 1, 'Monday', 'February', 1, False, False, 'winter', False), (20230228, '2023-02-28', 2023, 2, 28, 2, 'Tuesday', 'February', 1, False, False, 'winter', False), (20230301, '2023-03-01', 2023, 3, 1, 3, 'Wednesday', 'March', 1, False, False, 'winter', False), (20230302, '2023-03-02', 2023, 3, 2, 4, 'Thursday', 'March', 1, False, False, 'winter', False), (20230303, '2023-03-03', 2023, 3, 3, 5, 'Friday', 'March', 1, False, False, 'winter', False), (20230304, '2023-03-04', 2023, 3, 4, 6, 'Saturday', 'March', 1, True, False, 'winter', False), (20230305, '2023-03-05', 2023, 3, 5, 7, 'Sunday', 'March', 1, True, False, 'winter', False), (20230306, '2023-03-06', 2023, 3, 6, 1, 'Monday', 'March', 1, False, False, 'winter', False), (20230307, '2023-03-07', 2023, 3, 7, 2, 'Tuesday', 'March', 1, False, False, 'winter', False), (20230308, '2023-03-08', 2023, 3, 8, 3, 'Wednesday', 'March', 1, False, False, 'winter', False), (20230309, '2023-03-09', 2023, 3, 9, 4, 'Thursday', 'March', 1, False, False, 'winter', False), (20230310, '2023-03-10', 2023, 3, 10, 5, 'Friday', 'March', 1, False, False, 'winter', False), (20230311, '2023-03-11', 2023, 3, 11, 6, 'Saturday', 'March', 1, True, False, 'winter', False), (20230312, '2023-03-12', 2023, 3, 12, 7, 'Sunday', 'March', 1, True, False, 'winter', False), (20230313, '2023-03-13', 2023, 3, 13, 1, 'Monday', 'March', 1, False, False, 'winter', False), (20230314, '2023-03-14', 2023, 3, 14, 2, 'Tuesday', 'March', 1, False, False, 'winter', False), (20230315, '2023-03-15', 2023, 3, 15, 3, 'Wednesday', 'March', 1, False, False, 'winter', False), (20230316, '2023-03-16', 2023, 3, 16, 4, 'Thursday', 'March', 1, False, False, 'winter', False), (20230317, '2023-03-17', 2023, 3, 17, 5, 'Friday', 'March', 1, False, False, 'winter', False), (20230318, '2023-03-18', 2023, 3, 18, 6, 'Saturday', 'March', 1, True, False, 'winter', False), (20230319, '2023-03-19', 2023, 3, 19, 7, 'Sunday', 'March', 1, True, False, 'winter', False), (20230320, '2023-03-20', 2023, 3, 20, 1, 'Monday', 'March', 1, False, False, 'winter', False), (20230321, '2023-03-21', 2023, 3, 21, 2, 'Tuesday', 'March', 1, False, False, 'winter', False), (20230322, '2023-03-22', 2023, 3, 22, 3, 'Wednesday', 'March', 1, False, False, 'winter', False), (20230323, '2023-03-23', 2023, 3, 23, 4, 'Thursday', 'March', 1, False, False, 'winter', False), (20230324, '2023-03-24', 2023, 3, 24, 5, 'Friday', 'March', 1, False, False, 'winter', False), (20230325, '2023-03-25', 2023, 3, 25, 6, 'Saturday', 'March', 1, True, False, 'winter', False), (20230326, '2023-03-26', 2023, 3, 26, 7, 'Sunday', 'March', 1, True, False, 'winter', False), (20230327, '2023-03-27', 2023, 3, 27, 1, 'Monday', 'March', 1, False, False, 'winter', False), (20230328, '2023-03-28', 2023, 3, 28, 2, 'Tuesday', 'March', 1, False, False, 'winter', False), (20230329, '2023-03-29', 2023, 3, 29, 3, 'Wednesday', 'March', 1, False, False, 'winter', False), (20230330, '2023-03-30', 2023, 3, 30, 4, 'Thursday', 'March', 1, False, False, 'winter', False), (20230331, '2023-03-31', 2023, 3, 31, 5, 'Friday', 'March', 1, False, False, 'winter', False), (20230401, '2023-04-01', 2023, 4, 1, 6, 'Saturday', 'April', 2, True, False, 'summer', False), (20230402, '2023-04-02', 2023, 4, 2, 7, 'Sunday', 'April', 2, True, False, 'summer', False), (20230403, '2023-04-03', 2023, 4, 3, 1, 'Monday', 'April', 2, False, False, 'summer', False), (20230404, '2023-04-04', 2023, 4, 4, 2, 'Tuesday', 'April', 2, False, False, 'summer', False), (20230405, '2023-04-05', 2023, 4, 5, 3, 'Wednesday', 'April', 2, False, False, 'summer', False), (20230406, '2023-04-06', 2023, 4, 6, 4, 'Thursday', 'April', 2, False, False, 'summer', False), (20230407, '2023-04-07', 2023, 4, 7, 5, 'Friday', 'April', 2, False, False, 'summer', False), (20230408, '2023-04-08', 2023, 4, 8, 6, 'Saturday', 'April', 2, True, False, 'summer', False), (20230409, '2023-04-09', 2023, 4, 9, 7, 'Sunday', 'April', 2, True, False, 'summer', False), (20230410, '2023-04-10', 2023, 4, 10, 1, 'Monday', 'April', 2, False, False, 'summer', False), (20230411, '2023-04-11', 2023, 4, 11, 2, 'Tuesday', 'April', 2, False, False, 'summer', False), (20230412, '2023-04-12', 2023, 4, 12, 3, 'Wednesday', 'April', 2, False, False, 'summer', False), (20230413, '2023-04-13', 2023, 4, 13, 4, 'Thursday', 'April', 2, False, False, 'summer', False), (20230414, '2023-04-14', 2023, 4, 14, 5, 'Friday', 'April', 2, False, False, 'summer', False), (20230415, '2023-04-15', 2023, 4, 15, 6, 'Saturday', 'April', 2, True, False, 'summer', False), (20230416, '2023-04-16', 2023, 4, 16, 7, 'Sunday', 'April', 2, True, False, 'summer', False), (20230417, '2023-04-17', 2023, 4, 17, 1, 'Monday', 'April', 2, False, False, 'summer', False), (20230418, '2023-04-18', 2023, 4, 18, 2, 'Tuesday', 'April', 2, False, False, 'summer', False), (20230419, '2023-04-19', 2023, 4, 19, 3, 'Wednesday', 'April', 2, False, False, 'summer', False), (20230420, '2023-04-20', 2023, 4, 20, 4, 'Thursday', 'April', 2, False, False, 'summer', False), (20230421, '2023-04-21', 2023, 4, 21, 5, 'Friday', 'April', 2, False, False, 'summer', False), (20230422, '2023-04-22', 2023, 4, 22, 6, 'Saturday', 'April', 2, True, False, 'summer', False), (20230423, '2023-04-23', 2023, 4, 23, 7, 'Sunday', 'April', 2, True, False, 'summer', False), (20230424, '2023-04-24', 2023, 4, 24, 1, 'Monday', 'April', 2, False, False, 'summer', False), (20230425, '2023-04-25', 2023, 4, 25, 2, 'Tuesday', 'April', 2, False, False, 'summer', False), (20230426, '2023-04-26', 2023, 4, 26, 3, 'Wednesday', 'April', 2, False, False, 'summer', False), (20230427, '2023-04-27', 2023, 4, 27, 4, 'Thursday', 'April', 2, False, False, 'summer', False), (20230428, '2023-04-28', 2023, 4, 28, 5, 'Friday', 'April', 2, False, False, 'summer', False), (20230429, '2023-04-29', 2023, 4, 29, 6, 'Saturday', 'April', 2, True, False, 'summer', False), (20230430, '2023-04-30', 2023, 4, 30, 7, 'Sunday', 'April', 2, True, False, 'summer', False), (20230501, '2023-05-01', 2023, 5, 1, 1, 'Monday', 'May', 2, False, False, 'summer', False), (20230502, '2023-05-02', 2023, 5, 2, 2, 'Tuesday', 'May', 2, False, False, 'summer', False), (20230503, '2023-05-03', 2023, 5, 3, 3, 'Wednesday', 'May', 2, False, False, 'summer', False), (20230504, '2023-05-04', 2023, 5, 4, 4, 'Thursday', 'May', 2, False, False, 'summer', False), (20230505, '2023-05-05', 2023, 5, 5, 5, 'Friday', 'May', 2, False, False, 'summer', False), (20230506, '2023-05-06', 2023, 5, 6, 6, 'Saturday', 'May', 2, True, False, 'summer', False), (20230507, '2023-05-07', 2023, 5, 7, 7, 'Sunday', 'May', 2, True, False, 'summer', False), (20230508, '2023-05-08', 2023, 5, 8, 1, 'Monday', 'May', 2, False, False, 'summer', False), (20230509, '2023-05-09', 2023, 5, 9, 2, 'Tuesday', 'May', 2, False, False, 'summer', False), (20230510, '2023-05-10', 2023, 5, 10, 3, 'Wednesday', 'May', 2, False, False, 'summer', False), (20230511, '2023-05-11', 2023, 5, 11, 4, 'Thursday', 'May', 2, False, False, 'summer', False), (20230512, '2023-05-12', 2023, 5, 12, 5, 'Friday', 'May', 2, False, False, 'summer', False), (20230513, '2023-05-13', 2023, 5, 13, 6, 'Saturday', 'May', 2, True, False, 'summer', False), (20230514, '2023-05-14', 2023, 5, 14, 7, 'Sunday', 'May', 2, True, False, 'summer', False), (20230515, '2023-05-15', 2023, 5, 15, 1, 'Monday', 'May', 2, False, False, 'summer', False), (20230516, '2023-05-16', 2023, 5, 16, 2, 'Tuesday', 'May', 2, False, False, 'summer', False), (20230517, '2023-05-17', 2023, 5, 17, 3, 'Wednesday', 'May', 2, False, False, 'summer', False), (20230518, '2023-05-18', 2023, 5, 18, 4, 'Thursday', 'May', 2, False, False, 'summer', False), (20230519, '2023-05-19', 2023, 5, 19, 5, 'Friday', 'May', 2, False, False, 'summer', False), (20230520, '2023-05-20', 2023, 5, 20, 6, 'Saturday', 'May', 2, True, False, 'summer', False), (20230521, '2023-05-21', 2023, 5, 21, 7, 'Sunday', 'May', 2, True, False, 'summer', False), (20230522, '2023-05-22', 2023, 5, 22, 1, 'Monday', 'May', 2, False, False, 'summer', False), (20230523, '2023-05-23', 2023, 5, 23, 2, 'Tuesday', 'May', 2, False, False, 'summer', False), (20230524, '2023-05-24', 2023, 5, 24, 3, 'Wednesday', 'May', 2, False, False, 'summer', False), (20230525, '2023-05-25', 2023, 5, 25, 4, 'Thursday', 'May', 2, False, False, 'summer', False), (20230526, '2023-05-26', 2023, 5, 26, 5, 'Friday', 'May', 2, False, False, 'summer', False), (20230527, '2023-05-27', 2023, 5, 27, 6, 'Saturday', 'May', 2, True, False, 'summer', False), (20230528, '2023-05-28', 2023, 5, 28, 7, 'Sunday', 'May', 2, True, False, 'summer', False), (20230529, '2023-05-29', 2023, 5, 29, 1, 'Monday', 'May', 2, False, True, 'summer', False), (20230530, '2023-05-30', 2023, 5, 30, 2, 'Tuesday', 'May', 2, False, False, 'summer', False), (20230531, '2023-05-31', 2023, 5, 31, 3, 'Wednesday', 'May', 2, False, False, 'summer', False), (20230601, '2023-06-01', 2023, 6, 1, 4, 'Thursday', 'June', 2, False, False, 'summer', False), (20230602, '2023-06-02', 2023, 6, 2, 5, 'Friday', 'June', 2, False, False, 'summer', False), (20230603, '2023-06-03', 2023, 6, 3, 6, 'Saturday', 'June', 2, True, False, 'summer', False), (20230604, '2023-06-04', 2023, 6, 4, 7, 'Sunday', 'June', 2, True, False, 'summer', False), (20230605, '2023-06-05', 2023, 6, 5, 1, 'Monday', 'June', 2, False, False, 'summer', False), (20230606, '2023-06-06', 2023, 6, 6, 2, 'Tuesday', 'June', 2, False, False, 'summer', False), (20230607, '2023-06-07', 2023, 6, 7, 3, 'Wednesday', 'June', 2, False, False, 'summer', False), (20230608, '2023-06-08', 2023, 6, 8, 4, 'Thursday', 'June', 2, False, False, 'summer', False), (20230609, '2023-06-09', 2023, 6, 9, 5, 'Friday', 'June', 2, False, False, 'summer', False), (20230610, '2023-06-10', 2023, 6, 10, 6, 'Saturday', 'June', 2, True, False, 'summer', False), (20230611, '2023-06-11', 2023, 6, 11, 7, 'Sunday', 'June', 2, True, False, 'summer', False), (20230612, '2023-06-12', 2023, 6, 12, 1, 'Monday', 'June', 2, False, False, 'summer', False), (20230613, '2023-06-13', 2023, 6, 13, 2, 'Tuesday', 'June', 2, False, False, 'summer', False), (20230614, '2023-06-14', 2023, 6, 14, 3, 'Wednesday', 'June', 2, False, False, 'summer', False), (20230615, '2023-06-15', 2023, 6, 15, 4, 'Thursday', 'June', 2, False, False, 'summer', False), (20230616, '2023-06-16', 2023, 6, 16, 5, 'Friday', 'June', 2, False, False, 'summer', False), (20230617, '2023-06-17', 2023, 6, 17, 6, 'Saturday', 'June', 2, True, False, 'summer', False), (20230618, '2023-06-18', 2023, 6, 18, 7, 'Sunday', 'June', 2, True, False, 'summer', False), (20230619, '2023-06-19', 2023, 6, 19, 1, 'Monday', 'June', 2, False, True, 'summer', False), (20230620, '2023-06-20', 2023, 6, 20, 2, 'Tuesday', 'June', 2, False, False, 'summer', False), (20230621, '2023-06-21', 2023, 6, 21, 3, 'Wednesday', 'June', 2, False, False, 'summer', False), (20230622, '2023-06-22', 2023, 6, 22, 4, 'Thursday', 'June', 2, False, False, 'summer', False), (20230623, '2023-06-23', 2023, 6, 23, 5, 'Friday', 'June', 2, False, False, 'summer', False), (20230624, '2023-06-24', 2023, 6, 24, 6, 'Saturday', 'June', 2, True, False, 'summer', False), (20230625, '2023-06-25', 2023, 6, 25, 7, 'Sunday', 'June', 2, True, False, 'summer', False), (20230626, '2023-06-26', 2023, 6, 26, 1, 'Monday', 'June', 2, False, False, 'summer', False), (20230627, '2023-06-27', 2023, 6, 27, 2, 'Tuesday', 'June', 2, False, False, 'summer', False), (20230628, '2023-06-28', 2023, 6, 28, 3, 'Wednesday', 'June', 2, False, False, 'summer', False), (20230629, '2023-06-29', 2023, 6, 29, 4, 'Thursday', 'June', 2, False, False, 'summer', False), (20230630, '2023-06-30', 2023, 6, 30, 5, 'Friday', 'June', 2, False, False, 'summer', False), (20230701, '2023-07-01', 2023, 7, 1, 6, 'Saturday', 'July', 3, True, False, 'summer', False), (20230702, '2023-07-02', 2023, 7, 2, 7, 'Sunday', 'July', 3, True, False, 'summer', False), (20230703, '2023-07-03', 2023, 7, 3, 1, 'Monday', 'July', 3, False, False, 'summer', False), (20230704, '2023-07-04', 2023, 7, 4, 2, 'Tuesday', 'July', 3, False, True, 'summer', False), (20230705, '2023-07-05', 2023, 7, 5, 3, 'Wednesday', 'July', 3, False, False, 'summer', False), (20230706, '2023-07-06', 2023, 7, 6, 4, 'Thursday', 'July', 3, False, False, 'summer', False), (20230707, '2023-07-07', 2023, 7, 7, 5, 'Friday', 'July', 3, False, False, 'summer', False), (20230708, '2023-07-08', 2023, 7, 8, 6, 'Saturday', 'July', 3, True, False, 'summer', False), (20230709, '2023-07-09', 2023, 7, 9, 7, 'Sunday', 'July', 3, True, False, 'summer', False), (20230710, '2023-07-10', 2023, 7, 10, 1, 'Monday', 'July', 3, False, False, 'summer', False), (20230711, '2023-07-11', 2023, 7, 11, 2, 'Tuesday', 'July', 3, False, False, 'summer', False), (20230712, '2023-07-12', 2023, 7, 12, 3, 'Wednesday', 'July', 3, False, False, 'summer', False), (20230713, '2023-07-13', 2023, 7, 13, 4, 'Thursday', 'July', 3, False, False, 'summer', False), (20230714, '2023-07-14', 2023, 7, 14, 5, 'Friday', 'July', 3, False, False, 'summer', False), (20230715, '2023-07-15', 2023, 7, 15, 6, 'Saturday', 'July', 3, True, False, 'summer', False), (20230716, '2023-07-16', 2023, 7, 16, 7, 'Sunday', 'July', 3, True, False, 'summer', False), (20230717, '2023-07-17', 2023, 7, 17, 1, 'Monday', 'July', 3, False, False, 'summer', False), (20230718, '2023-07-18', 2023, 7, 18, 2, 'Tuesday', 'July', 3, False, False, 'summer', False), (20230719, '2023-07-19', 2023, 7, 19, 3, 'Wednesday', 'July', 3, False, False, 'summer', False), (20230720, '2023-07-20', 2023, 7, 20, 4, 'Thursday', 'July', 3, False, False, 'summer', False), (20230721, '2023-07-21', 2023, 7, 21, 5, 'Friday', 'July', 3, False, False, 'summer', False), (20230722, '2023-07-22', 2023, 7, 22, 6, 'Saturday', 'July', 3, True, False, 'summer', False), (20230723, '2023-07-23', 2023, 7, 23, 7, 'Sunday', 'July', 3, True, False, 'summer', False), (20230724, '2023-07-24', 2023, 7, 24, 1, 'Monday', 'July', 3, False, False, 'summer', False), (20230725, '2023-07-25', 2023, 7, 25, 2, 'Tuesday', 'July', 3, False, False, 'summer', False), (20230726, '2023-07-26', 2023, 7, 26, 3, 'Wednesday', 'July', 3, False, False, 'summer', False), (20230727, '2023-07-27', 2023, 7, 27, 4, 'Thursday', 'July', 3, False, False, 'summer', False), (20230728, '2023-07-28', 2023, 7, 28, 5, 'Friday', 'July', 3, False, False, 'summer', False), (20230729, '2023-07-29', 2023, 7, 29, 6, 'Saturday', 'July', 3, True, False, 'summer', False), (20230730, '2023-07-30', 2023, 7, 30, 7, 'Sunday', 'July', 3, True, False, 'summer', False), (20230731, '2023-07-31', 2023, 7, 31, 1, 'Monday', 'July', 3, False, False, 'summer', False), (20230801, '2023-08-01', 2023, 8, 1, 2, 'Tuesday', 'August', 3, False, False, 'summer', False), (20230802, '2023-08-02', 2023, 8, 2, 3, 'Wednesday', 'August', 3, False, False, 'summer', False), (20230803, '2023-08-03', 2023, 8, 3, 4, 'Thursday', 'August', 3, False, False, 'summer', False), (20230804, '2023-08-04', 2023, 8, 4, 5, 'Friday', 'August', 3, False, False, 'summer', False), (20230805, '2023-08-05', 2023, 8, 5, 6, 'Saturday', 'August', 3, True, False, 'summer', False), (20230806, '2023-08-06', 2023, 8, 6, 7, 'Sunday', 'August', 3, True, False, 'summer', False), (20230807, '2023-08-07', 2023, 8, 7, 1, 'Monday', 'August', 3, False, False, 'summer', False), (20230808, '2023-08-08', 2023, 8, 8, 2, 'Tuesday', 'August', 3, False, False, 'summer', False), (20230809, '2023-08-09', 2023, 8, 9, 3, 'Wednesday', 'August', 3, False, False, 'summer', False), (20230810, '2023-08-10', 2023, 8, 10, 4, 'Thursday', 'August', 3, False, False, 'summer', False), (20230811, '2023-08-11', 2023, 8, 11, 5, 'Friday', 'August', 3, False, False, 'summer', False), (20230812, '2023-08-12', 2023, 8, 12, 6, 'Saturday', 'August', 3, True, False, 'summer', False), (20230813, '2023-08-13', 2023, 8, 13, 7, 'Sunday', 'August', 3, True, False, 'summer', False), (20230814, '2023-08-14', 2023, 8, 14, 1, 'Monday', 'August', 3, False, False, 'summer', False), (20230815, '2023-08-15', 2023, 8, 15, 2, 'Tuesday', 'August', 3, False, False, 'summer', False), (20230816, '2023-08-16', 2023, 8, 16, 3, 'Wednesday', 'August', 3, False, False, 'summer', False), (20230817, '2023-08-17', 2023, 8, 17, 4, 'Thursday', 'August', 3, False, False, 'summer', False), (20230818, '2023-08-18', 2023, 8, 18, 5, 'Friday', 'August', 3, False, False, 'summer', False), (20230819, '2023-08-19', 2023, 8, 19, 6, 'Saturday', 'August', 3, True, False, 'summer', False), (20230820, '2023-08-20', 2023, 8, 20, 7, 'Sunday', 'August', 3, True, False, 'summer', False), (20230821, '2023-08-21', 2023, 8, 21, 1, 'Monday', 'August', 3, False, False, 'summer', False), (20230822, '2023-08-22', 2023, 8, 22, 2, 'Tuesday', 'August', 3, False, False, 'summer', False), (20230823, '2023-08-23', 2023, 8, 23, 3, 'Wednesday', 'August', 3, False, False, 'summer', False), (20230824, '2023-08-24', 2023, 8, 24, 4, 'Thursday', 'August', 3, False, False, 'summer', False), (20230825, '2023-08-25', 2023, 8, 25, 5, 'Friday', 'August', 3, False, False, 'summer', False), (20230826, '2023-08-26', 2023, 8, 26, 6, 'Saturday', 'August', 3, True, False, 'summer', False), (20230827, '2023-08-27', 2023, 8, 27, 7, 'Sunday', 'August', 3, True, False, 'summer', False), (20230828, '2023-08-28', 2023, 8, 28, 1, 'Monday', 'August', 3, False, False, 'summer', False), (20230829, '2023-08-29', 2023, 8, 29, 2, 'Tuesday', 'August', 3, False, False, 'summer', False), (20230830, '2023-08-30', 2023, 8, 30, 3, 'Wednesday', 'August', 3, False, False, 'summer', False), (20230831, '2023-08-31', 2023, 8, 31, 4, 'Thursday', 'August', 3, False, False, 'summer', False), (20230901, '2023-09-01', 2023, 9, 1, 5, 'Friday', 'September', 3, False, False, 'summer', False), (20230902, '2023-09-02', 2023, 9, 2, 6, 'Saturday', 'September', 3, True, False, 'summer', False), (20230903, '2023-09-03', 2023, 9, 3, 7, 'Sunday', 'September', 3, True, False, 'summer', False), (20230904, '2023-09-04', 2023, 9, 4, 1, 'Monday', 'September', 3, False, True, 'summer', False), (20230905, '2023-09-05', 2023, 9, 5, 2, 'Tuesday', 'September', 3, False, False, 'summer', False), (20230906, '2023-09-06', 2023, 9, 6, 3, 'Wednesday', 'September', 3, False, False, 'summer', False), (20230907, '2023-09-07', 2023, 9, 7, 4, 'Thursday', 'September', 3, False, False, 'summer', False), (20230908, '2023-09-08', 2023, 9, 8, 5, 'Friday', 'September', 3, False, False, 'summer', False), (20230909, '2023-09-09', 2023, 9, 9, 6, 'Saturday', 'September', 3, True, False, 'summer', False), (20230910, '2023-09-10', 2023, 9, 10, 7, 'Sunday', 'September', 3, True, False, 'summer', False), (20230911, '2023-09-11', 2023, 9, 11, 1, 'Monday', 'September', 3, False, False, 'summer', False), (20230912, '2023-09-12', 2023, 9, 12, 2, 'Tuesday', 'September', 3, False, False, 'summer', False), (20230913, '2023-09-13', 2023, 9, 13, 3, 'Wednesday', 'September', 3, False, False, 'summer', False), (20230914, '2023-09-14', 2023, 9, 14, 4, 'Thursday', 'September', 3, False, False, 'summer', False), (20230915, '2023-09-15', 2023, 9, 15, 5, 'Friday', 'September', 3, False, False, 'summer', False), (20230916, '2023-09-16', 2023, 9, 16, 6, 'Saturday', 'September', 3, True, False, 'summer', False), (20230917, '2023-09-17', 2023, 9, 17, 7, 'Sunday', 'September', 3, True, False, 'summer', False), (20230918, '2023-09-18', 2023, 9, 18, 1, 'Monday', 'September', 3, False, False, 'summer', False), (20230919, '2023-09-19', 2023, 9, 19, 2, 'Tuesday', 'September', 3, False, False, 'summer', False), (20230920, '2023-09-20', 2023, 9, 20, 3, 'Wednesday', 'September', 3, False, False, 'summer', False), (20230921, '2023-09-21', 2023, 9, 21, 4, 'Thursday', 'September', 3, False, False, 'summer', False), (20230922, '2023-09-22', 2023, 9, 22, 5, 'Friday', 'September', 3, False, False, 'summer', False), (20230923, '2023-09-23', 2023, 9, 23, 6, 'Saturday', 'September', 3, True, False, 'summer', False), (20230924, '2023-09-24', 2023, 9, 24, 7, 'Sunday', 'September', 3, True, False, 'summer', False), (20230925, '2023-09-25', 2023, 9, 25, 1, 'Monday', 'September', 3, False, False, 'summer', False), (20230926, '2023-09-26', 2023, 9, 26, 2, 'Tuesday', 'September', 3, False, False, 'summer', False), (20230927, '2023-09-27', 2023, 9, 27, 3, 'Wednesday', 'September', 3, False, False, 'summer', False), (20230928, '2023-09-28', 2023, 9, 28, 4, 'Thursday', 'September', 3, False, False, 'summer', False), (20230929, '2023-09-29', 2023, 9, 29, 5, 'Friday', 'September', 3, False, False, 'summer', False), (20230930, '2023-09-30', 2023, 9, 30, 6, 'Saturday', 'September', 3, True, False, 'summer', False), (20231001, '2023-10-01', 2023, 10, 1, 7, 'Sunday', 'October', 4, True, False, 'summer', False), (20231002, '2023-10-02', 2023, 10, 2, 1, 'Monday', 'October', 4, False, False, 'summer', False), (20231003, '2023-10-03', 2023, 10, 3, 2, 'Tuesday', 'October', 4, False, False, 'summer', False), (20231004, '2023-10-04', 2023, 10, 4, 3, 'Wednesday', 'October', 4, False, False, 'summer', False), (20231005, '2023-10-05', 2023, 10, 5, 4, 'Thursday', 'October', 4, False, False, 'summer', False), (20231006, '2023-10-06', 2023, 10, 6, 5, 'Friday', 'October', 4, False, False, 'summer', False), (20231007, '2023-10-07', 2023, 10, 7, 6, 'Saturday', 'October', 4, True, False, 'summer', False), (20231008, '2023-10-08', 2023, 10, 8, 7, 'Sunday', 'October', 4, True, False, 'summer', False), (20231009, '2023-10-09', 2023, 10, 9, 1, 'Monday', 'October', 4, False, True, 'summer', False), (20231010, '2023-10-10', 2023, 10, 10, 2, 'Tuesday', 'October', 4, False, False, 'summer', False), (20231011, '2023-10-11', 2023, 10, 11, 3, 'Wednesday', 'October', 4, False, False, 'summer', False), (20231012, '2023-10-12', 2023, 10, 12, 4, 'Thursday', 'October', 4, False, False, 'summer', False), (20231013, '2023-10-13', 2023, 10, 13, 5, 'Friday', 'October', 4, False, False, 'summer', False), (20231014, '2023-10-14', 2023, 10, 14, 6, 'Saturday', 'October', 4, True, False, 'summer', False), (20231015, '2023-10-15', 2023, 10, 15, 7, 'Sunday', 'October', 4, True, False, 'summer', False), (20231016, '2023-10-16', 2023, 10, 16, 1, 'Monday', 'October', 4, False, False, 'summer', False), (20231017, '2023-10-17', 2023, 10, 17, 2, 'Tuesday', 'October', 4, False, False, 'summer', False), (20231018, '2023-10-18', 2023, 10, 18, 3, 'Wednesday', 'October', 4, False, False, 'summer', False), (20231019, '2023-10-19', 2023, 10, 19, 4, 'Thursday', 'October', 4, False, False, 'summer', False), (20231020, '2023-10-20', 2023, 10, 20, 5, 'Friday', 'October', 4, False, False, 'summer', False), (20231021, '2023-10-21', 2023, 10, 21, 6, 'Saturday', 'October', 4, True, False, 'summer', False), (20231022, '2023-10-22', 2023, 10, 22, 7, 'Sunday', 'October', 4, True, False, 'summer', False), (20231023, '2023-10-23', 2023, 10, 23, 1, 'Monday', 'October', 4, False, False, 'summer', False), (20231024, '2023-10-24', 2023, 10, 24, 2, 'Tuesday', 'October', 4, False, False, 'summer', False), (20231025, '2023-10-25', 2023, 10, 25, 3, 'Wednesday', 'October', 4, False, False, 'summer', False), (20231026, '2023-10-26', 2023, 10, 26, 4, 'Thursday', 'October', 4, False, False, 'summer', False), (20231027, '2023-10-27', 2023, 10, 27, 5, 'Friday', 'October', 4, False, False, 'summer', False), (20231028, '2023-10-28', 2023, 10, 28, 6, 'Saturday', 'October', 4, True, False, 'summer', False), (20231029, '2023-10-29', 2023, 10, 29, 7, 'Sunday', 'October', 4, True, False, 'summer', False), (20231030, '2023-10-30', 2023, 10, 30, 1, 'Monday', 'October', 4, False, False, 'summer', False), (20231031, '2023-10-31', 2023, 10, 31, 2, 'Tuesday', 'October', 4, False, False, 'summer', False), (20231101, '2023-11-01', 2023, 11, 1, 3, 'Wednesday', 'November', 4, False, False, 'winter', False), (20231102, '2023-11-02', 2023, 11, 2, 4, 'Thursday', 'November', 4, False, False, 'winter', False), (20231103, '2023-11-03', 2023, 11, 3, 5, 'Friday', 'November', 4, False, False, 'winter', False), (20231104, '2023-11-04', 2023, 11, 4, 6, 'Saturday', 'November', 4, True, False, 'winter', False), (20231105, '2023-11-05', 2023, 11, 5, 7, 'Sunday', 'November', 4, True, False, 'winter', False), (20231106, '2023-11-06', 2023, 11, 6, 1, 'Monday', 'November', 4, False, False, 'winter', False), (20231107, '2023-11-07', 2023, 11, 7, 2, 'Tuesday', 'November', 4, False, False, 'winter', False), (20231108, '2023-11-08', 2023, 11, 8, 3, 'Wednesday', 'November', 4, False, False, 'winter', False), (20231109, '2023-11-09', 2023, 11, 9, 4, 'Thursday', 'November', 4, False, False, 'winter', False), (20231110, '2023-11-10', 2023, 11, 10, 5, 'Friday', 'November', 4, False, True, 'winter', False), (20231111, '2023-11-11', 2023, 11, 11, 6, 'Saturday', 'November', 4, True, False, 'winter', False), (20231112, '2023-11-12', 2023, 11, 12, 7, 'Sunday', 'November', 4, True, False, 'winter', False), (20231113, '2023-11-13', 2023, 11, 13, 1, 'Monday', 'November', 4, False, False, 'winter', False), (20231114, '2023-11-14', 2023, 11, 14, 2, 'Tuesday', 'November', 4, False, False, 'winter', False), (20231115, '2023-11-15', 2023, 11, 15, 3, 'Wednesday', 'November', 4, False, False, 'winter', False), (20231116, '2023-11-16', 2023, 11, 16, 4, 'Thursday', 'November', 4, False, False, 'winter', False), (20231117, '2023-11-17', 2023, 11, 17, 5, 'Friday', 'November', 4, False, False, 'winter', False), (20231118, '2023-11-18', 2023, 11, 18, 6, 'Saturday', 'November', 4, True, False, 'winter', False), (20231119, '2023-11-19', 2023, 11, 19, 7, 'Sunday', 'November', 4, True, False, 'winter', False), (20231120, '2023-11-20', 2023, 11, 20, 1, 'Monday', 'November', 4, False, False, 'winter', False), (20231121, '2023-11-21', 2023, 11, 21, 2, 'Tuesday', 'November', 4, False, False, 'winter', False), (20231122, '2023-11-22', 2023, 11, 22, 3, 'Wednesday', 'November', 4, False, False, 'winter', False), (20231123, '2023-11-23', 2023, 11, 23, 4, 'Thursday', 'November', 4, False, True, 'winter', False), (20231124, '2023-11-24', 2023, 11, 24, 5, 'Friday', 'November', 4, False, False, 'winter', False), (20231125, '2023-11-25', 2023, 11, 25, 6, 'Saturday', 'November', 4, True, False, 'winter', False), (20231126, '2023-11-26', 2023, 11, 26, 7, 'Sunday', 'November', 4, True, False, 'winter', False), (20231127, '2023-11-27', 2023, 11, 27, 1, 'Monday', 'November', 4, False, False, 'winter', False), (20231128, '2023-11-28', 2023, 11, 28, 2, 'Tuesday', 'November', 4, False, False, 'winter', False), (20231129, '2023-11-29', 2023, 11, 29, 3, 'Wednesday', 'November', 4, False, False, 'winter', False), (20231130, '2023-11-30', 2023, 11, 30, 4, 'Thursday', 'November', 4, False, False, 'winter', False), (20231201, '2023-12-01', 2023, 12, 1, 5, 'Friday', 'December', 4, False, False, 'winter', False), (20231202, '2023-12-02', 2023, 12, 2, 6, 'Saturday', 'December', 4, True, False, 'winter', False), (20231203, '2023-12-03', 2023, 12, 3, 7, 'Sunday', 'December', 4, True, False, 'winter', False), (20231204, '2023-12-04', 2023, 12, 4, 1, 'Monday', 'December', 4, False, False, 'winter', False), (20231205, '2023-12-05', 2023, 12, 5, 2, 'Tuesday', 'December', 4, False, False, 'winter', False), (20231206, '2023-12-06', 2023, 12, 6, 3, 'Wednesday', 'December', 4, False, False, 'winter', False), (20231207, '2023-12-07', 2023, 12, 7, 4, 'Thursday', 'December', 4, False, False, 'winter', False), (20231208, '2023-12-08', 2023, 12, 8, 5, 'Friday', 'December', 4, False, False, 'winter', False), (20231209, '2023-12-09', 2023, 12, 9, 6, 'Saturday', 'December', 4, True, False, 'winter', False), (20231210, '2023-12-10', 2023, 12, 10, 7, 'Sunday', 'December', 4, True, False, 'winter', False), (20231211, '2023-12-11', 2023, 12, 11, 1, 'Monday', 'December', 4, False, False, 'winter', False), (20231212, '2023-12-12', 2023, 12, 12, 2, 'Tuesday', 'December', 4, False, False, 'winter', False), (20231213, '2023-12-13', 2023, 12, 13, 3, 'Wednesday', 'December', 4, False, False, 'winter', False), (20231214, '2023-12-14', 2023, 12, 14, 4, 'Thursday', 'December', 4, False, False, 'winter', False), (20231215, '2023-12-15', 2023, 12, 15, 5, 'Friday', 'December', 4, False, False, 'winter', False), (20231216, '2023-12-16', 2023, 12, 16, 6, 'Saturday', 'December', 4, True, False, 'winter', False), (20231217, '2023-12-17', 2023, 12, 17, 7, 'Sunday', 'December', 4, True, False, 'winter', False), (20231218, '2023-12-18', 2023, 12, 18, 1, 'Monday', 'December', 4, False, False, 'winter', False), (20231219, '2023-12-19', 2023, 12, 19, 2, 'Tuesday', 'December', 4, False, False, 'winter', False), (20231220, '2023-12-20', 2023, 12, 20, 3, 'Wednesday', 'December', 4, False, False, 'winter', False), (20231221, '2023-12-21', 2023, 12, 21, 4, 'Thursday', 'December', 4, False, False, 'winter', False), (20231222, '2023-12-22', 2023, 12, 22, 5, 'Friday', 'December', 4, False, False, 'winter', False), (20231223, '2023-12-23', 2023, 12, 23, 6, 'Saturday', 'December', 4, True, False, 'winter', False), (20231224, '2023-12-24', 2023, 12, 24, 7, 'Sunday', 'December', 4, True, False, 'winter', False), (20231225, '2023-12-25', 2023, 12, 25, 1, 'Monday', 'December', 4, False, True, 'winter', False), (20231226, '2023-12-26', 2023, 12, 26, 2, 'Tuesday', 'December', 4, False, False, 'winter', False), (20231227, '2023-12-27', 2023, 12, 27, 3, 'Wednesday', 'December', 4, False, False, 'winter', False), (20231228, '2023-12-28', 2023, 12, 28, 4, 'Thursday', 'December', 4, False, False, 'winter', False), (20231229, '2023-12-29', 2023, 12, 29, 5, 'Friday', 'December', 4, False, False, 'winter', False), (20231230, '2023-12-30', 2023, 12, 30, 6, 'Saturday', 'December', 4, True, False, 'winter', False), (20231231, '2023-12-31', 2023, 12, 31, 7, 'Sunday', 'December', 4, True, False, 'winter', False), (20240101, '2024-01-01', 2024, 1, 1, 1, 'Monday', 'January', 1, False, True, 'winter', False), (20240102, '2024-01-02', 2024, 1, 2, 2, 'Tuesday', 'January', 1, False, False, 'winter', False), (20240103, '2024-01-03', 2024, 1, 3, 3, 'Wednesday', 'January', 1, False, False, 'winter', False), (20240104, '2024-01-04', 2024, 1, 4, 4, 'Thursday', 'January', 1, False, False, 'winter', False), (20240105, '2024-01-05', 2024, 1, 5, 5, 'Friday', 'January', 1, False, False, 'winter', False), (20240106, '2024-01-06', 2024, 1, 6, 6, 'Saturday', 'January', 1, True, False, 'winter', False), (20240107, '2024-01-07', 2024, 1, 7, 7, 'Sunday', 'January', 1, True, False, 'winter', False), (20240108, '2024-01-08', 2024, 1, 8, 1, 'Monday', 'January', 1, False, False, 'winter', False), (20240109, '2024-01-09', 2024, 1, 9, 2, 'Tuesday', 'January', 1, False, False, 'winter', False), (20240110, '2024-01-10', 2024, 1, 10, 3, 'Wednesday', 'January', 1, False, False, 'winter', False), (20240111, '2024-01-11', 2024, 1, 11, 4, 'Thursday', 'January', 1, False, False, 'winter', False), (20240112, '2024-01-12', 2024, 1, 12, 5, 'Friday', 'January', 1, False, False, 'winter', False), (20240113, '2024-01-13', 2024, 1, 13, 6, 'Saturday', 'January', 1, True, False, 'winter', False), (20240114, '2024-01-14', 2024, 1, 14, 7, 'Sunday', 'January', 1, True, False, 'winter', False), (20240115, '2024-01-15', 2024, 1, 15, 1, 'Monday', 'January', 1, False, True, 'winter', False), (20240116, '2024-01-16', 2024, 1, 16, 2, 'Tuesday', 'January', 1, False, False, 'winter', False), (20240117, '2024-01-17', 2024, 1, 17, 3, 'Wednesday', 'January', 1, False, False, 'winter', False), (20240118, '2024-01-18', 2024, 1, 18, 4, 'Thursday', 'January', 1, False, False, 'winter', False), (20240119, '2024-01-19', 2024, 1, 19, 5, 'Friday', 'January', 1, False, False, 'winter', False), (20240120, '2024-01-20', 2024, 1, 20, 6, 'Saturday', 'January', 1, True, False, 'winter', False), (20240121, '2024-01-21', 2024, 1, 21, 7, 'Sunday', 'January', 1, True, False, 'winter', False), (20240122, '2024-01-22', 2024, 1, 22, 1, 'Monday', 'January', 1, False, False, 'winter', False), (20240123, '2024-01-23', 2024, 1, 23, 2, 'Tuesday', 'January', 1, False, False, 'winter', False), (20240124, '2024-01-24', 2024, 1, 24, 3, 'Wednesday', 'January', 1, False, False, 'winter', False), (20240125, '2024-01-25', 2024, 1, 25, 4, 'Thursday', 'January', 1, False, False, 'winter', False), (20240126, '2024-01-26', 2024, 1, 26, 5, 'Friday', 'January', 1, False, False, 'winter', False), (20240127, '2024-01-27', 2024, 1, 27, 6, 'Saturday', 'January', 1, True, False, 'winter', False), (20240128, '2024-01-28', 2024, 1, 28, 7, 'Sunday', 'January', 1, True, False, 'winter', False), (20240129, '2024-01-29', 2024, 1, 29, 1, 'Monday', 'January', 1, False, False, 'winter', False), (20240130, '2024-01-30', 2024, 1, 30, 2, 'Tuesday', 'January', 1, False, False, 'winter', False), (20240131, '2024-01-31', 2024, 1, 31, 3, 'Wednesday', 'January', 1, False, False, 'winter', False), (20240201, '2024-02-01', 2024, 2, 1, 4, 'Thursday', 'February', 1, False, False, 'winter', False), (20240202, '2024-02-02', 2024, 2, 2, 5, 'Friday', 'February', 1, False, False, 'winter', False), (20240203, '2024-02-03', 2024, 2, 3, 6, 'Saturday', 'February', 1, True, False, 'winter', False), (20240204, '2024-02-04', 2024, 2, 4, 7, 'Sunday', 'February', 1, True, False, 'winter', False), (20240205, '2024-02-05', 2024, 2, 5, 1, 'Monday', 'February', 1, False, False, 'winter', False), (20240206, '2024-02-06', 2024, 2, 6, 2, 'Tuesday', 'February', 1, False, False, 'winter', False), (20240207, '2024-02-07', 2024, 2, 7, 3, 'Wednesday', 'February', 1, False, False, 'winter', False), (20240208, '2024-02-08', 2024, 2, 8, 4, 'Thursday', 'February', 1, False, False, 'winter', False), (20240209, '2024-02-09', 2024, 2, 9, 5, 'Friday', 'February', 1, False, False, 'winter', False), (20240210, '2024-02-10', 2024, 2, 10, 6, 'Saturday', 'February', 1, True, False, 'winter', False), (20240211, '2024-02-11', 2024, 2, 11, 7, 'Sunday', 'February', 1, True, False, 'winter', False), (20240212, '2024-02-12', 2024, 2, 12, 1, 'Monday', 'February', 1, False, False, 'winter', False), (20240213, '2024-02-13', 2024, 2, 13, 2, 'Tuesday', 'February', 1, False, False, 'winter', False), (20240214, '2024-02-14', 2024, 2, 14, 3, 'Wednesday', 'February', 1, False, False, 'winter', False), (20240215, '2024-02-15', 2024, 2, 15, 4, 'Thursday', 'February', 1, False, False, 'winter', False), (20240216, '2024-02-16', 2024, 2, 16, 5, 'Friday', 'February', 1, False, False, 'winter', False), (20240217, '2024-02-17', 2024, 2, 17, 6, 'Saturday', 'February', 1, True, False, 'winter', False), (20240218, '2024-02-18', 2024, 2, 18, 7, 'Sunday', 'February', 1, True, False, 'winter', False), (20240219, '2024-02-19', 2024, 2, 19, 1, 'Monday', 'February', 1, False, True, 'winter', False), (20240220, '2024-02-20', 2024, 2, 20, 2, 'Tuesday', 'February', 1, False, False, 'winter', False), (20240221, '2024-02-21', 2024, 2, 21, 3, 'Wednesday', 'February', 1, False, False, 'winter', False), (20240222, '2024-02-22', 2024, 2, 22, 4, 'Thursday', 'February', 1, False, False, 'winter', False), (20240223, '2024-02-23', 2024, 2, 23, 5, 'Friday', 'February', 1, False, False, 'winter', False), (20240224, '2024-02-24', 2024, 2, 24, 6, 'Saturday', 'February', 1, True, False, 'winter', False), (20240225, '2024-02-25', 2024, 2, 25, 7, 'Sunday', 'February', 1, True, False, 'winter', False), (20240226, '2024-02-26', 2024, 2, 26, 1, 'Monday', 'February', 1, False, False, 'winter', False), (20240227, '2024-02-27', 2024, 2, 27, 2, 'Tuesday', 'February', 1, False, False, 'winter', False), (20240228, '2024-02-28', 2024, 2, 28, 3, 'Wednesday', 'February', 1, False, False, 'winter', False), (20240229, '2024-02-29', 2024, 2, 29, 4, 'Thursday', 'February', 1, False, False, 'winter', False), (20240301, '2024-03-01', 2024, 3, 1, 5, 'Friday', 'March', 1, False, False, 'winter', False), (20240302, '2024-03-02', 2024, 3, 2, 6, 'Saturday', 'March', 1, True, False, 'winter', False), (20240303, '2024-03-03', 2024, 3, 3, 7, 'Sunday', 'March', 1, True, False, 'winter', False), (20240304, '2024-03-04', 2024, 3, 4, 1, 'Monday', 'March', 1, False, False, 'winter', False), (20240305, '2024-03-05', 2024, 3, 5, 2, 'Tuesday', 'March', 1, False, False, 'winter', False), (20240306, '2024-03-06', 2024, 3, 6, 3, 'Wednesday', 'March', 1, False, False, 'winter', False), (20240307, '2024-03-07', 2024, 3, 7, 4, 'Thursday', 'March', 1, False, False, 'winter', False), (20240308, '2024-03-08', 2024, 3, 8, 5, 'Friday', 'March', 1, False, False, 'winter', False), (20240309, '2024-03-09', 2024, 3, 9, 6, 'Saturday', 'March', 1, True, False, 'winter', False), (20240310, '2024-03-10', 2024, 3, 10, 7, 'Sunday', 'March', 1, True, False, 'winter', False), (20240311, '2024-03-11', 2024, 3, 11, 1, 'Monday', 'March', 1, False, False, 'winter', False), (20240312, '2024-03-12', 2024, 3, 12, 2, 'Tuesday', 'March', 1, False, False, 'winter', False), (20240313, '2024-03-13', 2024, 3, 13, 3, 'Wednesday', 'March', 1, False, False, 'winter', False), (20240314, '2024-03-14', 2024, 3, 14, 4, 'Thursday', 'March', 1, False, False, 'winter', False), (20240315, '2024-03-15', 2024, 3, 15, 5, 'Friday', 'March', 1, False, False, 'winter', False), (20240316, '2024-03-16', 2024, 3, 16, 6, 'Saturday', 'March', 1, True, False, 'winter', False), (20240317, '2024-03-17', 2024, 3, 17, 7, 'Sunday', 'March', 1, True, False, 'winter', False), (20240318, '2024-03-18', 2024, 3, 18, 1, 'Monday', 'March', 1, False, False, 'winter', False), (20240319, '2024-03-19', 2024, 3, 19, 2, 'Tuesday', 'March', 1, False, False, 'winter', False), (20240320, '2024-03-20', 2024, 3, 20, 3, 'Wednesday', 'March', 1, False, False, 'winter', False), (20240321, '2024-03-21', 2024, 3, 21, 4, 'Thursday', 'March', 1, False, False, 'winter', False), (20240322, '2024-03-22', 2024, 3, 22, 5, 'Friday', 'March', 1, False, False, 'winter', False), (20240323, '2024-03-23', 2024, 3, 23, 6, 'Saturday', 'March', 1, True, False, 'winter', False), (20240324, '2024-03-24', 2024, 3, 24, 7, 'Sunday', 'March', 1, True, False, 'winter', False), (20240325, '2024-03-25', 2024, 3, 25, 1, 'Monday', 'March', 1, False, False, 'winter', False), (20240326, '2024-03-26', 2024, 3, 26, 2, 'Tuesday', 'March', 1, False, False, 'winter', False), (20240327, '2024-03-27', 2024, 3, 27, 3, 'Wednesday', 'March', 1, False, False, 'winter', False), (20240328, '2024-03-28', 2024, 3, 28, 4, 'Thursday', 'March', 1, False, False, 'winter', False), (20240329, '2024-03-29', 2024, 3, 29, 5, 'Friday', 'March', 1, False, False, 'winter', False), (20240330, '2024-03-30', 2024, 3, 30, 6, 'Saturday', 'March', 1, True, False, 'winter', False), (20240331, '2024-03-31', 2024, 3, 31, 7, 'Sunday', 'March', 1, True, False, 'winter', False), (20240401, '2024-04-01', 2024, 4, 1, 1, 'Monday', 'April', 2, False, False, 'summer', False), (20240402, '2024-04-02', 2024, 4, 2, 2, 'Tuesday', 'April', 2, False, False, 'summer', False), (20240403, '2024-04-03', 2024, 4, 3, 3, 'Wednesday', 'April', 2, False, False, 'summer', False), (20240404, '2024-04-04', 2024, 4, 4, 4, 'Thursday', 'April', 2, False, False, 'summer', False), (20240405, '2024-04-05', 2024, 4, 5, 5, 'Friday', 'April', 2, False, False, 'summer', False), (20240406, '2024-04-06', 2024, 4, 6, 6, 'Saturday', 'April', 2, True, False, 'summer', False), (20240407, '2024-04-07', 2024, 4, 7, 7, 'Sunday', 'April', 2, True, False, 'summer', False), (20240408, '2024-04-08', 2024, 4, 8, 1, 'Monday', 'April', 2, False, False, 'summer', False), (20240409, '2024-04-09', 2024, 4, 9, 2, 'Tuesday', 'April', 2, False, False, 'summer', False), (20240410, '2024-04-10', 2024, 4, 10, 3, 'Wednesday', 'April', 2, False, False, 'summer', False), (20240411, '2024-04-11', 2024, 4, 11, 4, 'Thursday', 'April', 2, False, False, 'summer', False), (20240412, '2024-04-12', 2024, 4, 12, 5, 'Friday', 'April', 2, False, False, 'summer', False), (20240413, '2024-04-13', 2024, 4, 13, 6, 'Saturday', 'April', 2, True, False, 'summer', False), (20240414, '2024-04-14', 2024, 4, 14, 7, 'Sunday', 'April', 2, True, False, 'summer', False), (20240415, '2024-04-15', 2024, 4, 15, 1, 'Monday', 'April', 2, False, False, 'summer', False), (20240416, '2024-04-16', 2024, 4, 16, 2, 'Tuesday', 'April', 2, False, False, 'summer', False), (20240417, '2024-04-17', 2024, 4, 17, 3, 'Wednesday', 'April', 2, False, False, 'summer', False), (20240418, '2024-04-18', 2024, 4, 18, 4, 'Thursday', 'April', 2, False, False, 'summer', False), (20240419, '2024-04-19', 2024, 4, 19, 5, 'Friday', 'April', 2, False, False, 'summer', False), (20240420, '2024-04-20', 2024, 4, 20, 6, 'Saturday', 'April', 2, True, False, 'summer', False), (20240421, '2024-04-21', 2024, 4, 21, 7, 'Sunday', 'April', 2, True, False, 'summer', False), (20240422, '2024-04-22', 2024, 4, 22, 1, 'Monday', 'April', 2, False, False, 'summer', False), (20240423, '2024-04-23', 2024, 4, 23, 2, 'Tuesday', 'April', 2, False, False, 'summer', False), (20240424, '2024-04-24', 2024, 4, 24, 3, 'Wednesday', 'April', 2, False, False, 'summer', False), (20240425, '2024-04-25', 2024, 4, 25, 4, 'Thursday', 'April', 2, False, False, 'summer', False), (20240426, '2024-04-26', 2024, 4, 26, 5, 'Friday', 'April', 2, False, False, 'summer', False), (20240427, '2024-04-27', 2024, 4, 27, 6, 'Saturday', 'April', 2, True, False, 'summer', False), (20240428, '2024-04-28', 2024, 4, 28, 7, 'Sunday', 'April', 2, True, False, 'summer', False), (20240429, '2024-04-29', 2024, 4, 29, 1, 'Monday', 'April', 2, False, False, 'summer', False), (20240430, '2024-04-30', 2024, 4, 30, 2, 'Tuesday', 'April', 2, False, False, 'summer', False), (20240501, '2024-05-01', 2024, 5, 1, 3, 'Wednesday', 'May', 2, False, False, 'summer', False), (20240502, '2024-05-02', 2024, 5, 2, 4, 'Thursday', 'May', 2, False, False, 'summer', False), (20240503, '2024-05-03', 2024, 5, 3, 5, 'Friday', 'May', 2, False, False, 'summer', False), (20240504, '2024-05-04', 2024, 5, 4, 6, 'Saturday', 'May', 2, True, False, 'summer', False), (20240505, '2024-05-05', 2024, 5, 5, 7, 'Sunday', 'May', 2, True, False, 'summer', False), (20240506, '2024-05-06', 2024, 5, 6, 1, 'Monday', 'May', 2, False, False, 'summer', False), (20240507, '2024-05-07', 2024, 5, 7, 2, 'Tuesday', 'May', 2, False, False, 'summer', False), (20240508, '2024-05-08', 2024, 5, 8, 3, 'Wednesday', 'May', 2, False, False, 'summer', False), (20240509, '2024-05-09', 2024, 5, 9, 4, 'Thursday', 'May', 2, False, False, 'summer', False), (20240510, '2024-05-10', 2024, 5, 10, 5, 'Friday', 'May', 2, False, False, 'summer', False), (20240511, '2024-05-11', 2024, 5, 11, 6, 'Saturday', 'May', 2, True, False, 'summer', False), (20240512, '2024-05-12', 2024, 5, 12, 7, 'Sunday', 'May', 2, True, False, 'summer', False), (20240513, '2024-05-13', 2024, 5, 13, 1, 'Monday', 'May', 2, False, False, 'summer', False), (20240514, '2024-05-14', 2024, 5, 14, 2, 'Tuesday', 'May', 2, False, False, 'summer', False), (20240515, '2024-05-15', 2024, 5, 15, 3, 'Wednesday', 'May', 2, False, False, 'summer', False), (20240516, '2024-05-16', 2024, 5, 16, 4, 'Thursday', 'May', 2, False, False, 'summer', False), (20240517, '2024-05-17', 2024, 5, 17, 5, 'Friday', 'May', 2, False, False, 'summer', False), (20240518, '2024-05-18', 2024, 5, 18, 6, 'Saturday', 'May', 2, True, False, 'summer', False), (20240519, '2024-05-19', 2024, 5, 19, 7, 'Sunday', 'May', 2, True, False, 'summer', False), (20240520, '2024-05-20', 2024, 5, 20, 1, 'Monday', 'May', 2, False, False, 'summer', False), (20240521, '2024-05-21', 2024, 5, 21, 2, 'Tuesday', 'May', 2, False, False, 'summer', False), (20240522, '2024-05-22', 2024, 5, 22, 3, 'Wednesday', 'May', 2, False, False, 'summer', False), (20240523, '2024-05-23', 2024, 5, 23, 4, 'Thursday', 'May', 2, False, False, 'summer', False), (20240524, '2024-05-24', 2024, 5, 24, 5, 'Friday', 'May', 2, False, False, 'summer', False), (20240525, '2024-05-25', 2024, 5, 25, 6, 'Saturday', 'May', 2, True, False, 'summer', False), (20240526, '2024-05-26', 2024, 5, 26, 7, 'Sunday', 'May', 2, True, False, 'summer', False), (20240527, '2024-05-27', 2024, 5, 27, 1, 'Monday', 'May', 2, False, True, 'summer', False), (20240528, '2024-05-28', 2024, 5, 28, 2, 'Tuesday', 'May', 2, False, False, 'summer', False), (20240529, '2024-05-29', 2024, 5, 29, 3, 'Wednesday', 'May', 2, False, False, 'summer', False), (20240530, '2024-05-30', 2024, 5, 30, 4, 'Thursday', 'May', 2, False, False, 'summer', False), (20240531, '2024-05-31', 2024, 5, 31, 5, 'Friday', 'May', 2, False, False, 'summer', False), (20240601, '2024-06-01', 2024, 6, 1, 6, 'Saturday', 'June', 2, True, False, 'summer', False), (20240602, '2024-06-02', 2024, 6, 2, 7, 'Sunday', 'June', 2, True, False, 'summer', False), (20240603, '2024-06-03', 2024, 6, 3, 1, 'Monday', 'June', 2, False, False, 'summer', False), (20240604, '2024-06-04', 2024, 6, 4, 2, 'Tuesday', 'June', 2, False, False, 'summer', False), (20240605, '2024-06-05', 2024, 6, 5, 3, 'Wednesday', 'June', 2, False, False, 'summer', False), (20240606, '2024-06-06', 2024, 6, 6, 4, 'Thursday', 'June', 2, False, False, 'summer', False), (20240607, '2024-06-07', 2024, 6, 7, 5, 'Friday', 'June', 2, False, False, 'summer', False), (20240608, '2024-06-08', 2024, 6, 8, 6, 'Saturday', 'June', 2, True, False, 'summer', False), (20240609, '2024-06-09', 2024, 6, 9, 7, 'Sunday', 'June', 2, True, False, 'summer', False), (20240610, '2024-06-10', 2024, 6, 10, 1, 'Monday', 'June', 2, False, False, 'summer', False), (20240611, '2024-06-11', 2024, 6, 11, 2, 'Tuesday', 'June', 2, False, False, 'summer', False), (20240612, '2024-06-12', 2024, 6, 12, 3, 'Wednesday', 'June', 2, False, False, 'summer', False), (20240613, '2024-06-13', 2024, 6, 13, 4, 'Thursday', 'June', 2, False, False, 'summer', False), (20240614, '2024-06-14', 2024, 6, 14, 5, 'Friday', 'June', 2, False, False, 'summer', False), (20240615, '2024-06-15', 2024, 6, 15, 6, 'Saturday', 'June', 2, True, False, 'summer', False), (20240616, '2024-06-16', 2024, 6, 16, 7, 'Sunday', 'June', 2, True, False, 'summer', False), (20240617, '2024-06-17', 2024, 6, 17, 1, 'Monday', 'June', 2, False, False, 'summer', False), (20240618, '2024-06-18', 2024, 6, 18, 2, 'Tuesday', 'June', 2, False, False, 'summer', False), (20240619, '2024-06-19', 2024, 6, 19, 3, 'Wednesday', 'June', 2, False, True, 'summer', False), (20240620, '2024-06-20', 2024, 6, 20, 4, 'Thursday', 'June', 2, False, False, 'summer', False), (20240621, '2024-06-21', 2024, 6, 21, 5, 'Friday', 'June', 2, False, False, 'summer', False), (20240622, '2024-06-22', 2024, 6, 22, 6, 'Saturday', 'June', 2, True, False, 'summer', False), (20240623, '2024-06-23', 2024, 6, 23, 7, 'Sunday', 'June', 2, True, False, 'summer', False), (20240624, '2024-06-24', 2024, 6, 24, 1, 'Monday', 'June', 2, False, False, 'summer', False), (20240625, '2024-06-25', 2024, 6, 25, 2, 'Tuesday', 'June', 2, False, False, 'summer', False), (20240626, '2024-06-26', 2024, 6, 26, 3, 'Wednesday', 'June', 2, False, False, 'summer', False), (20240627, '2024-06-27', 2024, 6, 27, 4, 'Thursday', 'June', 2, False, False, 'summer', False), (20240628, '2024-06-28', 2024, 6, 28, 5, 'Friday', 'June', 2, False, False, 'summer', False), (20240629, '2024-06-29', 2024, 6, 29, 6, 'Saturday', 'June', 2, True, False, 'summer', False), (20240630, '2024-06-30', 2024, 6, 30, 7, 'Sunday', 'June', 2, True, False, 'summer', False), (20240701, '2024-07-01', 2024, 7, 1, 1, 'Monday', 'July', 3, False, False, 'summer', False), (20240702, '2024-07-02', 2024, 7, 2, 2, 'Tuesday', 'July', 3, False, False, 'summer', False), (20240703, '2024-07-03', 2024, 7, 3, 3, 'Wednesday', 'July', 3, False, False, 'summer', False), (20240704, '2024-07-04', 2024, 7, 4, 4, 'Thursday', 'July', 3, False, True, 'summer', False), (20240705, '2024-07-05', 2024, 7, 5, 5, 'Friday', 'July', 3, False, False, 'summer', False), (20240706, '2024-07-06', 2024, 7, 6, 6, 'Saturday', 'July', 3, True, False, 'summer', False), (20240707, '2024-07-07', 2024, 7, 7, 7, 'Sunday', 'July', 3, True, False, 'summer', False), (20240708, '2024-07-08', 2024, 7, 8, 1, 'Monday', 'July', 3, False, False, 'summer', False), (20240709, '2024-07-09', 2024, 7, 9, 2, 'Tuesday', 'July', 3, False, False, 'summer', False), (20240710, '2024-07-10', 2024, 7, 10, 3, 'Wednesday', 'July', 3, False, False, 'summer', False), (20240711, '2024-07-11', 2024, 7, 11, 4, 'Thursday', 'July', 3, False, False, 'summer', False), (20240712, '2024-07-12', 2024, 7, 12, 5, 'Friday', 'July', 3, False, False, 'summer', False), (20240713, '2024-07-13', 2024, 7, 13, 6, 'Saturday', 'July', 3, True, False, 'summer', False), (20240714, '2024-07-14', 2024, 7, 14, 7, 'Sunday', 'July', 3, True, False, 'summer', False), (20240715, '2024-07-15', 2024, 7, 15, 1, 'Monday', 'July', 3, False, False, 'summer', False), (20240716, '2024-07-16', 2024, 7, 16, 2, 'Tuesday', 'July', 3, False, False, 'summer', False), (20240717, '2024-07-17', 2024, 7, 17, 3, 'Wednesday', 'July', 3, False, False, 'summer', False), (20240718, '2024-07-18', 2024, 7, 18, 4, 'Thursday', 'July', 3, False, False, 'summer', False), (20240719, '2024-07-19', 2024, 7, 19, 5, 'Friday', 'July', 3, False, False, 'summer', False), (20240720, '2024-07-20', 2024, 7, 20, 6, 'Saturday', 'July', 3, True, False, 'summer', False), (20240721, '2024-07-21', 2024, 7, 21, 7, 'Sunday', 'July', 3, True, False, 'summer', False), (20240722, '2024-07-22', 2024, 7, 22, 1, 'Monday', 'July', 3, False, False, 'summer', False), (20240723, '2024-07-23', 2024, 7, 23, 2, 'Tuesday', 'July', 3, False, False, 'summer', False), (20240724, '2024-07-24', 2024, 7, 24, 3, 'Wednesday', 'July', 3, False, False, 'summer', False), (20240725, '2024-07-25', 2024, 7, 25, 4, 'Thursday', 'July', 3, False, False, 'summer', False), (20240726, '2024-07-26', 2024, 7, 26, 5, 'Friday', 'July', 3, False, False, 'summer', False), (20240727, '2024-07-27', 2024, 7, 27, 6, 'Saturday', 'July', 3, True, False, 'summer', False), (20240728, '2024-07-28', 2024, 7, 28, 7, 'Sunday', 'July', 3, True, False, 'summer', False), (20240729, '2024-07-29', 2024, 7, 29, 1, 'Monday', 'July', 3, False, False, 'summer', False), (20240730, '2024-07-30', 2024, 7, 30, 2, 'Tuesday', 'July', 3, False, False, 'summer', False), (20240731, '2024-07-31', 2024, 7, 31, 3, 'Wednesday', 'July', 3, False, False, 'summer', False), (20240801, '2024-08-01', 2024, 8, 1, 4, 'Thursday', 'August', 3, False, False, 'summer', False), (20240802, '2024-08-02', 2024, 8, 2, 5, 'Friday', 'August', 3, False, False, 'summer', False), (20240803, '2024-08-03', 2024, 8, 3, 6, 'Saturday', 'August', 3, True, False, 'summer', False), (20240804, '2024-08-04', 2024, 8, 4, 7, 'Sunday', 'August', 3, True, False, 'summer', False), (20240805, '2024-08-05', 2024, 8, 5, 1, 'Monday', 'August', 3, False, False, 'summer', False), (20240806, '2024-08-06', 2024, 8, 6, 2, 'Tuesday', 'August', 3, False, False, 'summer', False), (20240807, '2024-08-07', 2024, 8, 7, 3, 'Wednesday', 'August', 3, False, False, 'summer', False), (20240808, '2024-08-08', 2024, 8, 8, 4, 'Thursday', 'August', 3, False, False, 'summer', False), (20240809, '2024-08-09', 2024, 8, 9, 5, 'Friday', 'August', 3, False, False, 'summer', False), (20240810, '2024-08-10', 2024, 8, 10, 6, 'Saturday', 'August', 3, True, False, 'summer', False), (20240811, '2024-08-11', 2024, 8, 11, 7, 'Sunday', 'August', 3, True, False, 'summer', False), (20240812, '2024-08-12', 2024, 8, 12, 1, 'Monday', 'August', 3, False, False, 'summer', False), (20240813, '2024-08-13', 2024, 8, 13, 2, 'Tuesday', 'August', 3, False, False, 'summer', False), (20240814, '2024-08-14', 2024, 8, 14, 3, 'Wednesday', 'August', 3, False, False, 'summer', False), (20240815, '2024-08-15', 2024, 8, 15, 4, 'Thursday', 'August', 3, False, False, 'summer', False), (20240816, '2024-08-16', 2024, 8, 16, 5, 'Friday', 'August', 3, False, False, 'summer', False), (20240817, '2024-08-17', 2024, 8, 17, 6, 'Saturday', 'August', 3, True, False, 'summer', False), (20240818, '2024-08-18', 2024, 8, 18, 7, 'Sunday', 'August', 3, True, False, 'summer', False), (20240819, '2024-08-19', 2024, 8, 19, 1, 'Monday', 'August', 3, False, False, 'summer', False), (20240820, '2024-08-20', 2024, 8, 20, 2, 'Tuesday', 'August', 3, False, False, 'summer', False), (20240821, '2024-08-21', 2024, 8, 21, 3, 'Wednesday', 'August', 3, False, False, 'summer', False), (20240822, '2024-08-22', 2024, 8, 22, 4, 'Thursday', 'August', 3, False, False, 'summer', False), (20240823, '2024-08-23', 2024, 8, 23, 5, 'Friday', 'August', 3, False, False, 'summer', False), (20240824, '2024-08-24', 2024, 8, 24, 6, 'Saturday', 'August', 3, True, False, 'summer', False), (20240825, '2024-08-25', 2024, 8, 25, 7, 'Sunday', 'August', 3, True, False, 'summer', False), (20240826, '2024-08-26', 2024, 8, 26, 1, 'Monday', 'August', 3, False, False, 'summer', False), (20240827, '2024-08-27', 2024, 8, 27, 2, 'Tuesday', 'August', 3, False, False, 'summer', False), (20240828, '2024-08-28', 2024, 8, 28, 3, 'Wednesday', 'August', 3, False, False, 'summer', False), (20240829, '2024-08-29', 2024, 8, 29, 4, 'Thursday', 'August', 3, False, False, 'summer', False), (20240830, '2024-08-30', 2024, 8, 30, 5, 'Friday', 'August', 3, False, False, 'summer', False), (20240831, '2024-08-31', 2024, 8, 31, 6, 'Saturday', 'August', 3, True, False, 'summer', False), (20240901, '2024-09-01', 2024, 9, 1, 7, 'Sunday', 'September', 3, True, False, 'summer', False), (20240902, '2024-09-02', 2024, 9, 2, 1, 'Monday', 'September', 3, False, True, 'summer', False), (20240903, '2024-09-03', 2024, 9, 3, 2, 'Tuesday', 'September', 3, False, False, 'summer', False), (20240904, '2024-09-04', 2024, 9, 4, 3, 'Wednesday', 'September', 3, False, False, 'summer', False), (20240905, '2024-09-05', 2024, 9, 5, 4, 'Thursday', 'September', 3, False, False, 'summer', False), (20240906, '2024-09-06', 2024, 9, 6, 5, 'Friday', 'September', 3, False, False, 'summer', False), (20240907, '2024-09-07', 2024, 9, 7, 6, 'Saturday', 'September', 3, True, False, 'summer', False), (20240908, '2024-09-08', 2024, 9, 8, 7, 'Sunday', 'September', 3, True, False, 'summer', False), (20240909, '2024-09-09', 2024, 9, 9, 1, 'Monday', 'September', 3, False, False, 'summer', False), (20240910, '2024-09-10', 2024, 9, 10, 2, 'Tuesday', 'September', 3, False, False, 'summer', False), (20240911, '2024-09-11', 2024, 9, 11, 3, 'Wednesday', 'September', 3, False, False, 'summer', False), (20240912, '2024-09-12', 2024, 9, 12, 4, 'Thursday', 'September', 3, False, False, 'summer', False), (20240913, '2024-09-13', 2024, 9, 13, 5, 'Friday', 'September', 3, False, False, 'summer', False), (20240914, '2024-09-14', 2024, 9, 14, 6, 'Saturday', 'September', 3, True, False, 'summer', False), (20240915, '2024-09-15', 2024, 9, 15, 7, 'Sunday', 'September', 3, True, False, 'summer', False), (20240916, '2024-09-16', 2024, 9, 16, 1, 'Monday', 'September', 3, False, False, 'summer', False), (20240917, '2024-09-17', 2024, 9, 17, 2, 'Tuesday', 'September', 3, False, False, 'summer', False), (20240918, '2024-09-18', 2024, 9, 18, 3, 'Wednesday', 'September', 3, False, False, 'summer', False), (20240919, '2024-09-19', 2024, 9, 19, 4, 'Thursday', 'September', 3, False, False, 'summer', False), (20240920, '2024-09-20', 2024, 9, 20, 5, 'Friday', 'September', 3, False, False, 'summer', False), (20240921, '2024-09-21', 2024, 9, 21, 6, 'Saturday', 'September', 3, True, False, 'summer', False), (20240922, '2024-09-22', 2024, 9, 22, 7, 'Sunday', 'September', 3, True, False, 'summer', False), (20240923, '2024-09-23', 2024, 9, 23, 1, 'Monday', 'September', 3, False, False, 'summer', False), (20240924, '2024-09-24', 2024, 9, 24, 2, 'Tuesday', 'September', 3, False, False, 'summer', False), (20240925, '2024-09-25', 2024, 9, 25, 3, 'Wednesday', 'September', 3, False, False, 'summer', False), (20240926, '2024-09-26', 2024, 9, 26, 4, 'Thursday', 'September', 3, False, False, 'summer', False), (20240927, '2024-09-27', 2024, 9, 27, 5, 'Friday', 'September', 3, False, False, 'summer', False), (20240928, '2024-09-28', 2024, 9, 28, 6, 'Saturday', 'September', 3, True, False, 'summer', False), (20240929, '2024-09-29', 2024, 9, 29, 7, 'Sunday', 'September', 3, True, False, 'summer', False), (20240930, '2024-09-30', 2024, 9, 30, 1, 'Monday', 'September', 3, False, False, 'summer', False), (20241001, '2024-10-01', 2024, 10, 1, 2, 'Tuesday', 'October', 4, False, False, 'summer', False), (20241002, '2024-10-02', 2024, 10, 2, 3, 'Wednesday', 'October', 4, False, False, 'summer', False), (20241003, '2024-10-03', 2024, 10, 3, 4, 'Thursday', 'October', 4, False, False, 'summer', False), (20241004, '2024-10-04', 2024, 10, 4, 5, 'Friday', 'October', 4, False, False, 'summer', False), (20241005, '2024-10-05', 2024, 10, 5, 6, 'Saturday', 'October', 4, True, False, 'summer', False), (20241006, '2024-10-06', 2024, 10, 6, 7, 'Sunday', 'October', 4, True, False, 'summer', False), (20241007, '2024-10-07', 2024, 10, 7, 1, 'Monday', 'October', 4, False, False, 'summer', False), (20241008, '2024-10-08', 2024, 10, 8, 2, 'Tuesday', 'October', 4, False, False, 'summer', False), (20241009, '2024-10-09', 2024, 10, 9, 3, 'Wednesday', 'October', 4, False, False, 'summer', False), (20241010, '2024-10-10', 2024, 10, 10, 4, 'Thursday', 'October', 4, False, False, 'summer', False), (20241011, '2024-10-11', 2024, 10, 11, 5, 'Friday', 'October', 4, False, False, 'summer', False), (20241012, '2024-10-12', 2024, 10, 12, 6, 'Saturday', 'October', 4, True, False, 'summer', False), (20241013, '2024-10-13', 2024, 10, 13, 7, 'Sunday', 'October', 4, True, False, 'summer', False), (20241014, '2024-10-14', 2024, 10, 14, 1, 'Monday', 'October', 4, False, True, 'summer', False), (20241015, '2024-10-15', 2024, 10, 15, 2, 'Tuesday', 'October', 4, False, False, 'summer', False), (20241016, '2024-10-16', 2024, 10, 16, 3, 'Wednesday', 'October', 4, False, False, 'summer', False), (20241017, '2024-10-17', 2024, 10, 17, 4, 'Thursday', 'October', 4, False, False, 'summer', False), (20241018, '2024-10-18', 2024, 10, 18, 5, 'Friday', 'October', 4, False, False, 'summer', False), (20241019, '2024-10-19', 2024, 10, 19, 6, 'Saturday', 'October', 4, True, False, 'summer', False), (20241020, '2024-10-20', 2024, 10, 20, 7, 'Sunday', 'October', 4, True, False, 'summer', False), (20241021, '2024-10-21', 2024, 10, 21, 1, 'Monday', 'October', 4, False, False, 'summer', False), (20241022, '2024-10-22', 2024, 10, 22, 2, 'Tuesday', 'October', 4, False, False, 'summer', False), (20241023, '2024-10-23', 2024, 10, 23, 3, 'Wednesday', 'October', 4, False, False, 'summer', False), (20241024, '2024-10-24', 2024, 10, 24, 4, 'Thursday', 'October', 4, False, False, 'summer', False), (20241025, '2024-10-25', 2024, 10, 25, 5, 'Friday', 'October', 4, False, False, 'summer', False), (20241026, '2024-10-26', 2024, 10, 26, 6, 'Saturday', 'October', 4, True, False, 'summer', False), (20241027, '2024-10-27', 2024, 10, 27, 7, 'Sunday', 'October', 4, True, False, 'summer', False), (20241028, '2024-10-28', 2024, 10, 28, 1, 'Monday', 'October', 4, False, False, 'summer', False), (20241029, '2024-10-29', 2024, 10, 29, 2, 'Tuesday', 'October', 4, False, False, 'summer', False), (20241030, '2024-10-30', 2024, 10, 30, 3, 'Wednesday', 'October', 4, False, False, 'summer', False), (20241031, '2024-10-31', 2024, 10, 31, 4, 'Thursday', 'October', 4, False, False, 'summer', False), (20241101, '2024-11-01', 2024, 11, 1, 5, 'Friday', 'November', 4, False, False, 'winter', False), (20241102, '2024-11-02', 2024, 11, 2, 6, 'Saturday', 'November', 4, True, False, 'winter', False), (20241103, '2024-11-03', 2024, 11, 3, 7, 'Sunday', 'November', 4, True, False, 'winter', False), (20241104, '2024-11-04', 2024, 11, 4, 1, 'Monday', 'November', 4, False, False, 'winter', False), (20241105, '2024-11-05', 2024, 11, 5, 2, 'Tuesday', 'November', 4, False, False, 'winter', False), (20241106, '2024-11-06', 2024, 11, 6, 3, 'Wednesday', 'November', 4, False, False, 'winter', False), (20241107, '2024-11-07', 2024, 11, 7, 4, 'Thursday', 'November', 4, False, False, 'winter', False), (20241108, '2024-11-08', 2024, 11, 8, 5, 'Friday', 'November', 4, False, False, 'winter', False), (20241109, '2024-11-09', 2024, 11, 9, 6, 'Saturday', 'November', 4, True, False, 'winter', False), (20241110, '2024-11-10', 2024, 11, 10, 7, 'Sunday', 'November', 4, True, False, 'winter', False), (20241111, '2024-11-11', 2024, 11, 11, 1, 'Monday', 'November', 4, False, True, 'winter', False), (20241112, '2024-11-12', 2024, 11, 12, 2, 'Tuesday', 'November', 4, False, False, 'winter', False), (20241113, '2024-11-13', 2024, 11, 13, 3, 'Wednesday', 'November', 4, False, False, 'winter', False), (20241114, '2024-11-14', 2024, 11, 14, 4, 'Thursday', 'November', 4, False, False, 'winter', False), (20241115, '2024-11-15', 2024, 11, 15, 5, 'Friday', 'November', 4, False, False, 'winter', False), (20241116, '2024-11-16', 2024, 11, 16, 6, 'Saturday', 'November', 4, True, False, 'winter', False), (20241117, '2024-11-17', 2024, 11, 17, 7, 'Sunday', 'November', 4, True, False, 'winter', False), (20241118, '2024-11-18', 2024, 11, 18, 1, 'Monday', 'November', 4, False, False, 'winter', False), (20241119, '2024-11-19', 2024, 11, 19, 2, 'Tuesday', 'November', 4, False, False, 'winter', False), (20241120, '2024-11-20', 2024, 11, 20, 3, 'Wednesday', 'November', 4, False, False, 'winter', False), (20241121, '2024-11-21', 2024, 11, 21, 4, 'Thursday', 'November', 4, False, False, 'winter', False), (20241122, '2024-11-22', 2024, 11, 22, 5, 'Friday', 'November', 4, False, False, 'winter', False), (20241123, '2024-11-23', 2024, 11, 23, 6, 'Saturday', 'November', 4, True, False, 'winter', False), (20241124, '2024-11-24', 2024, 11, 24, 7, 'Sunday', 'November', 4, True, False, 'winter', False), (20241125, '2024-11-25', 2024, 11, 25, 1, 'Monday', 'November', 4, False, False, 'winter', False), (20241126, '2024-11-26', 2024, 11, 26, 2, 'Tuesday', 'November', 4, False, False, 'winter', False), (20241127, '2024-11-27', 2024, 11, 27, 3, 'Wednesday', 'November', 4, False, False, 'winter', False), (20241128, '2024-11-28', 2024, 11, 28, 4, 'Thursday', 'November', 4, False, True, 'winter', False), (20241129, '2024-11-29', 2024, 11, 29, 5, 'Friday', 'November', 4, False, False, 'winter', False), (20241130, '2024-11-30', 2024, 11, 30, 6, 'Saturday', 'November', 4, True, False, 'winter', False), (20241201, '2024-12-01', 2024, 12, 1, 7, 'Sunday', 'December', 4, True, False, 'winter', False), (20241202, '2024-12-02', 2024, 12, 2, 1, 'Monday', 'December', 4, False, False, 'winter', False), (20241203, '2024-12-03', 2024, 12, 3, 2, 'Tuesday', 'December', 4, False, False, 'winter', False), (20241204, '2024-12-04', 2024, 12, 4, 3, 'Wednesday', 'December', 4, False, False, 'winter', False), (20241205, '2024-12-05', 2024, 12, 5, 4, 'Thursday', 'December', 4, False, False, 'winter', False), (20241206, '2024-12-06', 2024, 12, 6, 5, 'Friday', 'December', 4, False, False, 'winter', False), (20241207, '2024-12-07', 2024, 12, 7, 6, 'Saturday', 'December', 4, True, False, 'winter', False), (20241208, '2024-12-08', 2024, 12, 8, 7, 'Sunday', 'December', 4, True, False, 'winter', False), (20241209, '2024-12-09', 2024, 12, 9, 1, 'Monday', 'December', 4, False, False, 'winter', False), (20241210, '2024-12-10', 2024, 12, 10, 2, 'Tuesday', 'December', 4, False, False, 'winter', False), (20241211, '2024-12-11', 2024, 12, 11, 3, 'Wednesday', 'December', 4, False, False, 'winter', False), (20241212, '2024-12-12', 2024, 12, 12, 4, 'Thursday', 'December', 4, False, False, 'winter', False), (20241213, '2024-12-13', 2024, 12, 13, 5, 'Friday', 'December', 4, False, False, 'winter', False), (20241214, '2024-12-14', 2024, 12, 14, 6, 'Saturday', 'December', 4, True, False, 'winter', False), (20241215, '2024-12-15', 2024, 12, 15, 7, 'Sunday', 'December', 4, True, False, 'winter', False), (20241216, '2024-12-16', 2024, 12, 16, 1, 'Monday', 'December', 4, False, False, 'winter', False), (20241217, '2024-12-17', 2024, 12, 17, 2, 'Tuesday', 'December', 4, False, False, 'winter', False), (20241218, '2024-12-18', 2024, 12, 18, 3, 'Wednesday', 'December', 4, False, False, 'winter', False), (20241219, '2024-12-19', 2024, 12, 19, 4, 'Thursday', 'December', 4, False, False, 'winter', False), (20241220, '2024-12-20', 2024, 12, 20, 5, 'Friday', 'December', 4, False, False, 'winter', False), (20241221, '2024-12-21', 2024, 12, 21, 6, 'Saturday', 'December', 4, True, False, 'winter', False), (20241222, '2024-12-22', 2024, 12, 22, 7, 'Sunday', 'December', 4, True, False, 'winter', False), (20241223, '2024-12-23', 2024, 12, 23, 1, 'Monday', 'December', 4, False, False, 'winter', False), (20241224, '2024-12-24', 2024, 12, 24, 2, 'Tuesday', 'December', 4, False, False, 'winter', False), (20241225, '2024-12-25', 2024, 12, 25, 3, 'Wednesday', 'December', 4, False, True, 'winter', False), (20241226, '2024-12-26', 2024, 12, 26, 4, 'Thursday', 'December', 4, False, False, 'winter', False), (20241227, '2024-12-27', 2024, 12, 27, 5, 'Friday', 'December', 4, False, False, 'winter', False), (20241228, '2024-12-28', 2024, 12, 28, 6, 'Saturday', 'December', 4, True, False, 'winter', False), (20241229, '2024-12-29', 2024, 12, 29, 7, 'Sunday', 'December', 4, True, False, 'winter', False), (20241230, '2024-12-30', 2024, 12, 30, 1, 'Monday', 'December', 4, False, False, 'winter', False), (20241231, '2024-12-31', 2024, 12, 31, 2, 'Tuesday', 'December', 4, False, False, 'winter', False), (20250101, '2025-01-01', 2025, 1, 1, 3, 'Wednesday', 'January', 1, False, True, 'winter', False), (20250102, '2025-01-02', 2025, 1, 2, 4, 'Thursday', 'January', 1, False, False, 'winter', False), (20250103, '2025-01-03', 2025, 1, 3, 5, 'Friday', 'January', 1, False, False, 'winter', False), (20250104, '2025-01-04', 2025, 1, 4, 6, 'Saturday', 'January', 1, True, False, 'winter', False), (20250105, '2025-01-05', 2025, 1, 5, 7, 'Sunday', 'January', 1, True, False, 'winter', False), (20250106, '2025-01-06', 2025, 1, 6, 1, 'Monday', 'January', 1, False, False, 'winter', False), (20250107, '2025-01-07', 2025, 1, 7, 2, 'Tuesday', 'January', 1, False, False, 'winter', False), (20250108, '2025-01-08', 2025, 1, 8, 3, 'Wednesday', 'January', 1, False, False, 'winter', False), (20250109, '2025-01-09', 2025, 1, 9, 4, 'Thursday', 'January', 1, False, False, 'winter', False), (20250110, '2025-01-10', 2025, 1, 10, 5, 'Friday', 'January', 1, False, False, 'winter', False), (20250111, '2025-01-11', 2025, 1, 11, 6, 'Saturday', 'January', 1, True, False, 'winter', False), (20250112, '2025-01-12', 2025, 1, 12, 7, 'Sunday', 'January', 1, True, False, 'winter', False), (20250113, '2025-01-13', 2025, 1, 13, 1, 'Monday', 'January', 1, False, False, 'winter', False), (20250114, '2025-01-14', 2025, 1, 14, 2, 'Tuesday', 'January', 1, False, False, 'winter', False), (20250115, '2025-01-15', 2025, 1, 15, 3, 'Wednesday', 'January', 1, False, False, 'winter', False), (20250116, '2025-01-16', 2025, 1, 16, 4, 'Thursday', 'January', 1, False, False, 'winter', False), (20250117, '2025-01-17', 2025, 1, 17, 5, 'Friday', 'January', 1, False, False, 'winter', False), (20250118, '2025-01-18', 2025, 1, 18, 6, 'Saturday', 'January', 1, True, False, 'winter', False), (20250119, '2025-01-19', 2025, 1, 19, 7, 'Sunday', 'January', 1, True, False, 'winter', False), (20250120, '2025-01-20', 2025, 1, 20, 1, 'Monday', 'January', 1, False, True, 'winter', False), (20250121, '2025-01-21', 2025, 1, 21, 2, 'Tuesday', 'January', 1, False, False, 'winter', False), (20250122, '2025-01-22', 2025, 1, 22, 3, 'Wednesday', 'January', 1, False, False, 'winter', False), (20250123, '2025-01-23', 2025, 1, 23, 4, 'Thursday', 'January', 1, False, False, 'winter', False), (20250124, '2025-01-24', 2025, 1, 24, 5, 'Friday', 'January', 1, False, False, 'winter', False), (20250125, '2025-01-25', 2025, 1, 25, 6, 'Saturday', 'January', 1, True, False, 'winter', False), (20250126, '2025-01-26', 2025, 1, 26, 7, 'Sunday', 'January', 1, True, False, 'winter', False), (20250127, '2025-01-27', 2025, 1, 27, 1, 'Monday', 'January', 1, False, False, 'winter', False), (20250128, '2025-01-28', 2025, 1, 28, 2, 'Tuesday', 'January', 1, False, False, 'winter', False), (20250129, '2025-01-29', 2025, 1, 29, 3, 'Wednesday', 'January', 1, False, False, 'winter', False), (20250130, '2025-01-30', 2025, 1, 30, 4, 'Thursday', 'January', 1, False, False, 'winter', False), (20250131, '2025-01-31', 2025, 1, 31, 5, 'Friday', 'January', 1, False, False, 'winter', False), (20250201, '2025-02-01', 2025, 2, 1, 6, 'Saturday', 'February', 1, True, False, 'winter', False), (20250202, '2025-02-02', 2025, 2, 2, 7, 'Sunday', 'February', 1, True, False, 'winter', False), (20250203, '2025-02-03', 2025, 2, 3, 1, 'Monday', 'February', 1, False, False, 'winter', False), (20250204, '2025-02-04', 2025, 2, 4, 2, 'Tuesday', 'February', 1, False, False, 'winter', False), (20250205, '2025-02-05', 2025, 2, 5, 3, 'Wednesday', 'February', 1, False, False, 'winter', False), (20250206, '2025-02-06', 2025, 2, 6, 4, 'Thursday', 'February', 1, False, False, 'winter', False), (20250207, '2025-02-07', 2025, 2, 7, 5, 'Friday', 'February', 1, False, False, 'winter', False), (20250208, '2025-02-08', 2025, 2, 8, 6, 'Saturday', 'February', 1, True, False, 'winter', False), (20250209, '2025-02-09', 2025, 2, 9, 7, 'Sunday', 'February', 1, True, False, 'winter', False), (20250210, '2025-02-10', 2025, 2, 10, 1, 'Monday', 'February', 1, False, False, 'winter', False), (20250211, '2025-02-11', 2025, 2, 11, 2, 'Tuesday', 'February', 1, False, False, 'winter', False), (20250212, '2025-02-12', 2025, 2, 12, 3, 'Wednesday', 'February', 1, False, False, 'winter', False), (20250213, '2025-02-13', 2025, 2, 13, 4, 'Thursday', 'February', 1, False, False, 'winter', False), (20250214, '2025-02-14', 2025, 2, 14, 5, 'Friday', 'February', 1, False, False, 'winter', False), (20250215, '2025-02-15', 2025, 2, 15, 6, 'Saturday', 'February', 1, True, False, 'winter', False), (20250216, '2025-02-16', 2025, 2, 16, 7, 'Sunday', 'February', 1, True, False, 'winter', False), (20250217, '2025-02-17', 2025, 2, 17, 1, 'Monday', 'February', 1, False, True, 'winter', False), (20250218, '2025-02-18', 2025, 2, 18, 2, 'Tuesday', 'February', 1, False, False, 'winter', False), (20250219, '2025-02-19', 2025, 2, 19, 3, 'Wednesday', 'February', 1, False, False, 'winter', False), (20250220, '2025-02-20', 2025, 2, 20, 4, 'Thursday', 'February', 1, False, False, 'winter', False), (20250221, '2025-02-21', 2025, 2, 21, 5, 'Friday', 'February', 1, False, False, 'winter', False), (20250222, '2025-02-22', 2025, 2, 22, 6, 'Saturday', 'February', 1, True, False, 'winter', False), (20250223, '2025-02-23', 2025, 2, 23, 7, 'Sunday', 'February', 1, True, False, 'winter', False), (20250224, '2025-02-24', 2025, 2, 24, 1, 'Monday', 'February', 1, False, False, 'winter', False), (20250225, '2025-02-25', 2025, 2, 25, 2, 'Tuesday', 'February', 1, False, False, 'winter', False), (20250226, '2025-02-26', 2025, 2, 26, 3, 'Wednesday', 'February', 1, False, False, 'winter', False), (20250227, '2025-02-27', 2025, 2, 27, 4, 'Thursday', 'February', 1, False, False, 'winter', False), (20250228, '2025-02-28', 2025, 2, 28, 5, 'Friday', 'February', 1, False, False, 'winter', False), (20250301, '2025-03-01', 2025, 3, 1, 6, 'Saturday', 'March', 1, True, False, 'winter', False), (20250302, '2025-03-02', 2025, 3, 2, 7, 'Sunday', 'March', 1, True, False, 'winter', False), (20250303, '2025-03-03', 2025, 3, 3, 1, 'Monday', 'March', 1, False, False, 'winter', False), (20250304, '2025-03-04', 2025, 3, 4, 2, 'Tuesday', 'March', 1, False, False, 'winter', False), (20250305, '2025-03-05', 2025, 3, 5, 3, 'Wednesday', 'March', 1, False, False, 'winter', False), (20250306, '2025-03-06', 2025, 3, 6, 4, 'Thursday', 'March', 1, False, False, 'winter', False), (20250307, '2025-03-07', 2025, 3, 7, 5, 'Friday', 'March', 1, False, False, 'winter', False), (20250308, '2025-03-08', 2025, 3, 8, 6, 'Saturday', 'March', 1, True, False, 'winter', False), (20250309, '2025-03-09', 2025, 3, 9, 7, 'Sunday', 'March', 1, True, False, 'winter', False), (20250310, '2025-03-10', 2025, 3, 10, 1, 'Monday', 'March', 1, False, False, 'winter', False), (20250311, '2025-03-11', 2025, 3, 11, 2, 'Tuesday', 'March', 1, False, False, 'winter', False), (20250312, '2025-03-12', 2025, 3, 12, 3, 'Wednesday', 'March', 1, False, False, 'winter', False), (20250313, '2025-03-13', 2025, 3, 13, 4, 'Thursday', 'March', 1, False, False, 'winter', False), (20250314, '2025-03-14', 2025, 3, 14, 5, 'Friday', 'March', 1, False, False, 'winter', False), (20250315, '2025-03-15', 2025, 3, 15, 6, 'Saturday', 'March', 1, True, False, 'winter', False), (20250316, '2025-03-16', 2025, 3, 16, 7, 'Sunday', 'March', 1, True, False, 'winter', False), (20250317, '2025-03-17', 2025, 3, 17, 1, 'Monday', 'March', 1, False, False, 'winter', False), (20250318, '2025-03-18', 2025, 3, 18, 2, 'Tuesday', 'March', 1, False, False, 'winter', False), (20250319, '2025-03-19', 2025, 3, 19, 3, 'Wednesday', 'March', 1, False, False, 'winter', False), (20250320, '2025-03-20', 2025, 3, 20, 4, 'Thursday', 'March', 1, False, False, 'winter', False), (20250321, '2025-03-21', 2025, 3, 21, 5, 'Friday', 'March', 1, False, False, 'winter', False), (20250322, '2025-03-22', 2025, 3, 22, 6, 'Saturday', 'March', 1, True, False, 'winter', False), (20250323, '2025-03-23', 2025, 3, 23, 7, 'Sunday', 'March', 1, True, False, 'winter', False), (20250324, '2025-03-24', 2025, 3, 24, 1, 'Monday', 'March', 1, False, False, 'winter', False), (20250325, '2025-03-25', 2025, 3, 25, 2, 'Tuesday', 'March', 1, False, False, 'winter', False), (20250326, '2025-03-26', 2025, 3, 26, 3, 'Wednesday', 'March', 1, False, False, 'winter', False), (20250327, '2025-03-27', 2025, 3, 27, 4, 'Thursday', 'March', 1, False, False, 'winter', False), (20250328, '2025-03-28', 2025, 3, 28, 5, 'Friday', 'March', 1, False, False, 'winter', False), (20250329, '2025-03-29', 2025, 3, 29, 6, 'Saturday', 'March', 1, True, False, 'winter', False), (20250330, '2025-03-30', 2025, 3, 30, 7, 'Sunday', 'March', 1, True, False, 'winter', False), (20250331, '2025-03-31', 2025, 3, 31, 1, 'Monday', 'March', 1, False, False, 'winter', False), (20250401, '2025-04-01', 2025, 4, 1, 2, 'Tuesday', 'April', 2, False, False, 'summer', False), (20250402, '2025-04-02', 2025, 4, 2, 3, 'Wednesday', 'April', 2, False, False, 'summer', False), (20250403, '2025-04-03', 2025, 4, 3, 4, 'Thursday', 'April', 2, False, False, 'summer', False), (20250404, '2025-04-04', 2025, 4, 4, 5, 'Friday', 'April', 2, False, False, 'summer', False), (20250405, '2025-04-05', 2025, 4, 5, 6, 'Saturday', 'April', 2, True, False, 'summer', False), (20250406, '2025-04-06', 2025, 4, 6, 7, 'Sunday', 'April', 2, True, False, 'summer', False), (20250407, '2025-04-07', 2025, 4, 7, 1, 'Monday', 'April', 2, False, False, 'summer', False), (20250408, '2025-04-08', 2025, 4, 8, 2, 'Tuesday', 'April', 2, False, False, 'summer', False), (20250409, '2025-04-09', 2025, 4, 9, 3, 'Wednesday', 'April', 2, False, False, 'summer', False), (20250410, '2025-04-10', 2025, 4, 10, 4, 'Thursday', 'April', 2, False, False, 'summer', False), (20250411, '2025-04-11', 2025, 4, 11, 5, 'Friday', 'April', 2, False, False, 'summer', False), (20250412, '2025-04-12', 2025, 4, 12, 6, 'Saturday', 'April', 2, True, False, 'summer', False), (20250413, '2025-04-13', 2025, 4, 13, 7, 'Sunday', 'April', 2, True, False, 'summer', False), (20250414, '2025-04-14', 2025, 4, 14, 1, 'Monday', 'April', 2, False, False, 'summer', False), (20250415, '2025-04-15', 2025, 4, 15, 2, 'Tuesday', 'April', 2, False, False, 'summer', False), (20250416, '2025-04-16', 2025, 4, 16, 3, 'Wednesday', 'April', 2, False, False, 'summer', False), (20250417, '2025-04-17', 2025, 4, 17, 4, 'Thursday', 'April', 2, False, False, 'summer', False), (20250418, '2025-04-18', 2025, 4, 18, 5, 'Friday', 'April', 2, False, False, 'summer', False), (20250419, '2025-04-19', 2025, 4, 19, 6, 'Saturday', 'April', 2, True, False, 'summer', False), (20250420, '2025-04-20', 2025, 4, 20, 7, 'Sunday', 'April', 2, True, False, 'summer', False), (20250421, '2025-04-21', 2025, 4, 21, 1, 'Monday', 'April', 2, False, False, 'summer', False), (20250422, '2025-04-22', 2025, 4, 22, 2, 'Tuesday', 'April', 2, False, False, 'summer', False), (20250423, '2025-04-23', 2025, 4, 23, 3, 'Wednesday', 'April', 2, False, False, 'summer', False), (20250424, '2025-04-24', 2025, 4, 24, 4, 'Thursday', 'April', 2, False, False, 'summer', False), (20250425, '2025-04-25', 2025, 4, 25, 5, 'Friday', 'April', 2, False, False, 'summer', False), (20250426, '2025-04-26', 2025, 4, 26, 6, 'Saturday', 'April', 2, True, False, 'summer', False), (20250427, '2025-04-27', 2025, 4, 27, 7, 'Sunday', 'April', 2, True, False, 'summer', False), (20250428, '2025-04-28', 2025, 4, 28, 1, 'Monday', 'April', 2, False, False, 'summer', False), (20250429, '2025-04-29', 2025, 4, 29, 2, 'Tuesday', 'April', 2, False, False, 'summer', False), (20250430, '2025-04-30', 2025, 4, 30, 3, 'Wednesday', 'April', 2, False, False, 'summer', False), (20250501, '2025-05-01', 2025, 5, 1, 4, 'Thursday', 'May', 2, False, False, 'summer', False), (20250502, '2025-05-02', 2025, 5, 2, 5, 'Friday', 'May', 2, False, False, 'summer', False), (20250503, '2025-05-03', 2025, 5, 3, 6, 'Saturday', 'May', 2, True, False, 'summer', False), (20250504, '2025-05-04', 2025, 5, 4, 7, 'Sunday', 'May', 2, True, False, 'summer', False), (20250505, '2025-05-05', 2025, 5, 5, 1, 'Monday', 'May', 2, False, False, 'summer', False), (20250506, '2025-05-06', 2025, 5, 6, 2, 'Tuesday', 'May', 2, False, False, 'summer', False), (20250507, '2025-05-07', 2025, 5, 7, 3, 'Wednesday', 'May', 2, False, False, 'summer', False), (20250508, '2025-05-08', 2025, 5, 8, 4, 'Thursday', 'May', 2, False, False, 'summer', False), (20250509, '2025-05-09', 2025, 5, 9, 5, 'Friday', 'May', 2, False, False, 'summer', False), (20250510, '2025-05-10', 2025, 5, 10, 6, 'Saturday', 'May', 2, True, False, 'summer', False), (20250511, '2025-05-11', 2025, 5, 11, 7, 'Sunday', 'May', 2, True, False, 'summer', False), (20250512, '2025-05-12', 2025, 5, 12, 1, 'Monday', 'May', 2, False, False, 'summer', False), (20250513, '2025-05-13', 2025, 5, 13, 2, 'Tuesday', 'May', 2, False, False, 'summer', False), (20250514, '2025-05-14', 2025, 5, 14, 3, 'Wednesday', 'May', 2, False, False, 'summer', False), (20250515, '2025-05-15', 2025, 5, 15, 4, 'Thursday', 'May', 2, False, False, 'summer', False), (20250516, '2025-05-16', 2025, 5, 16, 5, 'Friday', 'May', 2, False, False, 'summer', False), (20250517, '2025-05-17', 2025, 5, 17, 6, 'Saturday', 'May', 2, True, False, 'summer', False), (20250518, '2025-05-18', 2025, 5, 18, 7, 'Sunday', 'May', 2, True, False, 'summer', False), (20250519, '2025-05-19', 2025, 5, 19, 1, 'Monday', 'May', 2, False, False, 'summer', False), (20250520, '2025-05-20', 2025, 5, 20, 2, 'Tuesday', 'May', 2, False, False, 'summer', False), (20250521, '2025-05-21', 2025, 5, 21, 3, 'Wednesday', 'May', 2, False, False, 'summer', False), (20250522, '2025-05-22', 2025, 5, 22, 4, 'Thursday', 'May', 2, False, False, 'summer', False), (20250523, '2025-05-23', 2025, 5, 23, 5, 'Friday', 'May', 2, False, False, 'summer', False), (20250524, '2025-05-24', 2025, 5, 24, 6, 'Saturday', 'May', 2, True, False, 'summer', False), (20250525, '2025-05-25', 2025, 5, 25, 7, 'Sunday', 'May', 2, True, False, 'summer', False), (20250526, '2025-05-26', 2025, 5, 26, 1, 'Monday', 'May', 2, False, True, 'summer', False), (20250527, '2025-05-27', 2025, 5, 27, 2, 'Tuesday', 'May', 2, False, False, 'summer', False), (20250528, '2025-05-28', 2025, 5, 28, 3, 'Wednesday', 'May', 2, False, False, 'summer', False), (20250529, '2025-05-29', 2025, 5, 29, 4, 'Thursday', 'May', 2, False, False, 'summer', False), (20250530, '2025-05-30', 2025, 5, 30, 5, 'Friday', 'May', 2, False, False, 'summer', False), (20250531, '2025-05-31', 2025, 5, 31, 6, 'Saturday', 'May', 2, True, False, 'summer', False), (20250601, '2025-06-01', 2025, 6, 1, 7, 'Sunday', 'June', 2, True, False, 'summer', False), (20250602, '2025-06-02', 2025, 6, 2, 1, 'Monday', 'June', 2, False, False, 'summer', False), (20250603, '2025-06-03', 2025, 6, 3, 2, 'Tuesday', 'June', 2, False, False, 'summer', False), (20250604, '2025-06-04', 2025, 6, 4, 3, 'Wednesday', 'June', 2, False, False, 'summer', False), (20250605, '2025-06-05', 2025, 6, 5, 4, 'Thursday', 'June', 2, False, False, 'summer', False), (20250606, '2025-06-06', 2025, 6, 6, 5, 'Friday', 'June', 2, False, False, 'summer', False), (20250607, '2025-06-07', 2025, 6, 7, 6, 'Saturday', 'June', 2, True, False, 'summer', False), (20250608, '2025-06-08', 2025, 6, 8, 7, 'Sunday', 'June', 2, True, False, 'summer', False), (20250609, '2025-06-09', 2025, 6, 9, 1, 'Monday', 'June', 2, False, False, 'summer', False), (20250610, '2025-06-10', 2025, 6, 10, 2, 'Tuesday', 'June', 2, False, False, 'summer', False), (20250611, '2025-06-11', 2025, 6, 11, 3, 'Wednesday', 'June', 2, False, False, 'summer', False), (20250612, '2025-06-12', 2025, 6, 12, 4, 'Thursday', 'June', 2, False, False, 'summer', False), (20250613, '2025-06-13', 2025, 6, 13, 5, 'Friday', 'June', 2, False, False, 'summer', False), (20250614, '2025-06-14', 2025, 6, 14, 6, 'Saturday', 'June', 2, True, False, 'summer', False), (20250615, '2025-06-15', 2025, 6, 15, 7, 'Sunday', 'June', 2, True, False, 'summer', False), (20250616, '2025-06-16', 2025, 6, 16, 1, 'Monday', 'June', 2, False, False, 'summer', False), (20250617, '2025-06-17', 2025, 6, 17, 2, 'Tuesday', 'June', 2, False, False, 'summer', False), (20250618, '2025-06-18', 2025, 6, 18, 3, 'Wednesday', 'June', 2, False, False, 'summer', False), (20250619, '2025-06-19', 2025, 6, 19, 4, 'Thursday', 'June', 2, False, True, 'summer', False), (20250620, '2025-06-20', 2025, 6, 20, 5, 'Friday', 'June', 2, False, False, 'summer', False), (20250621, '2025-06-21', 2025, 6, 21, 6, 'Saturday', 'June', 2, True, False, 'summer', False), (20250622, '2025-06-22', 2025, 6, 22, 7, 'Sunday', 'June', 2, True, False, 'summer', False), (20250623, '2025-06-23', 2025, 6, 23, 1, 'Monday', 'June', 2, False, False, 'summer', False), (20250624, '2025-06-24', 2025, 6, 24, 2, 'Tuesday', 'June', 2, False, False, 'summer', False), (20250625, '2025-06-25', 2025, 6, 25, 3, 'Wednesday', 'June', 2, False, False, 'summer', False), (20250626, '2025-06-26', 2025, 6, 26, 4, 'Thursday', 'June', 2, False, False, 'summer', False), (20250627, '2025-06-27', 2025, 6, 27, 5, 'Friday', 'June', 2, False, False, 'summer', False), (20250628, '2025-06-28', 2025, 6, 28, 6, 'Saturday', 'June', 2, True, False, 'summer', False), (20250629, '2025-06-29', 2025, 6, 29, 7, 'Sunday', 'June', 2, True, False, 'summer', False), (20250630, '2025-06-30', 2025, 6, 30, 1, 'Monday', 'June', 2, False, False, 'summer', False), (20250701, '2025-07-01', 2025, 7, 1, 2, 'Tuesday', 'July', 3, False, False, 'summer', False), (20250702, '2025-07-02', 2025, 7, 2, 3, 'Wednesday', 'July', 3, False, False, 'summer', False), (20250703, '2025-07-03', 2025, 7, 3, 4, 'Thursday', 'July', 3, False, False, 'summer', False), (20250704, '2025-07-04', 2025, 7, 4, 5, 'Friday', 'July', 3, False, True, 'summer', False), (20250705, '2025-07-05', 2025, 7, 5, 6, 'Saturday', 'July', 3, True, False, 'summer', False), (20250706, '2025-07-06', 2025, 7, 6, 7, 'Sunday', 'July', 3, True, False, 'summer', False), (20250707, '2025-07-07', 2025, 7, 7, 1, 'Monday', 'July', 3, False, False, 'summer', False), (20250708, '2025-07-08', 2025, 7, 8, 2, 'Tuesday', 'July', 3, False, False, 'summer', False), (20250709, '2025-07-09', 2025, 7, 9, 3, 'Wednesday', 'July', 3, False, False, 'summer', False), (20250710, '2025-07-10', 2025, 7, 10, 4, 'Thursday', 'July', 3, False, False, 'summer', False), (20250711, '2025-07-11', 2025, 7, 11, 5, 'Friday', 'July', 3, False, False, 'summer', False), (20250712, '2025-07-12', 2025, 7, 12, 6, 'Saturday', 'July', 3, True, False, 'summer', False), (20250713, '2025-07-13', 2025, 7, 13, 7, 'Sunday', 'July', 3, True, False, 'summer', False), (20250714, '2025-07-14', 2025, 7, 14, 1, 'Monday', 'July', 3, False, False, 'summer', False), (20250715, '2025-07-15', 2025, 7, 15, 2, 'Tuesday', 'July', 3, False, False, 'summer', False), (20250716, '2025-07-16', 2025, 7, 16, 3, 'Wednesday', 'July', 3, False, False, 'summer', False), (20250717, '2025-07-17', 2025, 7, 17, 4, 'Thursday', 'July', 3, False, False, 'summer', False), (20250718, '2025-07-18', 2025, 7, 18, 5, 'Friday', 'July', 3, False, False, 'summer', False), (20250719, '2025-07-19', 2025, 7, 19, 6, 'Saturday', 'July', 3, True, False, 'summer', False), (20250720, '2025-07-20', 2025, 7, 20, 7, 'Sunday', 'July', 3, True, False, 'summer', False), (20250721, '2025-07-21', 2025, 7, 21, 1, 'Monday', 'July', 3, False, False, 'summer', False), (20250722, '2025-07-22', 2025, 7, 22, 2, 'Tuesday', 'July', 3, False, False, 'summer', False), (20250723, '2025-07-23', 2025, 7, 23, 3, 'Wednesday', 'July', 3, False, False, 'summer', False), (20250724, '2025-07-24', 2025, 7, 24, 4, 'Thursday', 'July', 3, False, False, 'summer', False), (20250725, '2025-07-25', 2025, 7, 25, 5, 'Friday', 'July', 3, False, False, 'summer', False), (20250726, '2025-07-26', 2025, 7, 26, 6, 'Saturday', 'July', 3, True, False, 'summer', False), (20250727, '2025-07-27', 2025, 7, 27, 7, 'Sunday', 'July', 3, True, False, 'summer', False), (20250728, '2025-07-28', 2025, 7, 28, 1, 'Monday', 'July', 3, False, False, 'summer', False), (20250729, '2025-07-29', 2025, 7, 29, 2, 'Tuesday', 'July', 3, False, False, 'summer', False), (20250730, '2025-07-30', 2025, 7, 30, 3, 'Wednesday', 'July', 3, False, False, 'summer', False), (20250731, '2025-07-31', 2025, 7, 31, 4, 'Thursday', 'July', 3, False, False, 'summer', False), (20250801, '2025-08-01', 2025, 8, 1, 5, 'Friday', 'August', 3, False, False, 'summer', False), (20250802, '2025-08-02', 2025, 8, 2, 6, 'Saturday', 'August', 3, True, False, 'summer', False), (20250803, '2025-08-03', 2025, 8, 3, 7, 'Sunday', 'August', 3, True, False, 'summer', False), (20250804, '2025-08-04', 2025, 8, 4, 1, 'Monday', 'August', 3, False, False, 'summer', False), (20250805, '2025-08-05', 2025, 8, 5, 2, 'Tuesday', 'August', 3, False, False, 'summer', False), (20250806, '2025-08-06', 2025, 8, 6, 3, 'Wednesday', 'August', 3, False, False, 'summer', False), (20250807, '2025-08-07', 2025, 8, 7, 4, 'Thursday', 'August', 3, False, False, 'summer', False), (20250808, '2025-08-08', 2025, 8, 8, 5, 'Friday', 'August', 3, False, False, 'summer', False), (20250809, '2025-08-09', 2025, 8, 9, 6, 'Saturday', 'August', 3, True, False, 'summer', False), (20250810, '2025-08-10', 2025, 8, 10, 7, 'Sunday', 'August', 3, True, False, 'summer', False), (20250811, '2025-08-11', 2025, 8, 11, 1, 'Monday', 'August', 3, False, False, 'summer', False), (20250812, '2025-08-12', 2025, 8, 12, 2, 'Tuesday', 'August', 3, False, False, 'summer', False), (20250813, '2025-08-13', 2025, 8, 13, 3, 'Wednesday', 'August', 3, False, False, 'summer', False), (20250814, '2025-08-14', 2025, 8, 14, 4, 'Thursday', 'August', 3, False, False, 'summer', False), (20250815, '2025-08-15', 2025, 8, 15, 5, 'Friday', 'August', 3, False, False, 'summer', False), (20250816, '2025-08-16', 2025, 8, 16, 6, 'Saturday', 'August', 3, True, False, 'summer', False), (20250817, '2025-08-17', 2025, 8, 17, 7, 'Sunday', 'August', 3, True, False, 'summer', False), (20250818, '2025-08-18', 2025, 8, 18, 1, 'Monday', 'August', 3, False, False, 'summer', False), (20250819, '2025-08-19', 2025, 8, 19, 2, 'Tuesday', 'August', 3, False, False, 'summer', False), (20250820, '2025-08-20', 2025, 8, 20, 3, 'Wednesday', 'August', 3, False, False, 'summer', False), (20250821, '2025-08-21', 2025, 8, 21, 4, 'Thursday', 'August', 3, False, False, 'summer', False), (20250822, '2025-08-22', 2025, 8, 22, 5, 'Friday', 'August', 3, False, False, 'summer', False), (20250823, '2025-08-23', 2025, 8, 23, 6, 'Saturday', 'August', 3, True, False, 'summer', False), (20250824, '2025-08-24', 2025, 8, 24, 7, 'Sunday', 'August', 3, True, False, 'summer', False), (20250825, '2025-08-25', 2025, 8, 25, 1, 'Monday', 'August', 3, False, False, 'summer', False), (20250826, '2025-08-26', 2025, 8, 26, 2, 'Tuesday', 'August', 3, False, False, 'summer', False), (20250827, '2025-08-27', 2025, 8, 27, 3, 'Wednesday', 'August', 3, False, False, 'summer', False), (20250828, '2025-08-28', 2025, 8, 28, 4, 'Thursday', 'August', 3, False, False, 'summer', False), (20250829, '2025-08-29', 2025, 8, 29, 5, 'Friday', 'August', 3, False, False, 'summer', False), (20250830, '2025-08-30', 2025, 8, 30, 6, 'Saturday', 'August', 3, True, False, 'summer', False), (20250831, '2025-08-31', 2025, 8, 31, 7, 'Sunday', 'August', 3, True, False, 'summer', False), (20250901, '2025-09-01', 2025, 9, 1, 1, 'Monday', 'September', 3, False, True, 'summer', False), (20250902, '2025-09-02', 2025, 9, 2, 2, 'Tuesday', 'September', 3, False, False, 'summer', False), (20250903, '2025-09-03', 2025, 9, 3, 3, 'Wednesday', 'September', 3, False, False, 'summer', False), (20250904, '2025-09-04', 2025, 9, 4, 4, 'Thursday', 'September', 3, False, False, 'summer', False), (20250905, '2025-09-05', 2025, 9, 5, 5, 'Friday', 'September', 3, False, False, 'summer', False), (20250906, '2025-09-06', 2025, 9, 6, 6, 'Saturday', 'September', 3, True, False, 'summer', False), (20250907, '2025-09-07', 2025, 9, 7, 7, 'Sunday', 'September', 3, True, False, 'summer', False), (20250908, '2025-09-08', 2025, 9, 8, 1, 'Monday', 'September', 3, False, False, 'summer', False), (20250909, '2025-09-09', 2025, 9, 9, 2, 'Tuesday', 'September', 3, False, False, 'summer', False), (20250910, '2025-09-10', 2025, 9, 10, 3, 'Wednesday', 'September', 3, False, False, 'summer', False), (20250911, '2025-09-11', 2025, 9, 11, 4, 'Thursday', 'September', 3, False, False, 'summer', False), (20250912, '2025-09-12', 2025, 9, 12, 5, 'Friday', 'September', 3, False, False, 'summer', False), (20250913, '2025-09-13', 2025, 9, 13, 6, 'Saturday', 'September', 3, True, False, 'summer', False), (20250914, '2025-09-14', 2025, 9, 14, 7, 'Sunday', 'September', 3, True, False, 'summer', False), (20250915, '2025-09-15', 2025, 9, 15, 1, 'Monday', 'September', 3, False, False, 'summer', False), (20250916, '2025-09-16', 2025, 9, 16, 2, 'Tuesday', 'September', 3, False, False, 'summer', False), (20250917, '2025-09-17', 2025, 9, 17, 3, 'Wednesday', 'September', 3, False, False, 'summer', False), (20250918, '2025-09-18', 2025, 9, 18, 4, 'Thursday', 'September', 3, False, False, 'summer', False), (20250919, '2025-09-19', 2025, 9, 19, 5, 'Friday', 'September', 3, False, False, 'summer', False), (20250920, '2025-09-20', 2025, 9, 20, 6, 'Saturday', 'September', 3, True, False, 'summer', False), (20250921, '2025-09-21', 2025, 9, 21, 7, 'Sunday', 'September', 3, True, False, 'summer', False), (20250922, '2025-09-22', 2025, 9, 22, 1, 'Monday', 'September', 3, False, False, 'summer', False), (20250923, '2025-09-23', 2025, 9, 23, 2, 'Tuesday', 'September', 3, False, False, 'summer', False), (20250924, '2025-09-24', 2025, 9, 24, 3, 'Wednesday', 'September', 3, False, False, 'summer', False), (20250925, '2025-09-25', 2025, 9, 25, 4, 'Thursday', 'September', 3, False, False, 'summer', False), (20250926, '2025-09-26', 2025, 9, 26, 5, 'Friday', 'September', 3, False, False, 'summer', False), (20250927, '2025-09-27', 2025, 9, 27, 6, 'Saturday', 'September', 3, True, False, 'summer', False), (20250928, '2025-09-28', 2025, 9, 28, 7, 'Sunday', 'September', 3, True, False, 'summer', False), (20250929, '2025-09-29', 2025, 9, 29, 1, 'Monday', 'September', 3, False, False, 'summer', False), (20250930, '2025-09-30', 2025, 9, 30, 2, 'Tuesday', 'September', 3, False, False, 'summer', False), (20251001, '2025-10-01', 2025, 10, 1, 3, 'Wednesday', 'October', 4, False, False, 'summer', False), (20251002, '2025-10-02', 2025, 10, 2, 4, 'Thursday', 'October', 4, False, False, 'summer', False), (20251003, '2025-10-03', 2025, 10, 3, 5, 'Friday', 'October', 4, False, False, 'summer', False), (20251004, '2025-10-04', 2025, 10, 4, 6, 'Saturday', 'October', 4, True, False, 'summer', False), (20251005, '2025-10-05', 2025, 10, 5, 7, 'Sunday', 'October', 4, True, False, 'summer', False), (20251006, '2025-10-06', 2025, 10, 6, 1, 'Monday', 'October', 4, False, False, 'summer', False), (20251007, '2025-10-07', 2025, 10, 7, 2, 'Tuesday', 'October', 4, False, False, 'summer', False), (20251008, '2025-10-08', 2025, 10, 8, 3, 'Wednesday', 'October', 4, False, False, 'summer', False), (20251009, '2025-10-09', 2025, 10, 9, 4, 'Thursday', 'October', 4, False, False, 'summer', False), (20251010, '2025-10-10', 2025, 10, 10, 5, 'Friday', 'October', 4, False, False, 'summer', False), (20251011, '2025-10-11', 2025, 10, 11, 6, 'Saturday', 'October', 4, True, False, 'summer', False), (20251012, '2025-10-12', 2025, 10, 12, 7, 'Sunday', 'October', 4, True, False, 'summer', False), (20251013, '2025-10-13', 2025, 10, 13, 1, 'Monday', 'October', 4, False, True, 'summer', False), (20251014, '2025-10-14', 2025, 10, 14, 2, 'Tuesday', 'October', 4, False, False, 'summer', False), (20251015, '2025-10-15', 2025, 10, 15, 3, 'Wednesday', 'October', 4, False, False, 'summer', False), (20251016, '2025-10-16', 2025, 10, 16, 4, 'Thursday', 'October', 4, False, False, 'summer', False), (20251017, '2025-10-17', 2025, 10, 17, 5, 'Friday', 'October', 4, False, False, 'summer', False), (20251018, '2025-10-18', 2025, 10, 18, 6, 'Saturday', 'October', 4, True, False, 'summer', False), (20251019, '2025-10-19', 2025, 10, 19, 7, 'Sunday', 'October', 4, True, False, 'summer', False), (20251020, '2025-10-20', 2025, 10, 20, 1, 'Monday', 'October', 4, False, False, 'summer', False), (20251021, '2025-10-21', 2025, 10, 21, 2, 'Tuesday', 'October', 4, False, False, 'summer', False), (20251022, '2025-10-22', 2025, 10, 22, 3, 'Wednesday', 'October', 4, False, False, 'summer', False), (20251023, '2025-10-23', 2025, 10, 23, 4, 'Thursday', 'October', 4, False, False, 'summer', False), (20251024, '2025-10-24', 2025, 10, 24, 5, 'Friday', 'October', 4, False, False, 'summer', False), (20251025, '2025-10-25', 2025, 10, 25, 6, 'Saturday', 'October', 4, True, False, 'summer', False), (20251026, '2025-10-26', 2025, 10, 26, 7, 'Sunday', 'October', 4, True, False, 'summer', False), (20251027, '2025-10-27', 2025, 10, 27, 1, 'Monday', 'October', 4, False, False, 'summer', False), (20251028, '2025-10-28', 2025, 10, 28, 2, 'Tuesday', 'October', 4, False, False, 'summer', False), (20251029, '2025-10-29', 2025, 10, 29, 3, 'Wednesday', 'October', 4, False, False, 'summer', False), (20251030, '2025-10-30', 2025, 10, 30, 4, 'Thursday', 'October', 4, False, False, 'summer', False), (20251031, '2025-10-31', 2025, 10, 31, 5, 'Friday', 'October', 4, False, False, 'summer', False), (20251101, '2025-11-01', 2025, 11, 1, 6, 'Saturday', 'November', 4, True, False, 'winter', False), (20251102, '2025-11-02', 2025, 11, 2, 7, 'Sunday', 'November', 4, True, False, 'winter', False), (20251103, '2025-11-03', 2025, 11, 3, 1, 'Monday', 'November', 4, False, False, 'winter', False), (20251104, '2025-11-04', 2025, 11, 4, 2, 'Tuesday', 'November', 4, False, False, 'winter', False), (20251105, '2025-11-05', 2025, 11, 5, 3, 'Wednesday', 'November', 4, False, False, 'winter', False), (20251106, '2025-11-06', 2025, 11, 6, 4, 'Thursday', 'November', 4, False, False, 'winter', False), (20251107, '2025-11-07', 2025, 11, 7, 5, 'Friday', 'November', 4, False, False, 'winter', False), (20251108, '2025-11-08', 2025, 11, 8, 6, 'Saturday', 'November', 4, True, False, 'winter', False), (20251109, '2025-11-09', 2025, 11, 9, 7, 'Sunday', 'November', 4, True, False, 'winter', False), (20251110, '2025-11-10', 2025, 11, 10, 1, 'Monday', 'November', 4, False, False, 'winter', False), (20251111, '2025-11-11', 2025, 11, 11, 2, 'Tuesday', 'November', 4, False, True, 'winter', False), (20251112, '2025-11-12', 2025, 11, 12, 3, 'Wednesday', 'November', 4, False, False, 'winter', False), (20251113, '2025-11-13', 2025, 11, 13, 4, 'Thursday', 'November', 4, False, False, 'winter', False), (20251114, '2025-11-14', 2025, 11, 14, 5, 'Friday', 'November', 4, False, False, 'winter', False), (20251115, '2025-11-15', 2025, 11, 15, 6, 'Saturday', 'November', 4, True, False, 'winter', False), (20251116, '2025-11-16', 2025, 11, 16, 7, 'Sunday', 'November', 4, True, False, 'winter', False), (20251117, '2025-11-17', 2025, 11, 17, 1, 'Monday', 'November', 4, False, False, 'winter', False), (20251118, '2025-11-18', 2025, 11, 18, 2, 'Tuesday', 'November', 4, False, False, 'winter', False), (20251119, '2025-11-19', 2025, 11, 19, 3, 'Wednesday', 'November', 4, False, False, 'winter', False), (20251120, '2025-11-20', 2025, 11, 20, 4, 'Thursday', 'November', 4, False, False, 'winter', False), (20251121, '2025-11-21', 2025, 11, 21, 5, 'Friday', 'November', 4, False, False, 'winter', False), (20251122, '2025-11-22', 2025, 11, 22, 6, 'Saturday', 'November', 4, True, False, 'winter', False), (20251123, '2025-11-23', 2025, 11, 23, 7, 'Sunday', 'November', 4, True, False, 'winter', False), (20251124, '2025-11-24', 2025, 11, 24, 1, 'Monday', 'November', 4, False, False, 'winter', False), (20251125, '2025-11-25', 2025, 11, 25, 2, 'Tuesday', 'November', 4, False, False, 'winter', False), (20251126, '2025-11-26', 2025, 11, 26, 3, 'Wednesday', 'November', 4, False, False, 'winter', False), (20251127, '2025-11-27', 2025, 11, 27, 4, 'Thursday', 'November', 4, False, True, 'winter', False), (20251128, '2025-11-28', 2025, 11, 28, 5, 'Friday', 'November', 4, False, False, 'winter', False), (20251129, '2025-11-29', 2025, 11, 29, 6, 'Saturday', 'November', 4, True, False, 'winter', False), (20251130, '2025-11-30', 2025, 11, 30, 7, 'Sunday', 'November', 4, True, False, 'winter', False), (20251201, '2025-12-01', 2025, 12, 1, 1, 'Monday', 'December', 4, False, False, 'winter', False), (20251202, '2025-12-02', 2025, 12, 2, 2, 'Tuesday', 'December', 4, False, False, 'winter', False), (20251203, '2025-12-03', 2025, 12, 3, 3, 'Wednesday', 'December', 4, False, False, 'winter', False), (20251204, '2025-12-04', 2025, 12, 4, 4, 'Thursday', 'December', 4, False, False, 'winter', False), (20251205, '2025-12-05', 2025, 12, 5, 5, 'Friday', 'December', 4, False, False, 'winter', False), (20251206, '2025-12-06', 2025, 12, 6, 6, 'Saturday', 'December', 4, True, False, 'winter', False), (20251207, '2025-12-07', 2025, 12, 7, 7, 'Sunday', 'December', 4, True, False, 'winter', False), (20251208, '2025-12-08', 2025, 12, 8, 1, 'Monday', 'December', 4, False, False, 'winter', False), (20251209, '2025-12-09', 2025, 12, 9, 2, 'Tuesday', 'December', 4, False, False, 'winter', False), (20251210, '2025-12-10', 2025, 12, 10, 3, 'Wednesday', 'December', 4, False, False, 'winter', False), (20251211, '2025-12-11', 2025, 12, 11, 4, 'Thursday', 'December', 4, False, False, 'winter', False), (20251212, '2025-12-12', 2025, 12, 12, 5, 'Friday', 'December', 4, False, False, 'winter', False), (20251213, '2025-12-13', 2025, 12, 13, 6, 'Saturday', 'December', 4, True, False, 'winter', False), (20251214, '2025-12-14', 2025, 12, 14, 7, 'Sunday', 'December', 4, True, False, 'winter', False), (20251215, '2025-12-15', 2025, 12, 15, 1, 'Monday', 'December', 4, False, False, 'winter', False), (20251216, '2025-12-16', 2025, 12, 16, 2, 'Tuesday', 'December', 4, False, False, 'winter', False), (20251217, '2025-12-17', 2025, 12, 17, 3, 'Wednesday', 'December', 4, False, False, 'winter', False), (20251218, '2025-12-18', 2025, 12, 18, 4, 'Thursday', 'December', 4, False, False, 'winter', False), (20251219, '2025-12-19', 2025, 12, 19, 5, 'Friday', 'December', 4, False, False, 'winter', False), (20251220, '2025-12-20', 2025, 12, 20, 6, 'Saturday', 'December', 4, True, False, 'winter', False), (20251221, '2025-12-21', 2025, 12, 21, 7, 'Sunday', 'December', 4, True, False, 'winter', False), (20251222, '2025-12-22', 2025, 12, 22, 1, 'Monday', 'December', 4, False, False, 'winter', False), (20251223, '2025-12-23', 2025, 12, 23, 2, 'Tuesday', 'December', 4, False, False, 'winter', False), (20251224, '2025-12-24', 2025, 12, 24, 3, 'Wednesday', 'December', 4, False, False, 'winter', False), (20251225, '2025-12-25', 2025, 12, 25, 4, 'Thursday', 'December', 4, False, True, 'winter', False), (20251226, '2025-12-26', 2025, 12, 26, 5, 'Friday', 'December', 4, False, False, 'winter', False), (20251227, '2025-12-27', 2025, 12, 27, 6, 'Saturday', 'December', 4, True, False, 'winter', False), (20251228, '2025-12-28', 2025, 12, 28, 7, 'Sunday', 'December', 4, True, False, 'winter', False), (20251229, '2025-12-29', 2025, 12, 29, 1, 'Monday', 'December', 4, False, False, 'winter', False), (20251230, '2025-12-30', 2025, 12, 30, 2, 'Tuesday', 'December', 4, False, False, 'winter', False), (20251231, '2025-12-31', 2025, 12, 31, 3, 'Wednesday', 'December', 4, False, False, 'winter', False), (20260101, '2026-01-01', 2026, 1, 1, 4, 'Thursday', 'January', 1, False, True, 'winter', False), (20260102, '2026-01-02', 2026, 1, 2, 5, 'Friday', 'January', 1, False, False, 'winter', False), (20260103, '2026-01-03', 2026, 1, 3, 6, 'Saturday', 'January', 1, True, False, 'winter', False), (20260104, '2026-01-04', 2026, 1, 4, 7, 'Sunday', 'January', 1, True, False, 'winter', False), (20260105, '2026-01-05', 2026, 1, 5, 1, 'Monday', 'January', 1, False, False, 'winter', False), (20260106, '2026-01-06', 2026, 1, 6, 2, 'Tuesday', 'January', 1, False, False, 'winter', False), (20260107, '2026-01-07', 2026, 1, 7, 3, 'Wednesday', 'January', 1, False, False, 'winter', False), (20260108, '2026-01-08', 2026, 1, 8, 4, 'Thursday', 'January', 1, False, False, 'winter', False), (20260109, '2026-01-09', 2026, 1, 9, 5, 'Friday', 'January', 1, False, False, 'winter', False), (20260110, '2026-01-10', 2026, 1, 10, 6, 'Saturday', 'January', 1, True, False, 'winter', False), (20260111, '2026-01-11', 2026, 1, 11, 7, 'Sunday', 'January', 1, True, False, 'winter', False), (20260112, '2026-01-12', 2026, 1, 12, 1, 'Monday', 'January', 1, False, False, 'winter', False), (20260113, '2026-01-13', 2026, 1, 13, 2, 'Tuesday', 'January', 1, False, False, 'winter', False), (20260114, '2026-01-14', 2026, 1, 14, 3, 'Wednesday', 'January', 1, False, False, 'winter', False), (20260115, '2026-01-15', 2026, 1, 15, 4, 'Thursday', 'January', 1, False, False, 'winter', False), (20260116, '2026-01-16', 2026, 1, 16, 5, 'Friday', 'January', 1, False, False, 'winter', False), (20260117, '2026-01-17', 2026, 1, 17, 6, 'Saturday', 'January', 1, True, False, 'winter', False), (20260118, '2026-01-18', 2026, 1, 18, 7, 'Sunday', 'January', 1, True, False, 'winter', False), (20260119, '2026-01-19', 2026, 1, 19, 1, 'Monday', 'January', 1, False, True, 'winter', False), (20260120, '2026-01-20', 2026, 1, 20, 2, 'Tuesday', 'January', 1, False, False, 'winter', False), (20260121, '2026-01-21', 2026, 1, 21, 3, 'Wednesday', 'January', 1, False, False, 'winter', False), (20260122, '2026-01-22', 2026, 1, 22, 4, 'Thursday', 'January', 1, False, False, 'winter', False), (20260123, '2026-01-23', 2026, 1, 23, 5, 'Friday', 'January', 1, False, False, 'winter', False), (20260124, '2026-01-24', 2026, 1, 24, 6, 'Saturday', 'January', 1, True, False, 'winter', False), (20260125, '2026-01-25', 2026, 1, 25, 7, 'Sunday', 'January', 1, True, False, 'winter', False), (20260126, '2026-01-26', 2026, 1, 26, 1, 'Monday', 'January', 1, False, False, 'winter', False), (20260127, '2026-01-27', 2026, 1, 27, 2, 'Tuesday', 'January', 1, False, False, 'winter', False), (20260128, '2026-01-28', 2026, 1, 28, 3, 'Wednesday', 'January', 1, False, False, 'winter', False), (20260129, '2026-01-29', 2026, 1, 29, 4, 'Thursday', 'January', 1, False, False, 'winter', False), (20260130, '2026-01-30', 2026, 1, 30, 5, 'Friday', 'January', 1, False, False, 'winter', False), (20260131, '2026-01-31', 2026, 1, 31, 6, 'Saturday', 'January', 1, True, False, 'winter', False), (20260201, '2026-02-01', 2026, 2, 1, 7, 'Sunday', 'February', 1, True, False, 'winter', False), (20260202, '2026-02-02', 2026, 2, 2, 1, 'Monday', 'February', 1, False, False, 'winter', False), (20260203, '2026-02-03', 2026, 2, 3, 2, 'Tuesday', 'February', 1, False, False, 'winter', False), (20260204, '2026-02-04', 2026, 2, 4, 3, 'Wednesday', 'February', 1, False, False, 'winter', False), (20260205, '2026-02-05', 2026, 2, 5, 4, 'Thursday', 'February', 1, False, False, 'winter', False), (20260206, '2026-02-06', 2026, 2, 6, 5, 'Friday', 'February', 1, False, False, 'winter', False), (20260207, '2026-02-07', 2026, 2, 7, 6, 'Saturday', 'February', 1, True, False, 'winter', False), (20260208, '2026-02-08', 2026, 2, 8, 7, 'Sunday', 'February', 1, True, False, 'winter', False), (20260209, '2026-02-09', 2026, 2, 9, 1, 'Monday', 'February', 1, False, False, 'winter', False), (20260210, '2026-02-10', 2026, 2, 10, 2, 'Tuesday', 'February', 1, False, False, 'winter', False), (20260211, '2026-02-11', 2026, 2, 11, 3, 'Wednesday', 'February', 1, False, False, 'winter', False), (20260212, '2026-02-12', 2026, 2, 12, 4, 'Thursday', 'February', 1, False, False, 'winter', False), (20260213, '2026-02-13', 2026, 2, 13, 5, 'Friday', 'February', 1, False, False, 'winter', False), (20260214, '2026-02-14', 2026, 2, 14, 6, 'Saturday', 'February', 1, True, False, 'winter', False), (20260215, '2026-02-15', 2026, 2, 15, 7, 'Sunday', 'February', 1, True, False, 'winter', False), (20260216, '2026-02-16', 2026, 2, 16, 1, 'Monday', 'February', 1, False, True, 'winter', False), (20260217, '2026-02-17', 2026, 2, 17, 2, 'Tuesday', 'February', 1, False, False, 'winter', False), (20260218, '2026-02-18', 2026, 2, 18, 3, 'Wednesday', 'February', 1, False, False, 'winter', False), (20260219, '2026-02-19', 2026, 2, 19, 4, 'Thursday', 'February', 1, False, False, 'winter', False), (20260220, '2026-02-20', 2026, 2, 20, 5, 'Friday', 'February', 1, False, False, 'winter', False), (20260221, '2026-02-21', 2026, 2, 21, 6, 'Saturday', 'February', 1, True, False, 'winter', False), (20260222, '2026-02-22', 2026, 2, 22, 7, 'Sunday', 'February', 1, True, False, 'winter', False), (20260223, '2026-02-23', 2026, 2, 23, 1, 'Monday', 'February', 1, False, False, 'winter', False), (20260224, '2026-02-24', 2026, 2, 24, 2, 'Tuesday', 'February', 1, False, False, 'winter', False), (20260225, '2026-02-25', 2026, 2, 25, 3, 'Wednesday', 'February', 1, False, False, 'winter', False), (20260226, '2026-02-26', 2026, 2, 26, 4, 'Thursday', 'February', 1, False, False, 'winter', False), (20260227, '2026-02-27', 2026, 2, 27, 5, 'Friday', 'February', 1, False, False, 'winter', False), (20260228, '2026-02-28', 2026, 2, 28, 6, 'Saturday', 'February', 1, True, False, 'winter', False), (20260301, '2026-03-01', 2026, 3, 1, 7, 'Sunday', 'March', 1, True, False, 'winter', False), (20260302, '2026-03-02', 2026, 3, 2, 1, 'Monday', 'March', 1, False, False, 'winter', False), (20260303, '2026-03-03', 2026, 3, 3, 2, 'Tuesday', 'March', 1, False, False, 'winter', False), (20260304, '2026-03-04', 2026, 3, 4, 3, 'Wednesday', 'March', 1, False, False, 'winter', False), (20260305, '2026-03-05', 2026, 3, 5, 4, 'Thursday', 'March', 1, False, False, 'winter', False), (20260306, '2026-03-06', 2026, 3, 6, 5, 'Friday', 'March', 1, False, False, 'winter', False), (20260307, '2026-03-07', 2026, 3, 7, 6, 'Saturday', 'March', 1, True, False, 'winter', False), (20260308, '2026-03-08', 2026, 3, 8, 7, 'Sunday', 'March', 1, True, False, 'winter', False), (20260309, '2026-03-09', 2026, 3, 9, 1, 'Monday', 'March', 1, False, False, 'winter', False), (20260310, '2026-03-10', 2026, 3, 10, 2, 'Tuesday', 'March', 1, False, False, 'winter', False), (20260311, '2026-03-11', 2026, 3, 11, 3, 'Wednesday', 'March', 1, False, False, 'winter', False), (20260312, '2026-03-12', 2026, 3, 12, 4, 'Thursday', 'March', 1, False, False, 'winter', False), (20260313, '2026-03-13', 2026, 3, 13, 5, 'Friday', 'March', 1, False, False, 'winter', False), (20260314, '2026-03-14', 2026, 3, 14, 6, 'Saturday', 'March', 1, True, False, 'winter', False), (20260315, '2026-03-15', 2026, 3, 15, 7, 'Sunday', 'March', 1, True, False, 'winter', False), (20260316, '2026-03-16', 2026, 3, 16, 1, 'Monday', 'March', 1, False, False, 'winter', False), (20260317, '2026-03-17', 2026, 3, 17, 2, 'Tuesday', 'March', 1, False, False, 'winter', False), (20260318, '2026-03-18', 2026, 3, 18, 3, 'Wednesday', 'March', 1, False, False, 'winter', False), (20260319, '2026-03-19', 2026, 3, 19, 4, 'Thursday', 'March', 1, False, False, 'winter', False), (20260320, '2026-03-20', 2026, 3, 20, 5, 'Friday', 'March', 1, False, False, 'winter', False), (20260321, '2026-03-21', 2026, 3, 21, 6, 'Saturday', 'March', 1, True, False, 'winter', False), (20260322, '2026-03-22', 2026, 3, 22, 7, 'Sunday', 'March', 1, True, False, 'winter', False), (20260323, '2026-03-23', 2026, 3, 23, 1, 'Monday', 'March', 1, False, False, 'winter', False), (20260324, '2026-03-24', 2026, 3, 24, 2, 'Tuesday', 'March', 1, False, False, 'winter', False), (20260325, '2026-03-25', 2026, 3, 25, 3, 'Wednesday', 'March', 1, False, False, 'winter', False), (20260326, '2026-03-26', 2026, 3, 26, 4, 'Thursday', 'March', 1, False, False, 'winter', False), (20260327, '2026-03-27', 2026, 3, 27, 5, 'Friday', 'March', 1, False, False, 'winter', False), (20260328, '2026-03-28', 2026, 3, 28, 6, 'Saturday', 'March', 1, True, False, 'winter', False), (20260329, '2026-03-29', 2026, 3, 29, 7, 'Sunday', 'March', 1, True, False, 'winter', False), (20260330, '2026-03-30', 2026, 3, 30, 1, 'Monday', 'March', 1, False, False, 'winter', False), (20260331, '2026-03-31', 2026, 3, 31, 2, 'Tuesday', 'March', 1, False, False, 'winter', False), (20260401, '2026-04-01', 2026, 4, 1, 3, 'Wednesday', 'April', 2, False, False, 'summer', False), (20260402, '2026-04-02', 2026, 4, 2, 4, 'Thursday', 'April', 2, False, False, 'summer', False), (20260403, '2026-04-03', 2026, 4, 3, 5, 'Friday', 'April', 2, False, False, 'summer', False), (20260404, '2026-04-04', 2026, 4, 4, 6, 'Saturday', 'April', 2, True, False, 'summer', False), (20260405, '2026-04-05', 2026, 4, 5, 7, 'Sunday', 'April', 2, True, False, 'summer', False), (20260406, '2026-04-06', 2026, 4, 6, 1, 'Monday', 'April', 2, False, False, 'summer', True), (20260407, '2026-04-07', 2026, 4, 7, 2, 'Tuesday', 'April', 2, False, False, 'summer', True), (20260408, '2026-04-08', 2026, 4, 8, 3, 'Wednesday', 'April', 2, False, False, 'summer', True), (20260409, '2026-04-09', 2026, 4, 9, 4, 'Thursday', 'April', 2, False, False, 'summer', True), (20260410, '2026-04-10', 2026, 4, 10, 5, 'Friday', 'April', 2, False, False, 'summer', True), (20260411, '2026-04-11', 2026, 4, 11, 6, 'Saturday', 'April', 2, True, False, 'summer', True), (20260412, '2026-04-12', 2026, 4, 12, 7, 'Sunday', 'April', 2, True, False, 'summer', True), (20260413, '2026-04-13', 2026, 4, 13, 1, 'Monday', 'April', 2, False, False, 'summer', True), (20260414, '2026-04-14', 2026, 4, 14, 2, 'Tuesday', 'April', 2, False, False, 'summer', True), (20260415, '2026-04-15', 2026, 4, 15, 3, 'Wednesday', 'April', 2, False, False, 'summer', True), (20260416, '2026-04-16', 2026, 4, 16, 4, 'Thursday', 'April', 2, False, False, 'summer', True), (20260417, '2026-04-17', 2026, 4, 17, 5, 'Friday', 'April', 2, False, False, 'summer', True), (20260418, '2026-04-18', 2026, 4, 18, 6, 'Saturday', 'April', 2, True, False, 'summer', True), (20260419, '2026-04-19', 2026, 4, 19, 7, 'Sunday', 'April', 2, True, False, 'summer', True), (20260420, '2026-04-20', 2026, 4, 20, 1, 'Monday', 'April', 2, False, False, 'summer', True), (20260421, '2026-04-21', 2026, 4, 21, 2, 'Tuesday', 'April', 2, False, False, 'summer', True), (20260422, '2026-04-22', 2026, 4, 22, 3, 'Wednesday', 'April', 2, False, False, 'summer', True), (20260423, '2026-04-23', 2026, 4, 23, 4, 'Thursday', 'April', 2, False, False, 'summer', True), (20260424, '2026-04-24', 2026, 4, 24, 5, 'Friday', 'April', 2, False, False, 'summer', True), (20260425, '2026-04-25', 2026, 4, 25, 6, 'Saturday', 'April', 2, True, False, 'summer', True), (20260426, '2026-04-26', 2026, 4, 26, 7, 'Sunday', 'April', 2, True, False, 'summer', True), (20260427, '2026-04-27', 2026, 4, 27, 1, 'Monday', 'April', 2, False, False, 'summer', True), (20260428, '2026-04-28', 2026, 4, 28, 2, 'Tuesday', 'April', 2, False, False, 'summer', True), (20260429, '2026-04-29', 2026, 4, 29, 3, 'Wednesday', 'April', 2, False, False, 'summer', True), (20260430, '2026-04-30', 2026, 4, 30, 4, 'Thursday', 'April', 2, False, False, 'summer', True), (20260501, '2026-05-01', 2026, 5, 1, 5, 'Friday', 'May', 2, False, False, 'summer', True), (20260502, '2026-05-02', 2026, 5, 2, 6, 'Saturday', 'May', 2, True, False, 'summer', True), (20260503, '2026-05-03', 2026, 5, 3, 7, 'Sunday', 'May', 2, True, False, 'summer', True), (20260504, '2026-05-04', 2026, 5, 4, 1, 'Monday', 'May', 2, False, False, 'summer', True), (20260505, '2026-05-05', 2026, 5, 5, 2, 'Tuesday', 'May', 2, False, False, 'summer', True), (20260506, '2026-05-06', 2026, 5, 6, 3, 'Wednesday', 'May', 2, False, False, 'summer', True), (20260507, '2026-05-07', 2026, 5, 7, 4, 'Thursday', 'May', 2, False, False, 'summer', True), (20260508, '2026-05-08', 2026, 5, 8, 5, 'Friday', 'May', 2, False, False, 'summer', True), (20260509, '2026-05-09', 2026, 5, 9, 6, 'Saturday', 'May', 2, True, False, 'summer', True), (20260510, '2026-05-10', 2026, 5, 10, 7, 'Sunday', 'May', 2, True, False, 'summer', True), (20260511, '2026-05-11', 2026, 5, 11, 1, 'Monday', 'May', 2, False, False, 'summer', True), (20260512, '2026-05-12', 2026, 5, 12, 2, 'Tuesday', 'May', 2, False, False, 'summer', True), (20260513, '2026-05-13', 2026, 5, 13, 3, 'Wednesday', 'May', 2, False, False, 'summer', True), (20260514, '2026-05-14', 2026, 5, 14, 4, 'Thursday', 'May', 2, False, False, 'summer', True), (20260515, '2026-05-15', 2026, 5, 15, 5, 'Friday', 'May', 2, False, False, 'summer', True), (20260516, '2026-05-16', 2026, 5, 16, 6, 'Saturday', 'May', 2, True, False, 'summer', True), (20260517, '2026-05-17', 2026, 5, 17, 7, 'Sunday', 'May', 2, True, False, 'summer', True), (20260518, '2026-05-18', 2026, 5, 18, 1, 'Monday', 'May', 2, False, False, 'summer', True), (20260519, '2026-05-19', 2026, 5, 19, 2, 'Tuesday', 'May', 2, False, False, 'summer', True), (20260520, '2026-05-20', 2026, 5, 20, 3, 'Wednesday', 'May', 2, False, False, 'summer', True), (20260521, '2026-05-21', 2026, 5, 21, 4, 'Thursday', 'May', 2, False, False, 'summer', True), (20260522, '2026-05-22', 2026, 5, 22, 5, 'Friday', 'May', 2, False, False, 'summer', True), (20260523, '2026-05-23', 2026, 5, 23, 6, 'Saturday', 'May', 2, True, False, 'summer', True), (20260524, '2026-05-24', 2026, 5, 24, 7, 'Sunday', 'May', 2, True, False, 'summer', True), (20260525, '2026-05-25', 2026, 5, 25, 1, 'Monday', 'May', 2, False, True, 'summer', True), (20260526, '2026-05-26', 2026, 5, 26, 2, 'Tuesday', 'May', 2, False, False, 'summer', True), (20260527, '2026-05-27', 2026, 5, 27, 3, 'Wednesday', 'May', 2, False, False, 'summer', True), (20260528, '2026-05-28', 2026, 5, 28, 4, 'Thursday', 'May', 2, False, False, 'summer', True), (20260529, '2026-05-29', 2026, 5, 29, 5, 'Friday', 'May', 2, False, False, 'summer', True), (20260530, '2026-05-30', 2026, 5, 30, 6, 'Saturday', 'May', 2, True, False, 'summer', True), (20260531, '2026-05-31', 2026, 5, 31, 7, 'Sunday', 'May', 2, True, False, 'summer', True), (20260601, '2026-06-01', 2026, 6, 1, 1, 'Monday', 'June', 2, False, False, 'summer', True), (20260602, '2026-06-02', 2026, 6, 2, 2, 'Tuesday', 'June', 2, False, False, 'summer', True), (20260603, '2026-06-03', 2026, 6, 3, 3, 'Wednesday', 'June', 2, False, False, 'summer', True), (20260604, '2026-06-04', 2026, 6, 4, 4, 'Thursday', 'June', 2, False, False, 'summer', True), (20260605, '2026-06-05', 2026, 6, 5, 5, 'Friday', 'June', 2, False, False, 'summer', True), (20260606, '2026-06-06', 2026, 6, 6, 6, 'Saturday', 'June', 2, True, False, 'summer', True), (20260607, '2026-06-07', 2026, 6, 7, 7, 'Sunday', 'June', 2, True, False, 'summer', True), (20260608, '2026-06-08', 2026, 6, 8, 1, 'Monday', 'June', 2, False, False, 'summer', True), (20260609, '2026-06-09', 2026, 6, 9, 2, 'Tuesday', 'June', 2, False, False, 'summer', True), (20260610, '2026-06-10', 2026, 6, 10, 3, 'Wednesday', 'June', 2, False, False, 'summer', True), (20260611, '2026-06-11', 2026, 6, 11, 4, 'Thursday', 'June', 2, False, False, 'summer', True), (20260612, '2026-06-12', 2026, 6, 12, 5, 'Friday', 'June', 2, False, False, 'summer', True), (20260613, '2026-06-13', 2026, 6, 13, 6, 'Saturday', 'June', 2, True, False, 'summer', True), (20260614, '2026-06-14', 2026, 6, 14, 7, 'Sunday', 'June', 2, True, False, 'summer', True), (20260615, '2026-06-15', 2026, 6, 15, 1, 'Monday', 'June', 2, False, False, 'summer', True), (20260616, '2026-06-16', 2026, 6, 16, 2, 'Tuesday', 'June', 2, False, False, 'summer', True), (20260617, '2026-06-17', 2026, 6, 17, 3, 'Wednesday', 'June', 2, False, False, 'summer', True), (20260618, '2026-06-18', 2026, 6, 18, 4, 'Thursday', 'June', 2, False, False, 'summer', True), (20260619, '2026-06-19', 2026, 6, 19, 5, 'Friday', 'June', 2, False, True, 'summer', True), (20260620, '2026-06-20', 2026, 6, 20, 6, 'Saturday', 'June', 2, True, False, 'summer', True), (20260621, '2026-06-21', 2026, 6, 21, 7, 'Sunday', 'June', 2, True, False, 'summer', True), (20260622, '2026-06-22', 2026, 6, 22, 1, 'Monday', 'June', 2, False, False, 'summer', True), (20260623, '2026-06-23', 2026, 6, 23, 2, 'Tuesday', 'June', 2, False, False, 'summer', True), (20260624, '2026-06-24', 2026, 6, 24, 3, 'Wednesday', 'June', 2, False, False, 'summer', True), (20260625, '2026-06-25', 2026, 6, 25, 4, 'Thursday', 'June', 2, False, False, 'summer', True), (20260626, '2026-06-26', 2026, 6, 26, 5, 'Friday', 'June', 2, False, False, 'summer', True), (20260627, '2026-06-27', 2026, 6, 27, 6, 'Saturday', 'June', 2, True, False, 'summer', True), (20260628, '2026-06-28', 2026, 6, 28, 7, 'Sunday', 'June', 2, True, False, 'summer', True), (20260629, '2026-06-29', 2026, 6, 29, 1, 'Monday', 'June', 2, False, False, 'summer', True), (20260630, '2026-06-30', 2026, 6, 30, 2, 'Tuesday', 'June', 2, False, False, 'summer', True), (20260701, '2026-07-01', 2026, 7, 1, 3, 'Wednesday', 'July', 3, False, False, 'summer', True), (20260702, '2026-07-02', 2026, 7, 2, 4, 'Thursday', 'July', 3, False, False, 'summer', True), (20260703, '2026-07-03', 2026, 7, 3, 5, 'Friday', 'July', 3, False, True, 'summer', True), (20260704, '2026-07-04', 2026, 7, 4, 6, 'Saturday', 'July', 3, True, False, 'summer', True), (20260705, '2026-07-05', 2026, 7, 5, 7, 'Sunday', 'July', 3, True, False, 'summer', True), (20260706, '2026-07-06', 2026, 7, 6, 1, 'Monday', 'July', 3, False, False, 'summer', True), (20260707, '2026-07-07', 2026, 7, 7, 2, 'Tuesday', 'July', 3, False, False, 'summer', True), (20260708, '2026-07-08', 2026, 7, 8, 3, 'Wednesday', 'July', 3, False, False, 'summer', True), (20260709, '2026-07-09', 2026, 7, 9, 4, 'Thursday', 'July', 3, False, False, 'summer', True), (20260710, '2026-07-10', 2026, 7, 10, 5, 'Friday', 'July', 3, False, False, 'summer', True), (20260711, '2026-07-11', 2026, 7, 11, 6, 'Saturday', 'July', 3, True, False, 'summer', True), (20260712, '2026-07-12', 2026, 7, 12, 7, 'Sunday', 'July', 3, True, False, 'summer', True), (20260713, '2026-07-13', 2026, 7, 13, 1, 'Monday', 'July', 3, False, False, 'summer', True), (20260714, '2026-07-14', 2026, 7, 14, 2, 'Tuesday', 'July', 3, False, False, 'summer', True), (20260715, '2026-07-15', 2026, 7, 15, 3, 'Wednesday', 'July', 3, False, False, 'summer', True), (20260716, '2026-07-16', 2026, 7, 16, 4, 'Thursday', 'July', 3, False, False, 'summer', True), (20260717, '2026-07-17', 2026, 7, 17, 5, 'Friday', 'July', 3, False, False, 'summer', True), (20260718, '2026-07-18', 2026, 7, 18, 6, 'Saturday', 'July', 3, True, False, 'summer', True), (20260719, '2026-07-19', 2026, 7, 19, 7, 'Sunday', 'July', 3, True, False, 'summer', True), (20260720, '2026-07-20', 2026, 7, 20, 1, 'Monday', 'July', 3, False, False, 'summer', True), (20260721, '2026-07-21', 2026, 7, 21, 2, 'Tuesday', 'July', 3, False, False, 'summer', True), (20260722, '2026-07-22', 2026, 7, 22, 3, 'Wednesday', 'July', 3, False, False, 'summer', True), (20260723, '2026-07-23', 2026, 7, 23, 4, 'Thursday', 'July', 3, False, False, 'summer', True), (20260724, '2026-07-24', 2026, 7, 24, 5, 'Friday', 'July', 3, False, False, 'summer', True), (20260725, '2026-07-25', 2026, 7, 25, 6, 'Saturday', 'July', 3, True, False, 'summer', True), (20260726, '2026-07-26', 2026, 7, 26, 7, 'Sunday', 'July', 3, True, False, 'summer', True), (20260727, '2026-07-27', 2026, 7, 27, 1, 'Monday', 'July', 3, False, False, 'summer', True), (20260728, '2026-07-28', 2026, 7, 28, 2, 'Tuesday', 'July', 3, False, False, 'summer', True), (20260729, '2026-07-29', 2026, 7, 29, 3, 'Wednesday', 'July', 3, False, False, 'summer', True), (20260730, '2026-07-30', 2026, 7, 30, 4, 'Thursday', 'July', 3, False, False, 'summer', True), (20260731, '2026-07-31', 2026, 7, 31, 5, 'Friday', 'July', 3, False, False, 'summer', True), (20260801, '2026-08-01', 2026, 8, 1, 6, 'Saturday', 'August', 3, True, False, 'summer', True), (20260802, '2026-08-02', 2026, 8, 2, 7, 'Sunday', 'August', 3, True, False, 'summer', True), (20260803, '2026-08-03', 2026, 8, 3, 1, 'Monday', 'August', 3, False, False, 'summer', True), (20260804, '2026-08-04', 2026, 8, 4, 2, 'Tuesday', 'August', 3, False, False, 'summer', True), (20260805, '2026-08-05', 2026, 8, 5, 3, 'Wednesday', 'August', 3, False, False, 'summer', True), (20260806, '2026-08-06', 2026, 8, 6, 4, 'Thursday', 'August', 3, False, False, 'summer', True), (20260807, '2026-08-07', 2026, 8, 7, 5, 'Friday', 'August', 3, False, False, 'summer', True), (20260808, '2026-08-08', 2026, 8, 8, 6, 'Saturday', 'August', 3, True, False, 'summer', True), (20260809, '2026-08-09', 2026, 8, 9, 7, 'Sunday', 'August', 3, True, False, 'summer', True), (20260810, '2026-08-10', 2026, 8, 10, 1, 'Monday', 'August', 3, False, False, 'summer', True), (20260811, '2026-08-11', 2026, 8, 11, 2, 'Tuesday', 'August', 3, False, False, 'summer', True), (20260812, '2026-08-12', 2026, 8, 12, 3, 'Wednesday', 'August', 3, False, False, 'summer', True), (20260813, '2026-08-13', 2026, 8, 13, 4, 'Thursday', 'August', 3, False, False, 'summer', True), (20260814, '2026-08-14', 2026, 8, 14, 5, 'Friday', 'August', 3, False, False, 'summer', True), (20260815, '2026-08-15', 2026, 8, 15, 6, 'Saturday', 'August', 3, True, False, 'summer', True), (20260816, '2026-08-16', 2026, 8, 16, 7, 'Sunday', 'August', 3, True, False, 'summer', True), (20260817, '2026-08-17', 2026, 8, 17, 1, 'Monday', 'August', 3, False, False, 'summer', True), (20260818, '2026-08-18', 2026, 8, 18, 2, 'Tuesday', 'August', 3, False, False, 'summer', True), (20260819, '2026-08-19', 2026, 8, 19, 3, 'Wednesday', 'August', 3, False, False, 'summer', True), (20260820, '2026-08-20', 2026, 8, 20, 4, 'Thursday', 'August', 3, False, False, 'summer', True), (20260821, '2026-08-21', 2026, 8, 21, 5, 'Friday', 'August', 3, False, False, 'summer', True), (20260822, '2026-08-22', 2026, 8, 22, 6, 'Saturday', 'August', 3, True, False, 'summer', True), (20260823, '2026-08-23', 2026, 8, 23, 7, 'Sunday', 'August', 3, True, False, 'summer', True), (20260824, '2026-08-24', 2026, 8, 24, 1, 'Monday', 'August', 3, False, False, 'summer', True), (20260825, '2026-08-25', 2026, 8, 25, 2, 'Tuesday', 'August', 3, False, False, 'summer', True), (20260826, '2026-08-26', 2026, 8, 26, 3, 'Wednesday', 'August', 3, False, False, 'summer', True), (20260827, '2026-08-27', 2026, 8, 27, 4, 'Thursday', 'August', 3, False, False, 'summer', True), (20260828, '2026-08-28', 2026, 8, 28, 5, 'Friday', 'August', 3, False, False, 'summer', True), (20260829, '2026-08-29', 2026, 8, 29, 6, 'Saturday', 'August', 3, True, False, 'summer', True), (20260830, '2026-08-30', 2026, 8, 30, 7, 'Sunday', 'August', 3, True, False, 'summer', True), (20260831, '2026-08-31', 2026, 8, 31, 1, 'Monday', 'August', 3, False, False, 'summer', True), (20260901, '2026-09-01', 2026, 9, 1, 2, 'Tuesday', 'September', 3, False, False, 'summer', True), (20260902, '2026-09-02', 2026, 9, 2, 3, 'Wednesday', 'September', 3, False, False, 'summer', True), (20260903, '2026-09-03', 2026, 9, 3, 4, 'Thursday', 'September', 3, False, False, 'summer', True), (20260904, '2026-09-04', 2026, 9, 4, 5, 'Friday', 'September', 3, False, False, 'summer', True), (20260905, '2026-09-05', 2026, 9, 5, 6, 'Saturday', 'September', 3, True, False, 'summer', True), (20260906, '2026-09-06', 2026, 9, 6, 7, 'Sunday', 'September', 3, True, False, 'summer', True), (20260907, '2026-09-07', 2026, 9, 7, 1, 'Monday', 'September', 3, False, True, 'summer', True), (20260908, '2026-09-08', 2026, 9, 8, 2, 'Tuesday', 'September', 3, False, False, 'summer', True), (20260909, '2026-09-09', 2026, 9, 9, 3, 'Wednesday', 'September', 3, False, False, 'summer', True), (20260910, '2026-09-10', 2026, 9, 10, 4, 'Thursday', 'September', 3, False, False, 'summer', True), (20260911, '2026-09-11', 2026, 9, 11, 5, 'Friday', 'September', 3, False, False, 'summer', True), (20260912, '2026-09-12', 2026, 9, 12, 6, 'Saturday', 'September', 3, True, False, 'summer', True), (20260913, '2026-09-13', 2026, 9, 13, 7, 'Sunday', 'September', 3, True, False, 'summer', True), (20260914, '2026-09-14', 2026, 9, 14, 1, 'Monday', 'September', 3, False, False, 'summer', True), (20260915, '2026-09-15', 2026, 9, 15, 2, 'Tuesday', 'September', 3, False, False, 'summer', True), (20260916, '2026-09-16', 2026, 9, 16, 3, 'Wednesday', 'September', 3, False, False, 'summer', True), (20260917, '2026-09-17', 2026, 9, 17, 4, 'Thursday', 'September', 3, False, False, 'summer', True), (20260918, '2026-09-18', 2026, 9, 18, 5, 'Friday', 'September', 3, False, False, 'summer', True), (20260919, '2026-09-19', 2026, 9, 19, 6, 'Saturday', 'September', 3, True, False, 'summer', True), (20260920, '2026-09-20', 2026, 9, 20, 7, 'Sunday', 'September', 3, True, False, 'summer', True), (20260921, '2026-09-21', 2026, 9, 21, 1, 'Monday', 'September', 3, False, False, 'summer', True), (20260922, '2026-09-22', 2026, 9, 22, 2, 'Tuesday', 'September', 3, False, False, 'summer', True), (20260923, '2026-09-23', 2026, 9, 23, 3, 'Wednesday', 'September', 3, False, False, 'summer', True), (20260924, '2026-09-24', 2026, 9, 24, 4, 'Thursday', 'September', 3, False, False, 'summer', True), (20260925, '2026-09-25', 2026, 9, 25, 5, 'Friday', 'September', 3, False, False, 'summer', True), (20260926, '2026-09-26', 2026, 9, 26, 6, 'Saturday', 'September', 3, True, False, 'summer', True), (20260927, '2026-09-27', 2026, 9, 27, 7, 'Sunday', 'September', 3, True, False, 'summer', True), (20260928, '2026-09-28', 2026, 9, 28, 1, 'Monday', 'September', 3, False, False, 'summer', True), (20260929, '2026-09-29', 2026, 9, 29, 2, 'Tuesday', 'September', 3, False, False, 'summer', True), (20260930, '2026-09-30', 2026, 9, 30, 3, 'Wednesday', 'September', 3, False, False, 'summer', True), (20261001, '2026-10-01', 2026, 10, 1, 4, 'Thursday', 'October', 4, False, False, 'summer', True), (20261002, '2026-10-02', 2026, 10, 2, 5, 'Friday', 'October', 4, False, False, 'summer', True), (20261003, '2026-10-03', 2026, 10, 3, 6, 'Saturday', 'October', 4, True, False, 'summer', True), (20261004, '2026-10-04', 2026, 10, 4, 7, 'Sunday', 'October', 4, True, False, 'summer', True), (20261005, '2026-10-05', 2026, 10, 5, 1, 'Monday', 'October', 4, False, False, 'summer', True), (20261006, '2026-10-06', 2026, 10, 6, 2, 'Tuesday', 'October', 4, False, False, 'summer', True), (20261007, '2026-10-07', 2026, 10, 7, 3, 'Wednesday', 'October', 4, False, False, 'summer', True), (20261008, '2026-10-08', 2026, 10, 8, 4, 'Thursday', 'October', 4, False, False, 'summer', True), (20261009, '2026-10-09', 2026, 10, 9, 5, 'Friday', 'October', 4, False, False, 'summer', True), (20261010, '2026-10-10', 2026, 10, 10, 6, 'Saturday', 'October', 4, True, False, 'summer', True), (20261011, '2026-10-11', 2026, 10, 11, 7, 'Sunday', 'October', 4, True, False, 'summer', True), (20261012, '2026-10-12', 2026, 10, 12, 1, 'Monday', 'October', 4, False, True, 'summer', True), (20261013, '2026-10-13', 2026, 10, 13, 2, 'Tuesday', 'October', 4, False, False, 'summer', True), (20261014, '2026-10-14', 2026, 10, 14, 3, 'Wednesday', 'October', 4, False, False, 'summer', True), (20261015, '2026-10-15', 2026, 10, 15, 4, 'Thursday', 'October', 4, False, False, 'summer', True), (20261016, '2026-10-16', 2026, 10, 16, 5, 'Friday', 'October', 4, False, False, 'summer', True), (20261017, '2026-10-17', 2026, 10, 17, 6, 'Saturday', 'October', 4, True, False, 'summer', True), (20261018, '2026-10-18', 2026, 10, 18, 7, 'Sunday', 'October', 4, True, False, 'summer', True), (20261019, '2026-10-19', 2026, 10, 19, 1, 'Monday', 'October', 4, False, False, 'summer', True), (20261020, '2026-10-20', 2026, 10, 20, 2, 'Tuesday', 'October', 4, False, False, 'summer', True), (20261021, '2026-10-21', 2026, 10, 21, 3, 'Wednesday', 'October', 4, False, False, 'summer', True), (20261022, '2026-10-22', 2026, 10, 22, 4, 'Thursday', 'October', 4, False, False, 'summer', True), (20261023, '2026-10-23', 2026, 10, 23, 5, 'Friday', 'October', 4, False, False, 'summer', True), (20261024, '2026-10-24', 2026, 10, 24, 6, 'Saturday', 'October', 4, True, False, 'summer', True), (20261025, '2026-10-25', 2026, 10, 25, 7, 'Sunday', 'October', 4, True, False, 'summer', True), (20261026, '2026-10-26', 2026, 10, 26, 1, 'Monday', 'October', 4, False, False, 'summer', True), (20261027, '2026-10-27', 2026, 10, 27, 2, 'Tuesday', 'October', 4, False, False, 'summer', True), (20261028, '2026-10-28', 2026, 10, 28, 3, 'Wednesday', 'October', 4, False, False, 'summer', True), (20261029, '2026-10-29', 2026, 10, 29, 4, 'Thursday', 'October', 4, False, False, 'summer', True), (20261030, '2026-10-30', 2026, 10, 30, 5, 'Friday', 'October', 4, False, False, 'summer', True), (20261031, '2026-10-31', 2026, 10, 31, 6, 'Saturday', 'October', 4, True, False, 'summer', True), (20261101, '2026-11-01', 2026, 11, 1, 7, 'Sunday', 'November', 4, True, False, 'winter', True), (20261102, '2026-11-02', 2026, 11, 2, 1, 'Monday', 'November', 4, False, False, 'winter', True), (20261103, '2026-11-03', 2026, 11, 3, 2, 'Tuesday', 'November', 4, False, False, 'winter', True), (20261104, '2026-11-04', 2026, 11, 4, 3, 'Wednesday', 'November', 4, False, False, 'winter', True), (20261105, '2026-11-05', 2026, 11, 5, 4, 'Thursday', 'November', 4, False, False, 'winter', True), (20261106, '2026-11-06', 2026, 11, 6, 5, 'Friday', 'November', 4, False, False, 'winter', True), (20261107, '2026-11-07', 2026, 11, 7, 6, 'Saturday', 'November', 4, True, False, 'winter', True), (20261108, '2026-11-08', 2026, 11, 8, 7, 'Sunday', 'November', 4, True, False, 'winter', True), (20261109, '2026-11-09', 2026, 11, 9, 1, 'Monday', 'November', 4, False, False, 'winter', True), (20261110, '2026-11-10', 2026, 11, 10, 2, 'Tuesday', 'November', 4, False, False, 'winter', True), (20261111, '2026-11-11', 2026, 11, 11, 3, 'Wednesday', 'November', 4, False, True, 'winter', True), (20261112, '2026-11-12', 2026, 11, 12, 4, 'Thursday', 'November', 4, False, False, 'winter', True), (20261113, '2026-11-13', 2026, 11, 13, 5, 'Friday', 'November', 4, False, False, 'winter', True), (20261114, '2026-11-14', 2026, 11, 14, 6, 'Saturday', 'November', 4, True, False, 'winter', True), (20261115, '2026-11-15', 2026, 11, 15, 7, 'Sunday', 'November', 4, True, False, 'winter', True), (20261116, '2026-11-16', 2026, 11, 16, 1, 'Monday', 'November', 4, False, False, 'winter', True), (20261117, '2026-11-17', 2026, 11, 17, 2, 'Tuesday', 'November', 4, False, False, 'winter', True), (20261118, '2026-11-18', 2026, 11, 18, 3, 'Wednesday', 'November', 4, False, False, 'winter', True), (20261119, '2026-11-19', 2026, 11, 19, 4, 'Thursday', 'November', 4, False, False, 'winter', True), (20261120, '2026-11-20', 2026, 11, 20, 5, 'Friday', 'November', 4, False, False, 'winter', True), (20261121, '2026-11-21', 2026, 11, 21, 6, 'Saturday', 'November', 4, True, False, 'winter', True), (20261122, '2026-11-22', 2026, 11, 22, 7, 'Sunday', 'November', 4, True, False, 'winter', True), (20261123, '2026-11-23', 2026, 11, 23, 1, 'Monday', 'November', 4, False, False, 'winter', True), (20261124, '2026-11-24', 2026, 11, 24, 2, 'Tuesday', 'November', 4, False, False, 'winter', True), (20261125, '2026-11-25', 2026, 11, 25, 3, 'Wednesday', 'November', 4, False, False, 'winter', True), (20261126, '2026-11-26', 2026, 11, 26, 4, 'Thursday', 'November', 4, False, True, 'winter', True), (20261127, '2026-11-27', 2026, 11, 27, 5, 'Friday', 'November', 4, False, False, 'winter', True), (20261128, '2026-11-28', 2026, 11, 28, 6, 'Saturday', 'November', 4, True, False, 'winter', True), (20261129, '2026-11-29', 2026, 11, 29, 7, 'Sunday', 'November', 4, True, False, 'winter', True), (20261130, '2026-11-30', 2026, 11, 30, 1, 'Monday', 'November', 4, False, False, 'winter', True), (20261201, '2026-12-01', 2026, 12, 1, 2, 'Tuesday', 'December', 4, False, False, 'winter', True), (20261202, '2026-12-02', 2026, 12, 2, 3, 'Wednesday', 'December', 4, False, False, 'winter', True), (20261203, '2026-12-03', 2026, 12, 3, 4, 'Thursday', 'December', 4, False, False, 'winter', True), (20261204, '2026-12-04', 2026, 12, 4, 5, 'Friday', 'December', 4, False, False, 'winter', True), (20261205, '2026-12-05', 2026, 12, 5, 6, 'Saturday', 'December', 4, True, False, 'winter', True), (20261206, '2026-12-06', 2026, 12, 6, 7, 'Sunday', 'December', 4, True, False, 'winter', True), (20261207, '2026-12-07', 2026, 12, 7, 1, 'Monday', 'December', 4, False, False, 'winter', True), (20261208, '2026-12-08', 2026, 12, 8, 2, 'Tuesday', 'December', 4, False, False, 'winter', True), (20261209, '2026-12-09', 2026, 12, 9, 3, 'Wednesday', 'December', 4, False, False, 'winter', True), (20261210, '2026-12-10', 2026, 12, 10, 4, 'Thursday', 'December', 4, False, False, 'winter', True), (20261211, '2026-12-11', 2026, 12, 11, 5, 'Friday', 'December', 4, False, False, 'winter', True), (20261212, '2026-12-12', 2026, 12, 12, 6, 'Saturday', 'December', 4, True, False, 'winter', True), (20261213, '2026-12-13', 2026, 12, 13, 7, 'Sunday', 'December', 4, True, False, 'winter', True), (20261214, '2026-12-14', 2026, 12, 14, 1, 'Monday', 'December', 4, False, False, 'winter', True), (20261215, '2026-12-15', 2026, 12, 15, 2, 'Tuesday', 'December', 4, False, False, 'winter', True), (20261216, '2026-12-16', 2026, 12, 16, 3, 'Wednesday', 'December', 4, False, False, 'winter', True), (20261217, '2026-12-17', 2026, 12, 17, 4, 'Thursday', 'December', 4, False, False, 'winter', True), (20261218, '2026-12-18', 2026, 12, 18, 5, 'Friday', 'December', 4, False, False, 'winter', True), (20261219, '2026-12-19', 2026, 12, 19, 6, 'Saturday', 'December', 4, True, False, 'winter', True), (20261220, '2026-12-20', 2026, 12, 20, 7, 'Sunday', 'December', 4, True, False, 'winter', True), (20261221, '2026-12-21', 2026, 12, 21, 1, 'Monday', 'December', 4, False, False, 'winter', True), (20261222, '2026-12-22', 2026, 12, 22, 2, 'Tuesday', 'December', 4, False, False, 'winter', True), (20261223, '2026-12-23', 2026, 12, 23, 3, 'Wednesday', 'December', 4, False, False, 'winter', True), (20261224, '2026-12-24', 2026, 12, 24, 4, 'Thursday', 'December', 4, False, False, 'winter', True), (20261225, '2026-12-25', 2026, 12, 25, 5, 'Friday', 'December', 4, False, True, 'winter', True), (20261226, '2026-12-26', 2026, 12, 26, 6, 'Saturday', 'December', 4, True, False, 'winter', True), (20261227, '2026-12-27', 2026, 12, 27, 7, 'Sunday', 'December', 4, True, False, 'winter', True), (20261228, '2026-12-28', 2026, 12, 28, 1, 'Monday', 'December', 4, False, False, 'winter', True), (20261229, '2026-12-29', 2026, 12, 29, 2, 'Tuesday', 'December', 4, False, False, 'winter', True), (20261230, '2026-12-30', 2026, 12, 30, 3, 'Wednesday', 'December', 4, False, False, 'winter', True), (20261231, '2026-12-31', 2026, 12, 31, 4, 'Thursday', 'December', 4, False, False, 'winter', True)]
    cursor.executemany(
        "INSERT INTO dbo.dim_date (date_key, full_date, [year], [month], [day], day_of_week, day_name, month_name, quarter, is_weekend, is_holiday, schedule_season, is_post_dfw_rebank) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)",
        dim_date_rows,
    )
    print(f"inserted {len(dim_date_rows):,} dim_date rows")
    write_diag("gold_load_dim_date", {"rows": len(dim_date_rows)})
except Exception as e:
    write_diag('gold_load_dim_date_ERROR', {'error': repr(e), 'traceback': _tb.format_exc()})
    raise


In [ ]:
try:
    # dim_time_15: 96 rows (24h x 4 15-min slots), generated in Python.
    dim_time_rows = [(0, 0, 0, '00:00', 'Overnight'), (1, 0, 15, '00:15', 'Overnight'), (2, 0, 30, '00:30', 'Overnight'), (3, 0, 45, '00:45', 'Overnight'), (4, 1, 0, '01:00', 'Overnight'), (5, 1, 15, '01:15', 'Overnight'), (6, 1, 30, '01:30', 'Overnight'), (7, 1, 45, '01:45', 'Overnight'), (8, 2, 0, '02:00', 'Overnight'), (9, 2, 15, '02:15', 'Overnight'), (10, 2, 30, '02:30', 'Overnight'), (11, 2, 45, '02:45', 'Overnight'), (12, 3, 0, '03:00', 'Overnight'), (13, 3, 15, '03:15', 'Overnight'), (14, 3, 30, '03:30', 'Overnight'), (15, 3, 45, '03:45', 'Overnight'), (16, 4, 0, '04:00', 'Overnight'), (17, 4, 15, '04:15', 'Overnight'), (18, 4, 30, '04:30', 'Overnight'), (19, 4, 45, '04:45', 'Overnight'), (20, 5, 0, '05:00', 'Early Morning'), (21, 5, 15, '05:15', 'Early Morning'), (22, 5, 30, '05:30', 'Early Morning'), (23, 5, 45, '05:45', 'Early Morning'), (24, 6, 0, '06:00', 'Early Morning'), (25, 6, 15, '06:15', 'Early Morning'), (26, 6, 30, '06:30', 'Early Morning'), (27, 6, 45, '06:45', 'Early Morning'), (28, 7, 0, '07:00', 'Morning'), (29, 7, 15, '07:15', 'Morning'), (30, 7, 30, '07:30', 'Morning'), (31, 7, 45, '07:45', 'Morning'), (32, 8, 0, '08:00', 'Morning'), (33, 8, 15, '08:15', 'Morning'), (34, 8, 30, '08:30', 'Morning'), (35, 8, 45, '08:45', 'Morning'), (36, 9, 0, '09:00', 'Morning'), (37, 9, 15, '09:15', 'Morning'), (38, 9, 30, '09:30', 'Morning'), (39, 9, 45, '09:45', 'Morning'), (40, 10, 0, '10:00', 'Morning'), (41, 10, 15, '10:15', 'Morning'), (42, 10, 30, '10:30', 'Morning'), (43, 10, 45, '10:45', 'Morning'), (44, 11, 0, '11:00', 'Morning'), (45, 11, 15, '11:15', 'Morning'), (46, 11, 30, '11:30', 'Morning'), (47, 11, 45, '11:45', 'Morning'), (48, 12, 0, '12:00', 'Midday'), (49, 12, 15, '12:15', 'Midday'), (50, 12, 30, '12:30', 'Midday'), (51, 12, 45, '12:45', 'Midday'), (52, 13, 0, '13:00', 'Midday'), (53, 13, 15, '13:15', 'Midday'), (54, 13, 30, '13:30', 'Midday'), (55, 13, 45, '13:45', 'Midday'), (56, 14, 0, '14:00', 'Afternoon'), (57, 14, 15, '14:15', 'Afternoon'), (58, 14, 30, '14:30', 'Afternoon'), (59, 14, 45, '14:45', 'Afternoon'), (60, 15, 0, '15:00', 'Afternoon'), (61, 15, 15, '15:15', 'Afternoon'), (62, 15, 30, '15:30', 'Afternoon'), (63, 15, 45, '15:45', 'Afternoon'), (64, 16, 0, '16:00', 'Afternoon'), (65, 16, 15, '16:15', 'Afternoon'), (66, 16, 30, '16:30', 'Afternoon'), (67, 16, 45, '16:45', 'Afternoon'), (68, 17, 0, '17:00', 'Evening'), (69, 17, 15, '17:15', 'Evening'), (70, 17, 30, '17:30', 'Evening'), (71, 17, 45, '17:45', 'Evening'), (72, 18, 0, '18:00', 'Evening'), (73, 18, 15, '18:15', 'Evening'), (74, 18, 30, '18:30', 'Evening'), (75, 18, 45, '18:45', 'Evening'), (76, 19, 0, '19:00', 'Evening'), (77, 19, 15, '19:15', 'Evening'), (78, 19, 30, '19:30', 'Evening'), (79, 19, 45, '19:45', 'Evening'), (80, 20, 0, '20:00', 'Night'), (81, 20, 15, '20:15', 'Night'), (82, 20, 30, '20:30', 'Night'), (83, 20, 45, '20:45', 'Night'), (84, 21, 0, '21:00', 'Night'), (85, 21, 15, '21:15', 'Night'), (86, 21, 30, '21:30', 'Night'), (87, 21, 45, '21:45', 'Night'), (88, 22, 0, '22:00', 'Night'), (89, 22, 15, '22:15', 'Night'), (90, 22, 30, '22:30', 'Night'), (91, 22, 45, '22:45', 'Night'), (92, 23, 0, '23:00', 'Night'), (93, 23, 15, '23:15', 'Night'), (94, 23, 30, '23:30', 'Night'), (95, 23, 45, '23:45', 'Night')]
    cursor.executemany(
        "INSERT INTO dbo.dim_time_15 (time_key, hour24, minute15, hhmm, daypart) VALUES (?, ?, ?, ?, ?)",
        dim_time_rows,
    )
    print(f"inserted {len(dim_time_rows)} dim_time_15 rows")
    write_diag("gold_load_dim_time", {"rows": len(dim_time_rows)})
except Exception as e:
    write_diag('gold_load_dim_time_ERROR', {'error': repr(e), 'traceback': _tb.format_exc()})
    raise


In [ ]:
try:
    # dim_delay_cause, dim_cancel_reason, dim_carrier: real BTS-defined codes
    # and real carrier designators, not fabricated. dim_carrier's 21 codes are
    # the EXACT set observed in lh_silver.fact_flight_leg (confirmed via a
    # throwaway diagnostic before writing this) -- see build_gold_notebook.py's
    # REAL_CARRIERS comment for the is_aa_family reasoning.
    cursor.executemany(
        "INSERT INTO dbo.dim_delay_cause (cause_code, label, is_controllable) VALUES (?, ?, ?)",
        [('CarrierDelay', 'Air Carrier Delay', True), ('WeatherDelay', 'Weather Delay', False), ('NASDelay', 'National Aviation System Delay', False), ('SecurityDelay', 'Security Delay', False), ('LateAircraftDelay', 'Late Aircraft Delay', False)],
    )
    cursor.executemany(
        "INSERT INTO dbo.dim_cancel_reason (reason_code, label, is_controllable) VALUES (?, ?, ?)",
        [('A', 'Carrier', True), ('B', 'Weather', False), ('C', 'National Air System', False), ('D', 'Security', False)],
    )
    cursor.executemany(
        "INSERT INTO dbo.dim_carrier (carrier_key, carrier_name, is_mainline, is_aa_family) VALUES (?, ?, ?, ?)",
        [('AA', 'American Airlines', True, True), ('MQ', 'Envoy Air', False, True), ('OH', 'PSA Airlines', False, True), ('PT', 'Piedmont Airlines', False, True), ('DL', 'Delta Air Lines', True, False), ('9E', 'Endeavor Air', False, False), ('UA', 'United Airlines', True, False), ('C5', 'CommutAir', False, False), ('G7', 'GoJet Airlines', False, False), ('WN', 'Southwest Airlines', True, False), ('AS', 'Alaska Airlines', True, False), ('QX', 'Horizon Air', False, False), ('B6', 'JetBlue Airways', True, False), ('NK', 'Spirit Airlines', True, False), ('F9', 'Frontier Airlines', True, False), ('G4', 'Allegiant Air', True, False), ('HA', 'Hawaiian Airlines', True, False), ('OO', 'SkyWest Airlines', False, False), ('YV', 'Mesa Airlines', False, False), ('YX', 'Republic Airways', False, False), ('ZW', 'Air Wisconsin', False, False)],
    )
    print("inserted dim_delay_cause, dim_cancel_reason, dim_carrier")
    write_diag("gold_load_dim_static", {"delay_causes": 5, "cancel_reasons": 4, "carriers": 21})
except Exception as e:
    write_diag('gold_load_dim_static_ERROR', {'error': repr(e), 'traceback': _tb.format_exc()})
    raise


In [ ]:
try:
    # dim_airport, dim_aircraft: cross-database INSERT...SELECT straight from
    # the Lakehouse Delta tables (no Spark data movement needed).
    cursor.execute("""
        INSERT INTO dbo.dim_airport (airport_key, airport_name, municipality, iso_region, latitude, longitude, tz, is_aa_hub)
        SELECT iata, name, municipality, iso_region, latitude, longitude, tz, is_aa_hub
        FROM [lh_bronze].[dbo].[dim_airport]
    """)
    cursor.execute("""
        INSERT INTO dbo.dim_aircraft (tail_number, manufacturer, model, fleet_family, seats, num_engines, year_mfr)
        SELECT tail_number, manufacturer, model, model, seats, num_engines, year_mfr
        FROM [lh_bronze].[dbo].[dim_aircraft]
    """)
    cursor.execute("SELECT COUNT(*) FROM dbo.dim_airport")
    airport_count = cursor.fetchone()[0]
    cursor.execute("SELECT COUNT(*) FROM dbo.dim_aircraft")
    aircraft_count = cursor.fetchone()[0]
    print(f"dim_airport: {airport_count:,} rows, dim_aircraft: {aircraft_count:,} rows")
    write_diag("gold_load_dim_crossdb", {"dim_airport_rows": airport_count, "dim_aircraft_rows": aircraft_count})
except Exception as e:
    write_diag('gold_load_dim_crossdb_ERROR', {'error': repr(e), 'traceback': _tb.format_exc()})
    raise


In [ ]:
try:
    # dim_bank, fact_rebank_effect, fact_rebank_event_study: real output from
    # the local analyses (analysis/dfw_bank_reconstruction.py,
    # analysis/dfw_rebank_diffindiff.py -- see docs/analysis/*.csv), uploaded
    # to Files/analysis/ via the established PowerShell OneLake upload
    # pattern (ADR 0006). Tiny (35/3/16 rows) -- read with Spark, insert via
    # pyodbc rather than a cross-db query, since these aren't Lakehouse Delta
    # tables, just landed CSVs.
    analysis_root = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{LH_BRONZE_ID}/Files/analysis"

    bank_df = spark.read.option("header", True).csv(f"{analysis_root}/dim_bank_prototype.csv")
    bank_rows = [
        (i + 1, r["station"], r["period"], r["direction"], int(r["bank_number"]), r["peak_time"], float(r["peak_avg_flights"]), r["window_start"], r["window_end"])
        for i, r in enumerate(bank_df.collect())
    ]
    cursor.executemany(
        "INSERT INTO dbo.dim_bank (bank_key, station, period, direction, bank_number, peak_time, peak_avg_flights, window_start, window_end) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)",
        bank_rows,
    )

    effect_df = spark.read.option("header", True).csv(f"{analysis_root}/fact_rebank_effect_prototype.csv")
    effect_rows = [
        (r["hub"], r["metric"], r["model"], float(r["estimate"]), float(r["se"]), float(r["ci_low"]), float(r["ci_high"]), float(r["p_value"]), int(r["n"]), r["controls"])
        for r in effect_df.collect()
    ]
    cursor.executemany(
        "INSERT INTO dbo.fact_rebank_effect (hub, metric, model, estimate, se, ci_low, ci_high, p_value, n, controls) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)",
        effect_rows,
    )

    event_df = spark.read.option("header", True).csv(f"{analysis_root}/event_study_d0.csv")
    event_rows = [
        (int(r["week_relative"]), float(r["coef"]), float(r["ci_low"]), float(r["ci_high"]))
        for r in event_df.collect()
    ]
    cursor.executemany(
        "INSERT INTO dbo.fact_rebank_event_study (week_relative, coef, ci_low, ci_high) VALUES (?, ?, ?, ?)",
        event_rows,
    )
    print(f"dim_bank: {len(bank_rows)} rows, fact_rebank_effect: {len(effect_rows)} rows, fact_rebank_event_study: {len(event_rows)} rows")
    write_diag("gold_load_bank_rebank", {"dim_bank_rows": len(bank_rows), "fact_rebank_effect_rows": len(effect_rows), "fact_rebank_event_study_rows": len(event_rows)})
except Exception as e:
    write_diag('gold_load_bank_rebank_ERROR', {'error': repr(e), 'traceback': _tb.format_exc()})
    raise


In [ ]:
try:
    # fact_flight_leg: cross-database INSERT...SELECT from lh_silver's full
    # 44-month/~27.8M-row Delta table. Pure server-side T-SQL -- no Spark
    # DataFrame movement. TRY_CAST handles BTS's blank-string-for-missing
    # convention (e.g. TaxiIn blank on cancelled legs) by returning NULL
    # instead of erroring, matching this project's "don't treat a blank as
    # zero" rule for the delay-cause columns (kpi_spec.md).
    cursor.execute("""
        INSERT INTO dbo.fact_flight_leg (
            date_key, origin_airport_key, dest_airport_key, marketing_carrier_key, operating_carrier_key,
            flight_number, tail_number, sched_dep_utc, sched_arr_utc, actual_dep_utc, actual_arr_utc,
            dep_delay_min, arr_delay_min, taxi_out_min, taxi_in_min, crs_elapsed_min, actual_elapsed_min,
            carrier_delay_min, weather_delay_min, nas_delay_min, security_delay_min, late_aircraft_delay_min,
            is_cancelled, is_diverted, is_d0, is_a14, is_b0, is_controllable_cancel, cancellation_code, distance_mi
        )
        SELECT
            CAST(REPLACE(FlightDate, '-', '') AS INT),
            Origin,
            Dest,
            Marketing_Airline_Network,
            Operating_Airline,
            Flight_Number_Marketing_Airline,
            NULLIF(Tail_Number, ''),
            sched_dep_utc,
            sched_arr_utc,
            actual_dep_utc,
            actual_arr_utc,
            TRY_CAST(DepDelay AS FLOAT),
            TRY_CAST(ArrDelay AS FLOAT),
            txo_min,
            TRY_CAST(TaxiIn AS FLOAT),
            TRY_CAST(CRSElapsedTime AS FLOAT),
            TRY_CAST(ActualElapsedTime AS FLOAT),
            TRY_CAST(CarrierDelay AS FLOAT),
            TRY_CAST(WeatherDelay AS FLOAT),
            TRY_CAST(NASDelay AS FLOAT),
            TRY_CAST(SecurityDelay AS FLOAT),
            TRY_CAST(LateAircraftDelay AS FLOAT),
            is_cancelled,
            is_diverted,
            is_d0,
            is_a14,
            is_b0,
            is_controllable_cancel,
            NULLIF(CancellationCode, ''),
            TRY_CAST(Distance AS FLOAT)
        FROM [lh_silver].[dbo].[fact_flight_leg]
    """)
    cursor.execute("SELECT COUNT(*) FROM dbo.fact_flight_leg")
    leg_count = cursor.fetchone()[0]
    print(f"fact_flight_leg: {leg_count:,} rows")
    write_diag("gold_load_fact_flight_leg", {"rows": leg_count})
except Exception as e:
    write_diag('gold_load_fact_flight_leg_ERROR', {'error': repr(e), 'traceback': _tb.format_exc()})
    raise


In [ ]:
try:
    # fact_checkpoint_hour, fact_cbp_wait_hour: cross-database INSERT...SELECT
    # from lh_bronze. hour is stored as a string in bronze TSA data
    # (e.g. "0500") -- cast to the leading two digits as an int.
    cursor.execute("""
        INSERT INTO dbo.fact_checkpoint_hour (station, [checkpoint], date_key, hour24, week_start, passengers_screened)
        SELECT
            iata,
            [checkpoint],
            CAST(REPLACE(CAST([date] AS VARCHAR(10)), '-', '') AS INT),
            CAST(LEFT(hour, 2) AS INT),
            week_start,
            pax
        FROM [lh_bronze].[dbo].[tsa_checkpoint_hour]
    """)
    cursor.execute("""
        INSERT INTO dbo.fact_cbp_wait_hour (
            station, terminal, date_key, hour_range, avg_wait_us, max_wait_us,
            avg_wait_non_us, max_wait_non_us, booths_open, flight_count, total_passengers
        )
        SELECT
            AirportCode,
            TerminalName,
            CAST(REPLACE(CAST(FlightDate AS VARCHAR(10)), '-', '') AS INT),
            HourRange,
            UsaPassengerAverageWait,
            UsaPassengerMaxWait,
            NonUsaPassengerAverageWait,
            NonUsaPassengerMaxWait,
            BoothsUsed,
            FlightCount,
            TotalPassengerCount
        FROM [lh_bronze].[dbo].[cbp_wait_hour]
    """)
    cursor.execute("SELECT COUNT(*) FROM dbo.fact_checkpoint_hour")
    checkpoint_count = cursor.fetchone()[0]
    cursor.execute("SELECT COUNT(*) FROM dbo.fact_cbp_wait_hour")
    cbp_count = cursor.fetchone()[0]
    print(f"fact_checkpoint_hour: {checkpoint_count:,} rows, fact_cbp_wait_hour: {cbp_count:,} rows")
    write_diag("gold_load_fact_checkpoint_cbp", {"fact_checkpoint_hour_rows": checkpoint_count, "fact_cbp_wait_hour_rows": cbp_count})
except Exception as e:
    write_diag('gold_load_fact_checkpoint_cbp_ERROR', {'error': repr(e), 'traceback': _tb.format_exc()})
    raise


In [ ]:
try:
    # Phase 3 "done when" (revised per ADR 0012 -- DOT reconciliation
    # replaced since DOT is out of scope): gold-layer D0/A14/CF aggregated by
    # month must match the same metrics computed directly from
    # lh_bronze.bts_marketing_ontime (the untransformed source), to the
    # decimal, for every month checked. This still catches real bugs (a
    # broken join, a dropped partition, a miscounted cancellation) using
    # only data this project actually has.
    cursor.execute("""
        WITH gold_monthly AS (
            SELECT
                date_key / 100 AS year_month,
                CAST(SUM(CASE WHEN is_cancelled = 0 AND is_d0 = 1 THEN 1 ELSE 0 END) AS FLOAT)
                    / NULLIF(SUM(CASE WHEN is_cancelled = 0 THEN 1 ELSE 0 END), 0) AS d0_rate,
                CAST(SUM(CASE WHEN is_cancelled = 0 THEN 1 ELSE 0 END) AS FLOAT)
                    / NULLIF(COUNT(*), 0) AS cf_rate
            FROM dbo.fact_flight_leg
            GROUP BY date_key / 100
        ),
        bronze_monthly AS (
            SELECT
                CAST(Year AS INT) * 100 + CAST(Month AS INT) AS year_month,
                CAST(SUM(CASE WHEN TRY_CAST(Cancelled AS FLOAT) = 0 AND TRY_CAST(DepDelay AS FLOAT) <= 0 THEN 1 ELSE 0 END) AS FLOAT)
                    / NULLIF(SUM(CASE WHEN TRY_CAST(Cancelled AS FLOAT) = 0 THEN 1 ELSE 0 END), 0) AS d0_rate,
                CAST(SUM(CASE WHEN TRY_CAST(Cancelled AS FLOAT) = 0 THEN 1 ELSE 0 END) AS FLOAT)
                    / NULLIF(COUNT(*), 0) AS cf_rate
            FROM [lh_bronze].[dbo].[bts_marketing_ontime]
            GROUP BY CAST(Year AS INT) * 100 + CAST(Month AS INT)
        )
        SELECT
            g.year_month,
            g.d0_rate AS gold_d0, b.d0_rate AS bronze_d0, ABS(g.d0_rate - b.d0_rate) AS d0_diff,
            g.cf_rate AS gold_cf, b.cf_rate AS bronze_cf, ABS(g.cf_rate - b.cf_rate) AS cf_diff
        FROM gold_monthly g
        JOIN bronze_monthly b ON g.year_month = b.year_month
        ORDER BY g.year_month
    """)
    rows = cursor.fetchall()
    reconciliation = [
        {
            "year_month": r[0], "gold_d0": r[1], "bronze_d0": r[2], "d0_diff": r[3],
            "gold_cf": r[4], "bronze_cf": r[5], "cf_diff": r[6],
        }
        for r in rows
    ]
    max_d0_diff = max((r["d0_diff"] for r in reconciliation), default=None)
    max_cf_diff = max((r["cf_diff"] for r in reconciliation), default=None)
    print(f"months checked: {len(reconciliation)}, max D0 diff: {max_d0_diff}, max CF diff: {max_cf_diff}")
    write_diag("gold_reconciliation", {"months_checked": len(reconciliation), "max_d0_diff": max_d0_diff, "max_cf_diff": max_cf_diff, "detail": reconciliation})
except Exception as e:
    write_diag('gold_reconciliation_ERROR', {'error': repr(e), 'traceback': _tb.format_exc()})
    raise


In [ ]:
try:
    # Final row-count summary across every gold table, for a single
    # self-verifiable readout.
    tables = [
        "dim_date", "dim_time_15", "dim_airport", "dim_carrier", "dim_aircraft",
        "dim_delay_cause", "dim_cancel_reason", "dim_bank",
        "fact_flight_leg", "fact_checkpoint_hour", "fact_cbp_wait_hour",
        "fact_rebank_effect", "fact_rebank_event_study",
    ]
    summary = {}
    for t in tables:
        cursor.execute(f"SELECT COUNT(*) FROM dbo.{t}")
        summary[t] = cursor.fetchone()[0]
    print(_json.dumps(summary, indent=2))
    write_diag("gold_summary", summary)
except Exception as e:
    write_diag('gold_summary_ERROR', {'error': repr(e), 'traceback': _tb.format_exc()})
    raise
